# Data preparation — Icentia11k

Notebook autónomo para preparar a amostra local de 20 pacientes do Icentia11k. O código da pipeline está nas células abaixo; não depende de um módulo `.py` auxiliar.

A divisão é feita por paciente: **16 train, 2 dev e 2 test**. Dev/test são mantidos fora do treino. A frequência de origem é 250 Hz e o sinal é reamostrado para **200 Hz**. Cada batimento é cortado numa janela de 1 segundo (200 amostras) e interpolado para **500 pontos**, conforme o fluxo descrito no artigo.

O notebook associa as anotações `.atr` aos picos detetados, guarda rótulos N/S/V/Q, calcula a avaliação SQI fuzzy de Zhao et al. (2018) com adaptações explicitadas abaixo e distribui os exemplos train por 16 clientes de treino, um por paciente, além de 2 clientes dev e 2 test reservados para avaliação.

## Etapas implementadas

1. Ler registos WFDB dos pacientes selecionados.
2. Reamostrar 250 → 200 Hz.
3. Aplicar Butterworth high-pass de 5.ª ordem a 0,5 Hz e notch de 60 Hz.
4. Detetar picos R com NeuroKit2.
5. Extrair janelas de 1 segundo centradas nos picos R, descartando janelas incompletas nas extremidades.
6. Interpolar para 500 valores e aplicar min-max por batimento para [0, 1].
7. Associar os rótulos `.atr` N/S/V/Q às deteções R por correspondência temporal 1:1.
8. Calcular os índices q/p/k/bas e a qualidade fuzzy de Zhao et al.; atribuir um cliente fixo a cada paciente, preservando a distribuição natural das classes.
9. Guardar um `.npz` por batimento, manifesto global, manifestos de cliente e manifestos dev/test.

O output preparado fica em `datasets/icentia11_prepared/` e deve permanecer local, fora do Git.

In [ ]:
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import neurokit2 as nk
import numpy as np
import pandas as pd
import wfdb
from scipy import signal as scipy_signal
from scipy.interpolate import interp1d
from scipy.signal import resample_poly

SOURCE_FS = 250
TARGET_FS = 200
TARGET_BEAT_SAMPLES = 500
N_TRAIN_PATIENTS = 16
N_DEV_PATIENTS = 2
N_TEST_PATIENTS = 2
RANDOM_SEED = 2024
LIMIT_RECORDS_PER_PATIENT = None  # Preparar todos os registos locais.
RUN_PREPARATION = False  # Mudar para True depois de confirmar o espaço em disco.

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "DataPreparation":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATASET_ROOT = PROJECT_ROOT / "datasets" / "icentia11k_sample_20"
SELECTED_PATIENTS_FILE = DATASET_ROOT / "selected_patients.csv"
OUTPUT_ROOT = PROJECT_ROOT / "datasets" / "icentia11_prepared"

if not SELECTED_PATIENTS_FILE.is_file():
    raise FileNotFoundError(f"Não encontrei a seleção de pacientes: {SELECTED_PATIENTS_FILE}")
if not (PROJECT_ROOT / "requirements.txt").is_file():
    raise FileNotFoundError("Executa o notebook a partir da raiz do projeto ou de DataPreparation/.")

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Dados de entrada: {DATASET_ROOT}")
print(f"Saída: {OUTPUT_ROOT}")

In [ ]:
def get_patient_directories() -> List[Path]:
    """Lê a seleção reproduzível e valida a existência das pastas locais."""
    selected = pd.read_csv(SELECTED_PATIENTS_FILE)
    if "patient_dir" not in selected.columns:
        raise ValueError("selected_patients.csv tem de conter a coluna 'patient_dir'.")
    if selected["patient_dir"].duplicated().any():
        raise ValueError("selected_patients.csv contém pacientes duplicados.")

    patients = [DATASET_ROOT / str(value) for value in selected["patient_dir"]]
    missing = [path for path in patients if not path.is_dir()]
    if missing:
        raise FileNotFoundError(f"Faltam {len(missing)} pastas de pacientes; exemplos: {missing[:3]}")
    expected = N_TRAIN_PATIENTS + N_DEV_PATIENTS + N_TEST_PATIENTS
    if len(patients) != expected:
        raise ValueError(f"Esperava {expected} pacientes selecionados; encontrei {len(patients)}.")
    return sorted(patients)


def assign_patient_splits(patient_paths: Iterable[Path], seed: int = RANDOM_SEED) -> Dict[str, set]:
    """Divide pacientes sem sobreposição em 16 train, 2 dev e 2 test."""
    patient_ids = [path.name for path in sorted(patient_paths)]
    expected = N_TRAIN_PATIENTS + N_DEV_PATIENTS + N_TEST_PATIENTS
    if len(patient_ids) != expected or len(set(patient_ids)) != expected:
        raise ValueError(f"A divisão exige exatamente {expected} pacientes únicos.")

    shuffled = np.random.default_rng(seed).permutation(patient_ids).tolist()
    train_end = N_TRAIN_PATIENTS
    dev_end = train_end + N_DEV_PATIENTS
    return {
        "train": set(shuffled[:train_end]),
        "dev": set(shuffled[train_end:dev_end]),
        "test": set(shuffled[dev_end:]),
    }


def list_records(patient_dir: Path) -> List[Path]:
    """Lista prefixos de registos que têm header WFDB."""
    return [header.with_suffix("") for header in sorted(patient_dir.glob("*.hea"))]


def read_and_resample_record(record_path: Path) -> Tuple[np.ndarray, int]:
    """Lê o ECG a 250 Hz e reamostra para 200 Hz."""
    values, fields = wfdb.rdsamp(str(record_path), channels=[0], return_res=64)
    signal = np.asarray(values, dtype=np.float64).reshape(-1)
    source_fs = int(fields["fs"])
    if source_fs != SOURCE_FS:
        raise ValueError(f"{record_path.name}: frequência {source_fs} Hz; esperava {SOURCE_FS} Hz.")
    if not np.isfinite(signal).all():
        raise ValueError(f"O registo {record_path.name} contém NaN ou infinito.")
    return resample_poly(signal, up=4, down=5).astype(np.float64, copy=False), TARGET_FS


def filter_ecg(signal: np.ndarray, fs: int) -> np.ndarray:
    """Remove deriva da linha de base e interferência de rede a 60 Hz."""
    nyquist = fs / 2
    if not 0 < 0.5 < nyquist:
        raise ValueError(f"Frequência inválida para filtro: {fs} Hz.")
    b, a = scipy_signal.butter(5, 0.5 / nyquist, btype="highpass")
    filtered = scipy_signal.filtfilt(b, a, signal)
    line_frequency = 60
    if line_frequency < nyquist:
        b_notch, a_notch = scipy_signal.iirnotch(line_frequency, Q=35, fs=fs)
        filtered = scipy_signal.filtfilt(b_notch, a_notch, filtered)
    return filtered


def detect_r_peaks(signal: np.ndarray, fs: int) -> np.ndarray:
    """Deteta picos R com o método NeuroKit."""
    peak_data, _ = nk.ecg_peaks(signal, sampling_rate=fs, method="neurokit")
    return np.flatnonzero(np.asarray(peak_data["ECG_R_Peaks"], dtype=bool))


def make_beat(signal: np.ndarray, peak_index: int, fs: int = TARGET_FS) -> np.ndarray:
    """Extrai 1 s centrado no R, interpola para 500 e normaliza para [0, 1]."""
    half_window = int(round(fs * 0.5))
    start = peak_index - half_window
    end = peak_index + half_window
    if start < 0 or end > len(signal):
        return np.empty(0, dtype=np.float32)

    segment = np.asarray(signal[start:end], dtype=np.float64)
    if segment.size != fs:
        return np.empty(0, dtype=np.float32)
    x_source = np.linspace(0.0, 1.0, num=segment.size)
    x_target = np.linspace(0.0, 1.0, num=TARGET_BEAT_SAMPLES)
    interpolated = interp1d(x_source, segment, kind="linear")(x_target)
    minimum = float(interpolated.min())
    maximum = float(interpolated.max())
    if not np.isfinite(maximum - minimum) or maximum == minimum:
        return np.empty(0, dtype=np.float32)
    normalized = (interpolated - minimum) / (maximum - minimum)
    return normalized.astype(np.float32)



In [ ]:
patients = get_patient_directories()
splits = assign_patient_splits(patients)
assert {name: len(ids) for name, ids in splits.items()} == {"train": 16, "dev": 2, "test": 2}
assert not (splits["train"] & splits["dev"] or splits["train"] & splits["test"] or splits["dev"] & splits["test"])

patient_split = pd.DataFrame(
    [(patient.name, split) for split, ids in splits.items() for patient in patients if patient.name in ids],
    columns=["patient_id", "split"],
).sort_values(["split", "patient_id"]).reset_index(drop=True)
display(patient_split.groupby("split").size().rename("n_patients").to_frame())
display(patient_split)

In [ ]:
# Teste de uma amostra, sem gravar ficheiros preparados.
example_record = list_records(patients[0])[0]
example_signal, example_fs = read_and_resample_record(example_record)
example_filtered = filter_ecg(example_signal, example_fs)
example_peaks = detect_r_peaks(example_filtered, example_fs)
if len(example_peaks) == 0:
    raise RuntimeError(f"NeuroKit não detetou picos em {example_record.name}.")
example_beat = make_beat(example_filtered, int(example_peaks[len(example_peaks) // 2]), example_fs)

print(f"Registo: {example_record.name}")
print(f"Frequência de amostragem: {example_fs} Hz")
print(f"Amostras do registo: {len(example_signal):,}")
print(f"Picos R detetados: {len(example_peaks):,}")
print(f"Forma do batimento: {example_beat.shape}")
print(f"Amplitude normalizada: [{example_beat.min():.3f}, {example_beat.max():.3f}]")

## Executar a preparação

O notebook deixa a execução desligada por omissão. Primeiro executa a célula de teste acima. Para um teste rápido, mantém `LIMIT_RECORDS_PER_PATIENT = 1`. Para toda a amostra, define esse limite como `None`; confirma antes que tens espaço em disco suficiente, pois serão gravados milhares de ficheiros de batimentos.

In [ ]:
print("Usa a célula 'Executar a pipeline supervisionada' no fim do notebook.")

## Limitações metodológicas ainda a considerar

- A deteção de picos é feita por NeuroKit e comparada com `pantompkins1985` para qSQI, em vez dos detetores Hilbert-adaptativo e wavelet originais.
- O artigo avalia segmentos mais longos; aqui os índices SQI são calculados por janela de batimento de 1 segundo. Os limiares e a fusão fuzzy são os publicados, mas o resultado é uma adaptação à unidade de exemplo do projeto.
- Cada paciente corresponde a um cliente fixo. A heterogeneidade resulta da distribuição natural das classes por paciente; não há redistribuição sintética dos batimentos.
- A célula de teste rápido cobre um registo; o dataset local completo já foi preparado e os seus arrays foram validados.

Os outputs incluem rótulos supervisionados, mas a adaptação metodológica deve ser reportada ao comparar resultados com o artigo.
### Decisões mantidas e verificações adicionais

Mantêm-se 500 pontos e min-max para preservar a representação atual. Os 500 pontos são interpolados de uma janela de 200 amostras, não 2,5 segundos a 200 Hz. Comparar 200/500 pontos e min-max/z-score exige experiências de treino; não está validado por este notebook. Q permanece uma classe de batimento não classificável, sem ser equiparada a arritmia.

As rejeições e a cobertura das anotações por classe são exportadas em `record_processing_stats.csv`. Amplitudes originais são registadas no manifesto para inspeção, sem impor um limiar clínico arbitrário. Todas as categorias SQI finitas são mantidas. Erros de leitura/deteção ao nível do registo interrompem a preparação para inspeção.

A atribuição por paciente é validada como uma correspondência 1:1 entre paciente e cliente. Todos os clientes devem ter batimentos utilizáveis. O manifesto e os arrays são verificados antes da exportação dos manifestos.


## Compatibilidade com PhysioNet 2017

Ambos os sinais contínuos passam a 200 Hz, mas as unidades e dimensões dos exemplos ainda diferem: aqui guardamos cada batimento com 500 pontos; o notebook PhysioNet produz janelas de registo de 30 segundos. Para alimentar um único modelo, será necessário decidir uma unidade de exemplo e comprimento compatíveis. A divisão de avaliação permanece por paciente e nunca deve ser agregada como cliente de treino.

## Rótulos e SQI de Zhao et al. (2018)

Os rótulos `N`, `S`, `V` e `Q` são lidos das anotações `.atr`; marcadores como `+` são excluídos. Os pares candidatos são ordenados pelo erro temporal; os mais próximos são atribuídos primeiro, sem reutilizar deteções ou anotações, com tolerância de 100 ms a 200 Hz. Em conflitos, esta regra gulosa não garante a atribuição global ótima. Correspondências fora da tolerância são descartadas.

O SQI implementa os quatro índices q/p/k/bas, as funções de pertença fuzzy publicadas, os pesos (0.4, 0.4, 0.1, 0.1), a soma limitada e a decisão por média ponderada de Zhao et al. (2018). Para qSQI, esta implementação usa NeuroKit `neurokit` e `pantompkins1985`, em vez dos detetores Hilbert-adaptativo e wavelet do artigo. O artigo avalia janelas de sinal mais longas; aqui os índices são adaptados a janelas de batimento de 1 segundo. É, portanto, uma adaptação documentada dos limiares e da fusão, não uma reprodução experimental exata do estudo.

Referência: Zhao & Zhang, *SQI Quality Evaluation Mechanism of Single-Lead ECG Signal Based on Simple Heuristic Fusion and Fuzzy Comprehensive Evaluation* (2018), [doi:10.3389/fphys.2018.00727](https://doi.org/10.3389/fphys.2018.00727).

In [ ]:
import json
from scipy.integrate import trapezoid
from scipy.stats import kurtosis

LABEL_TO_ID = {"N": 0, "S": 1, "V": 2, "Q": 3}
N_TRAIN_CLIENTS = 16
ANNOTATION_TOLERANCE_SAMPLES = int(round(0.1 * TARGET_FS))
DETECTOR_TOLERANCE_SAMPLES = int(round(0.05 * TARGET_FS))

def detect_r_peaks(signal, fs, method="neurokit"):
    peak_data, _ = nk.ecg_peaks(signal, sampling_rate=fs, method=method)
    return np.flatnonzero(np.asarray(peak_data["ECG_R_Peaks"], dtype=bool))

def match_peaks_to_annotations(detected_peaks, annotation_samples, annotation_symbols):
    """One-to-one minimum-error-first match; non-beat markers such as '+' are excluded."""
    mask = np.isin(annotation_symbols, list(LABEL_TO_ID))
    ann_samples = np.rint(annotation_samples[mask] * TARGET_FS / SOURCE_FS).astype(int)
    ann_symbols = annotation_symbols[mask]
    order = np.argsort(ann_samples, kind="stable")
    ann_samples, ann_symbols = ann_samples[order], ann_symbols[order]
    # Resolve primeiro os pares com menor erro; cada deteção/anotação é usada uma vez.
    # O custo cresce apenas com os candidatos na tolerância temporal.
    candidates = []
    for i, peak in enumerate(detected_peaks):
        left = np.searchsorted(ann_samples, peak - ANNOTATION_TOLERANCE_SAMPLES)
        right = np.searchsorted(ann_samples, peak + ANNOTATION_TOLERANCE_SAMPLES, side="right")
        candidates.extend((abs(int(peak) - int(ann_samples[j])), i, j) for j in range(left, right))
    matches, used_annotations = {}, set()
    for _, i, j in sorted(candidates):
        if i not in matches and j not in used_annotations:
            matches[i] = (str(ann_symbols[j]), int(ann_samples[j]), int(detected_peaks[i]) - int(ann_samples[j]))
            used_annotations.add(j)
    return matches

def count_peak_matches(peaks_a, peaks_b, tolerance=DETECTOR_TOLERANCE_SAMPLES):
    i, j, count = 0, 0, 0
    while i < len(peaks_a) and j < len(peaks_b):
        delta = int(peaks_a[i]) - int(peaks_b[j])
        if abs(delta) <= tolerance:
            count += 1
            i, j = i + 1, j + 1
        elif delta < 0:
            i += 1
        else:
            j += 1
    return count

def band_power(frequencies, power, low_hz, high_hz):
    mask = (frequencies >= low_hz) & (frequencies <= high_hz)
    return float(trapezoid(power[mask], frequencies[mask])) if mask.sum() >= 2 else 0.0

def zhao_memberships(q, p, k, bas_percent):
    """Published q/p/k/bas membership functions; columns: E, B, U."""
    q, p, b = float(np.clip(q, 0, 100)), float(np.clip(p, 0, 1)), float(np.clip(bas_percent, 0, 100))
    q_e = 0.0 if q <= 80 else (1 / (1 + (0.3 * (q - 80)) ** -2) if q < 90 else 1.0)
    q_u = 1.0 if q <= 55 else 1 / (1 + ((q - 55) / 5) ** 2)
    q_b = 1 / (1 + ((q - 75) / 7.5) ** 2)
    p_e = 0.0 if p <= .25 else (10 * (p - .25) if p < .35 else 1.0)
    p_u = 1.0 if p < .15 else (10 * (.25 - p) if p <= .25 else 0.0)
    if p < .18:
        p_b = 0.0
    elif p < .22:
        p_b = 25 * (p - .18)
    elif p < .28:
        p_b = 1.0
    elif p < .32:
        p_b = 25 * (.32 - p)
    else:
        p_b = 0.0
    k_row = [1.0, 0.0, 0.0] if k > 5 else [0.0, 0.0, 1.0]
    bas_e = 0.0 if b <= 90 else (1 / (1 + (0.8718 * (b - 90)) ** -2) if b < 95 else 1.0)
    bas_u = 1.0 if b <= 85 else 1 / (1 + ((b - 85) / 5) ** 2)
    bas_b = 1 / (1 + ((b - 92) / 2.5) ** 2)
    return np.asarray([[q_e, q_b, q_u], [p_e, p_b, p_u], k_row, [bas_e, bas_b, bas_u]], dtype=float)

def zhao_sqi_for_beat(raw_window, main_in_window, second_in_window, fs):
    """Compute Zhao q/p/k/bas features and the weighted fuzzy E/B/U decision."""
    matches = count_peak_matches(main_in_window, second_in_window)
    detector_count = len(main_in_window) + len(second_in_window)
    q = 100 * (2 * matches / detector_count) if detector_count else 0.0
    frequencies, power = scipy_signal.periodogram(raw_window, fs=fs, window="hann", detrend=False, scaling="density")
    p_denom = band_power(frequencies, power, 5, 40)
    p = band_power(frequencies, power, 5, 15) / p_denom if p_denom > np.finfo(float).eps else 0.0
    total_power = band_power(frequencies, power, 0, 40)
    bas = 100 * (1 - band_power(frequencies, power, 0, 1) / total_power) if total_power > np.finfo(float).eps else 0.0
    if raw_window.size != fs or not np.isfinite(raw_window).all():
        raise ValueError("SQI requires a complete finite 1-second ECG window.")
    k = float(kurtosis(raw_window, fisher=False, bias=True))
    if not np.isfinite([q, p, k, bas]).all():
        raise ValueError("Non-finite q/p/k/bas SQI feature; inspect the source window.")
    memberships = zhao_memberships(q, p, k, bas)
    weighted = np.minimum(1.0, np.sum(np.asarray([.4, .4, .1, .1])[:, None] * memberships, axis=0))
    if weighted.sum() <= np.finfo(float).eps:
        raise RuntimeError("Empty fuzzy membership vector from Zhao SQI.")
    squared = weighted ** 2
    score = float(np.dot(squared, [1, 2, 3]) / squared.sum())
    quality = "Excellent" if score <= 1.5 else ("Barely acceptable" if score < 2.4 else "Unacceptable")
    return quality, {"q_sqi_percent": q, "p_sqi": p, "k_sqi": k, "bas_sqi_percent": bas, "fuzzy_quality_score": score}

In [ ]:
def validate_prepared_dataset(manifest, validate_arrays=True):
    """Valida o manifesto e todos os arrays exportados antes de publicar os manifestos."""
    assert not manifest.empty
    assert not manifest["segment_path"].duplicated().any(), "Caminhos duplicados"
    assert not manifest.duplicated(["patient_id", "record_id", "peak_index_200hz"]).any()
    assert set(manifest["split"]) == {"train", "dev", "test"}
    assert manifest.groupby("patient_id")["split"].nunique().eq(1).all(), "Patient leakage"
    assert manifest["label"].isin(LABEL_TO_ID).all()
    assert manifest["label_id"].eq(manifest["label"].map(LABEL_TO_ID)).all()
    assert manifest["sampling_rate_hz"].eq(TARGET_FS).all()
    assert manifest["n_samples"].eq(TARGET_BEAT_SAMPLES).all()
    assert manifest["annotation_alignment_error_samples"].abs().le(ANNOTATION_TOLERANCE_SAMPLES).all()
    train = manifest["split"].eq("train")
    assert manifest.loc[train, "client_id"].notna().all()
    assert manifest.loc[train, "client_id"].nunique() == N_TRAIN_CLIENTS
    assert manifest["client_id"].notna().all()
    assert manifest.groupby("patient_id")["client_id"].nunique().eq(1).all()
    assert manifest.groupby("client_id")["patient_id"].nunique().eq(1).all()
    assert manifest.groupby("client_id")["split"].nunique().eq(1).all()
    assert manifest.groupby("split")["client_id"].nunique().to_dict() == {
        "train": N_TRAIN_PATIENTS, "dev": N_DEV_PATIENTS, "test": N_TEST_PATIENTS,
    }
    for row in (manifest.itertuples() if validate_arrays else []):
        with np.load(OUTPUT_ROOT / row.segment_path, allow_pickle=False) as data:
            beat = data["signal"]
            assert beat.shape == (TARGET_BEAT_SAMPLES,) and beat.dtype == np.float32
            assert np.isfinite(beat).all() and beat.min() >= 0 and beat.max() <= 1
            assert str(data["label"]) == row.label and int(data["label_id"]) == row.label_id
    return True

RECORD_STATS = []

def process_record(record_path, split_name):
    """Filter a record, attach its WFDB beat labels, score SQI, and save labelled beats."""
    cache_dir = OUTPUT_ROOT / "record_checkpoints" / split_name / record_path.parent.name
    cache_csv = cache_dir / f"{record_path.name}.csv"
    cache_stats = cache_dir / f"{record_path.name}.json"
    if cache_csv.is_file() and cache_stats.is_file():
        cached = pd.read_csv(cache_csv)
        RECORD_STATS.append(json.loads(cache_stats.read_text()))
        return cached.to_dict("records")
    raw_signal, fs = read_and_resample_record(record_path)
    filtered = filter_ecg(raw_signal, fs)
    main_peaks = detect_r_peaks(filtered, fs, method="neurokit")
    second_peaks = detect_r_peaks(filtered, fs, method="pantompkins1985")
    annotation = wfdb.rdann(str(record_path), "atr")
    matches = match_peaks_to_annotations(main_peaks, np.asarray(annotation.sample), np.asarray(annotation.symbol, dtype=str))
    patient_id = record_path.parent.name
    rows = []
    stats = {"patient_id": patient_id, "record_id": record_path.name, "split": split_name,
             "detected_peaks": len(main_peaks), "matched_peaks": len(matches),
             "unmatched_detections": len(main_peaks) - len(matches),
             "rejected_boundary": 0, "rejected_invalid_or_constant": 0,
             "rejected_sqi": 0, "accepted": 0}
    for label in LABEL_TO_ID:
        count = int(np.sum(np.asarray(annotation.symbol) == label))
        matched = sum(value[0] == label for value in matches.values())
        stats[f"annotated_{label}"] = count
        stats[f"unmatched_annotations_{label}"] = count - matched
        stats[f"accepted_{label}"] = 0
    half_window = fs // 2

    for peak_number, peak_index in enumerate(main_peaks):
        if peak_number not in matches:
            continue
        label, annotation_index, alignment_error = matches[peak_number]
        start, end = int(peak_index) - half_window, int(peak_index) + half_window
        if start < 0 or end > len(raw_signal):
            stats["rejected_boundary"] += 1
            continue
        raw_window = raw_signal[start:end]
        beat = make_beat(filtered, int(peak_index), fs)
        if beat.size == 0 or not np.isfinite(raw_window).all() or np.ptp(raw_window) <= np.finfo(float).eps:
            stats["rejected_invalid_or_constant"] += 1
            continue
        main_local = main_peaks[(main_peaks >= start) & (main_peaks < end)]
        second_local = second_peaks[(second_peaks >= start) & (second_peaks < end)]
        try:
            quality, features = zhao_sqi_for_beat(raw_window, main_local, second_local, fs)
        except ValueError:
            stats["rejected_sqi"] += 1
            continue
        stats["accepted"] += 1
        stats[f"accepted_{label}"] += 1
        relative_path = Path(split_name) / patient_id / f"{record_path.name}_beat_{peak_number:05d}.npz"
        output_path = OUTPUT_ROOT / relative_path
        output_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(output_path, signal=beat, label=label, label_id=np.int64(LABEL_TO_ID[label]), quality=quality, **features)
        rows.append({
            "patient_id": patient_id, "record_id": record_path.name, "split": split_name,
            "peak_index_200hz": int(peak_index), "annotation_index_200hz": annotation_index,
            "annotation_alignment_error_samples": alignment_error, "sampling_rate_hz": fs,
            "n_samples": int(beat.size), "label": label, "label_id": LABEL_TO_ID[label],
            "quality_zhao_fuzzy": quality, **features,
            "raw_peak_to_peak": float(np.ptp(raw_window)),
            "raw_max_abs": float(np.max(np.abs(raw_window))), "segment_path": str(relative_path),
        })
    cache_dir.mkdir(parents=True, exist_ok=True)
    if rows:
        pd.DataFrame(rows).to_csv(cache_csv.with_suffix(".csv.tmp"), index=False)
        cache_csv.with_suffix(".csv.tmp").replace(cache_csv)
        cache_stats.write_text(json.dumps(stats, indent=2))
    RECORD_STATS.append(stats)
    return rows

def assign_patient_clients(manifest, seed=RANDOM_SEED):
    """Um cliente por paciente; os splits dev/test são exclusivos de avaliação."""
    splits = assign_patient_splits(get_patient_directories(), seed=seed)
    ordered = [(split, patient) for split in ("train", "dev", "test") for patient in sorted(splits[split])]
    mapping = {patient: f"client_{i:02d}" for i, (_, patient) in enumerate(ordered)}
    split_mapping = {patient: split for split, patient in ordered}
    result = manifest.copy()
    expected_split = result["patient_id"].map(split_mapping)
    if expected_split.isna().any() or not expected_split.eq(result["split"]).all():
        raise ValueError("Paciente desconhecido ou split incompatível com a divisão por paciente.")
    result["client_id"] = result["patient_id"].map(mapping).astype("string")
    return result

def run_preparation(limit_records_per_patient=1, seed=RANDOM_SEED):
    """Run preprocessing, export the labelled manifest, clients and held-out manifests."""
    RECORD_STATS.clear()
    patients = get_patient_directories()
    patient_splits = assign_patient_splits(patients, seed=seed)
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    frames = []
    for patient in patients:
        split = next(name for name, ids in patient_splits.items() if patient.name in ids)
        records = list_records(patient)
        if limit_records_per_patient is not None:
            records = records[:limit_records_per_patient]
        for record in records:
            cache_dir = OUTPUT_ROOT / "record_checkpoints" / split / patient.name
            cached_csv = cache_dir / f"{record.name}.csv"
            cached_stats = cache_dir / f"{record.name}.json"
            if cached_csv.is_file() and cached_stats.is_file():
                frames.append(pd.read_csv(cached_csv))
                RECORD_STATS.append(json.loads(cached_stats.read_text()))
            else:
                prepared_rows = process_record(record, split)
                if prepared_rows:
                    frames.append(pd.DataFrame(prepared_rows))
    manifest = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    del frames
    if manifest.empty:
        raise RuntimeError("No labelled beats were prepared; check WFDB annotations and match tolerance.")
    manifest = assign_patient_clients(manifest, seed=seed)
    validate_prepared_dataset(manifest)
    manifest.to_csv(OUTPUT_ROOT / "manifest.csv", index=False)
    pd.DataFrame(RECORD_STATS).to_csv(OUTPUT_ROOT / "record_processing_stats.csv", index=False)
    class_counts = pd.crosstab(manifest["split"], manifest["label"]).reindex(index=["train", "dev", "test"], columns=list(LABEL_TO_ID), fill_value=0)
    class_counts.to_csv(OUTPUT_ROOT / "split_class_counts.csv")
    client_counts = pd.crosstab(manifest["client_id"], manifest["label"]).reindex(index=[f"client_{i:02d}" for i in range(N_TRAIN_CLIENTS)], columns=list(LABEL_TO_ID), fill_value=0)
    client_counts.to_csv(OUTPUT_ROOT / "client_class_counts.csv")
    clients_dir = OUTPUT_ROOT / "clients"
    clients_dir.mkdir(parents=True, exist_ok=True)
    train = manifest[manifest["split"] == "train"]
    for client_id, client_rows in train.groupby("client_id", sort=True):
        client_rows.to_csv(clients_dir / f"{client_id}.csv", index=False)
    evaluation_clients_dir = OUTPUT_ROOT / "evaluation_clients"
    evaluation_clients_dir.mkdir(parents=True, exist_ok=True)
    for client_id, client_rows in manifest[manifest["split"] != "train"].groupby("client_id", sort=True):
        client_rows.to_csv(evaluation_clients_dir / f"{client_id}.csv", index=False)
    patient_counts = manifest.groupby(["client_id", "patient_id", "split", "label"]).size().unstack(fill_value=0).reindex(columns=list(LABEL_TO_ID), fill_value=0)
    patient_counts.to_csv(OUTPUT_ROOT / "patient_client_class_counts.csv")
    for split in ("dev", "test"):
        manifest[manifest["split"] == split].to_csv(OUTPUT_ROOT / f"{split}_evaluation_manifest.csv", index=False)
    config = {
        "source_fs_hz": SOURCE_FS, "target_fs_hz": TARGET_FS, "beat_length_samples": TARGET_BEAT_SAMPLES,
        "patient_split": {name: len(ids) for name, ids in patient_splits.items()},
        "client_partition": "one_client_per_patient", "seed": seed,
        "train_clients": N_TRAIN_PATIENTS, "dev_clients": N_DEV_PATIENTS, "test_clients": N_TEST_PATIENTS,
        "patient_to_client": manifest.drop_duplicates("patient_id").set_index("patient_id")["client_id"].to_dict(),
        "evaluation_policy": "Only train clients participate in federated training; dev/test clients are held out",
        "annotation_tolerance_samples": ANNOTATION_TOLERANCE_SAMPLES,
        "annotation_matching": "minimum-error candidate first, one-to-one; deterministic ties",
        "limit_records_per_patient": limit_records_per_patient,
        "normalization": "per-beat min-max", "window_seconds": 1,
        "interpolation": "linear, 200 to 500 representation points; not native 200-Hz samples",
        "filters": {"highpass_hz": 0.5, "order": 5, "notch_hz": 60, "notch_q": 35},
        "patient_ids": {key: sorted(value) for key, value in patient_splits.items()},
        "quality_policy": "retain all finite SQI grades; reject invalid or constant windows",
        "Q_policy": "retain as unclassifiable beat, not automatically an arrhythmia",
        "versions": {name: __import__("importlib.metadata", fromlist=["version"]).version(name) for name in ["numpy", "scipy", "pandas", "wfdb", "neurokit2"]},
        "sqi_reference": "https://doi.org/10.3389/fphys.2018.00727",
        "sqi_deviation": "NeuroKit neurokit/pantompkins1985 detector pair and 1-s beat windows; published detector pair and longer evaluation windows differ.",
    }
    (OUTPUT_ROOT / "preparation_config.json").write_text(json.dumps(config, indent=2))
    return manifest

In [ ]:
# Validate annotation alignment and published SQI membership/fusion on one record without saving outputs.
example_record = list_records(patients[0])[0]
raw_signal, fs = read_and_resample_record(example_record)
filtered = filter_ecg(raw_signal, fs)
main_peaks = detect_r_peaks(filtered, fs, method="neurokit")
second_peaks = detect_r_peaks(filtered, fs, method="pantompkins1985")
ann = wfdb.rdann(str(example_record), "atr")
matches = match_peaks_to_annotations(main_peaks, np.asarray(ann.sample), np.asarray(ann.symbol, dtype=str))
assert len(matches) > 0 and set(label for label, _, _ in matches.values()).issubset(LABEL_TO_ID)
assert len(matches) / len(main_peaks) > 0.95, "Low annotation-to-detection match rate; inspect tolerance or detector."
complete_matches = [
    (index, label_data) for index, label_data in matches.items()
    if fs // 2 <= main_peaks[index] <= len(raw_signal) - fs // 2
]
peak_number, _ = complete_matches[len(complete_matches) // 2]
peak_index = int(main_peaks[peak_number])
window = raw_signal[peak_index - fs // 2:peak_index + fs // 2]
main_local = main_peaks[(main_peaks >= peak_index - fs // 2) & (main_peaks < peak_index + fs // 2)]
second_local = second_peaks[(second_peaks >= peak_index - fs // 2) & (second_peaks < peak_index + fs // 2)]
assert window.size == fs
quality, features = zhao_sqi_for_beat(window, main_local, second_local, fs)
assert quality in {"Excellent", "Barely acceptable", "Unacceptable"}
print(f"Matched annotations: {len(matches):,}/{len(main_peaks):,} ({len(matches) / len(main_peaks):.1%})")
print("Matched classes:", pd.Series([x[0] for x in matches.values()]).value_counts().to_dict())
print("One-beat SQI:", quality, features)

# Exercise the complete record writer in a temporary directory, then test the patient-to-client assignment.
import tempfile
from collections import Counter
configured_output_root = OUTPUT_ROOT
try:
    with tempfile.TemporaryDirectory() as temporary_output:
        OUTPUT_ROOT = Path(temporary_output)
        record_rows = process_record(example_record, "train")
        assert record_rows and set(row["label"] for row in record_rows).issubset(LABEL_TO_ID)
        assert all(np.isfinite([row["q_sqi_percent"], row["p_sqi"], row["k_sqi"], row["bas_sqi_percent"]]).all() for row in record_rows)
        client_test = assign_patient_clients(pd.DataFrame(record_rows), seed=RANDOM_SEED)
        assert client_test["client_id"].notna().all()
        assert client_test["client_id"].nunique() == 1
        sample_path = OUTPUT_ROOT / record_rows[0]["segment_path"]
        with np.load(sample_path) as beat_file:
            assert beat_file["signal"].shape == (TARGET_BEAT_SAMPLES,)
            assert str(beat_file["label"]) == record_rows[0]["label"]
        print(f"End-to-end record rows: {len(record_rows):,}; client IDs: {client_test['client_id'].nunique()}")
        print("Client label distributions:")
        display(client_test.groupby(["client_id", "label"]).size().unstack(fill_value=0))
finally:
    OUTPUT_ROOT = configured_output_root


## Executar a pipeline supervisionada

Cada paciente é um cliente fixo: 16 train, 2 dev e 2 test. A separação por paciente é preservada, sem redistribuir batimentos entre clientes. Apenas os clientes train participam no treino federado.

Os manifestos de treino ficam em `clients/`; os quatro manifestos por cliente de avaliação ficam em `evaluation_clients/`, além dos manifestos globais dev/test. O `client_id` identifica o paciente em todos os splits; o campo `split` define a sua utilização.

O dataset já está preparado. `RUN_PREPARATION = False` evita repetir a execução automaticamente. Os sinais e os splits não mudaram com a correção da atribuição dos clientes.


In [ ]:
if RUN_PREPARATION:
    manifest = run_preparation(limit_records_per_patient=LIMIT_RECORDS_PER_PATIENT, seed=RANDOM_SEED)
    display(manifest.groupby(["split", "label"]).size().rename("n_beats").to_frame())
    display(manifest[manifest["split"] == "train"].groupby(["client_id", "label"]).size().unstack(fill_value=0))
    print(f"Prepared labelled beats: {len(manifest):,}")
    print(f"Manifests and beat files: {OUTPUT_ROOT}")
else:
    print("Execution remains disabled. Set RUN_PREPARATION = True after checking disk space.")

In [ ]:
# Inspeção visual de um exemplo preparado por classe, quando a preparação é ativada.
if RUN_PREPARATION:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(len(LABEL_TO_ID), 1, figsize=(10, 8), constrained_layout=True)
    for ax, label in zip(axes, LABEL_TO_ID):
        examples = manifest[manifest["label"] == label]
        if examples.empty:
            ax.set_title(f"{label}: sem exemplos preparados")
            continue
        row = examples.iloc[0]
        with np.load(OUTPUT_ROOT / row["segment_path"]) as data:
            ax.plot(np.linspace(-0.5, 0.5, TARGET_BEAT_SAMPLES, endpoint=False), data["signal"])
        ax.set_title(f"{label} | {row['quality_zhao_fuzzy']} | {row['record_id']}")
        ax.set_xlabel("Tempo relativo aproximado ao pico R (s)")
    plt.show()


## Contagem de S e V por split

Anotações originais, antes da segmentação e das rejeições, usando a divisão por paciente do notebook. Os resultados abaixo correspondem aos registos locais contados nesta revisão; a célula seguinte recalcula e exporta as tabelas.

### Todos os registos locais

| Split | S | V | S + V |
| --- | ---: | ---: | ---: |
| Train | 4 993 | 40 429 | 45 422 |
| Dev | 6 510 | 12 | 6 522 |
| Test | 36 386 | 5 507 | 41 893 |

### Limite de 1 registo por paciente

| Split | S | V | S + V |
| --- | ---: | ---: | ---: |
| Train | 65 | 280 | 345 |
| Dev | 217 | 1 | 218 |
| Test | 1 030 | 10 | 1 040 |

Dev contém apenas 12 anotações V na amostra local completa. Os totais utilizáveis podem diminuir após a preparação.


In [ ]:
# Contagens das anotações originais: antes da deteção, segmentação e rejeições.
# Não lê os sinais ECG nem escreve batimentos preparados.
from collections import Counter

annotation_count_rows = []
for patient in patients:
    split = next(name for name, ids in splits.items() if patient.name in ids)
    for record_number, record in enumerate(list_records(patient)):
        ann = wfdb.rdann(str(record), "atr")
        counts = Counter(ann.symbol)
        annotation_count_rows.append({
            "patient_id": patient.name, "record_id": record.name, "split": split,
            "within_configured_limit": LIMIT_RECORDS_PER_PATIENT is None or record_number < LIMIT_RECORDS_PER_PATIENT,
            **{label: counts[label] for label in ("N", "S", "V", "Q")},
        })
annotation_counts = pd.DataFrame(annotation_count_rows)
if annotation_counts.empty:
    raise RuntimeError("Não foram encontrados registos para contar anotações.")
annotation_counts["S+V"] = annotation_counts["S"] + annotation_counts["V"]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for scope, selected in (
    ("configured_limit", annotation_counts[annotation_counts["within_configured_limit"]]),
    ("all_local_records", annotation_counts),
):
    summary = selected.groupby("split")[["N", "S", "V", "Q", "S+V"]].sum().reindex(["train", "dev", "test"], fill_value=0)
    summary["n_records"] = selected.groupby("split").size().reindex(summary.index, fill_value=0)
    print(f"Anotações originais — {scope}")
    display(summary)
print("As contagens finais utilizáveis podem ser menores após a preparação e as rejeições.")


Split,S,V,S + V
train,4993,40429,45422
dev,6510,12,6522
test,36386,5507,41893


Split,S,V,S + V
train,65,280,345
dev,217,1,218
test,1030,10,1040


## Contagens originais incorporadas — detalhe por paciente e registo

Conteúdo dos quatro CSV da pasta antiga, preservado neste notebook. `configured_limit` corresponde à contagem histórica de um registo por paciente; `all_local_records` cobre os 981 registos locais. São anotações anteriores às rejeições da preparação. A pasta antiga foi removida.


In [ ]:
# Contagens das anotações originais anteriores à preparação (snapshot local).
# A informação está incorporada no notebook, sem depender da pasta antiga de CSV.
import pandas as pd
from IPython.display import display, HTML
annotation_count_snapshot = {'annotation_counts_all_local_records': [{'split': 'train', 'N': '3049803', 'S': '4993', 'V': '40429', 'Q': '976126', 'S+V': '45422', 'n_records': '781'}, {'split': 'dev', 'N': '338004', 'S': '6510', 'V': '12', 'Q': '145912', 'S+V': '6522', 'n_records': '100'}, {'split': 'test', 'N': '380274', 'S': '36386', 'V': '5507', 'Q': '74175', 'S+V': '41893', 'n_records': '100'}], 'annotation_counts_by_patient': [{'split': 'dev', 'patient_id': 'p00877', 'N': '197007', 'S': '6508', 'V': '5', 'Q': '43073', 'S+V': '6513'}, {'split': 'dev', 'patient_id': 'p09134', 'N': '140997', 'S': '2', 'V': '7', 'Q': '102839', 'S+V': '9'}, {'split': 'test', 'patient_id': 'p01965', 'N': '232878', 'S': '271', 'V': '5240', 'Q': '19767', 'S+V': '5511'}, {'split': 'test', 'patient_id': 'p04071', 'N': '147396', 'S': '36115', 'V': '267', 'Q': '54408', 'S+V': '36382'}, {'split': 'train', 'patient_id': 'p00290', 'N': '271348', 'S': '120', 'V': '345', 'Q': '32576', 'S+V': '465'}, {'split': 'train', 'patient_id': 'p01083', 'N': '116025', 'S': '819', 'V': '4', 'Q': '93605', 'S+V': '823'}, {'split': 'train', 'patient_id': 'p01950', 'N': '267440', 'S': '3', 'V': '349', 'Q': '25236', 'S+V': '352'}, {'split': 'train', 'patient_id': 'p03281', 'N': '246864', 'S': '2', 'V': '3', 'Q': '34188', 'S+V': '5'}, {'split': 'train', 'patient_id': 'p03900', 'N': '255402', 'S': '20', 'V': '10', 'Q': '37888', 'S+V': '30'}, {'split': 'train', 'patient_id': 'p04014', 'N': '105368', 'S': '11', 'V': '0', 'Q': '169907', 'S+V': '11'}, {'split': 'train', 'patient_id': 'p05133', 'N': '146729', 'S': '83', 'V': '13', 'Q': '79124', 'S+V': '96'}, {'split': 'train', 'patient_id': 'p07028', 'N': '202647', 'S': '2125', 'V': '402', 'Q': '34856', 'S+V': '2527'}, {'split': 'train', 'patient_id': 'p07070', 'N': '162937', 'S': '1712', 'V': '3', 'Q': '75079', 'S+V': '1715'}, {'split': 'train', 'patient_id': 'p07179', 'N': '170613', 'S': '7', 'V': '0', 'Q': '71918', 'S+V': '7'}, {'split': 'train', 'patient_id': 'p07743', 'N': '208716', 'S': '10', 'V': '1', 'Q': '65409', 'S+V': '11'}, {'split': 'train', 'patient_id': 'p07929', 'N': '207493', 'S': '27', 'V': '1', 'Q': '30037', 'S+V': '28'}, {'split': 'train', 'patient_id': 'p08689', 'N': '147955', 'S': '0', 'V': '7963', 'Q': '63820', 'S+V': '7963'}, {'split': 'train', 'patient_id': 'p09354', 'N': '159069', 'S': '26', 'V': '55', 'Q': '45679', 'S+V': '81'}, {'split': 'train', 'patient_id': 'p09440', 'N': '228305', 'S': '19', 'V': '0', 'Q': '67746', 'S+V': '19'}, {'split': 'train', 'patient_id': 'p09951', 'N': '152892', 'S': '9', 'V': '31280', 'Q': '49058', 'S+V': '31289'}], 'annotation_counts_by_record': [{'patient_id': 'p00290', 'record_id': 'p00290_s00', 'split': 'train', 'within_configured_limit': 'True', 'N': '5197', 'S': '1', 'V': '11', 'Q': '1228', 'S+V': '12'}, {'patient_id': 'p00290', 'record_id': 'p00290_s01', 'split': 'train', 'within_configured_limit': 'False', 'N': '4980', 'S': '1', 'V': '5', 'Q': '593', 'S+V': '6'}, {'patient_id': 'p00290', 'record_id': 'p00290_s02', 'split': 'train', 'within_configured_limit': 'False', 'N': '4720', 'S': '0', 'V': '12', 'Q': '1129', 'S+V': '12'}, {'patient_id': 'p00290', 'record_id': 'p00290_s03', 'split': 'train', 'within_configured_limit': 'False', 'N': '5627', 'S': '0', 'V': '2', 'Q': '335', 'S+V': '2'}, {'patient_id': 'p00290', 'record_id': 'p00290_s04', 'split': 'train', 'within_configured_limit': 'False', 'N': '5528', 'S': '2', 'V': '4', 'Q': '99', 'S+V': '6'}, {'patient_id': 'p00290', 'record_id': 'p00290_s05', 'split': 'train', 'within_configured_limit': 'False', 'N': '5555', 'S': '1', 'V': '2', 'Q': '82', 'S+V': '3'}, {'patient_id': 'p00290', 'record_id': 'p00290_s06', 'split': 'train', 'within_configured_limit': 'False', 'N': '5985', 'S': '0', 'V': '3', 'Q': '272', 'S+V': '3'}, {'patient_id': 'p00290', 'record_id': 'p00290_s07', 'split': 'train', 'within_configured_limit': 'False', 'N': '6139', 'S': '0', 'V': '0', 'Q': '71', 'S+V': '0'}, {'patient_id': 'p00290', 'record_id': 'p00290_s08', 'split': 'train', 'within_configured_limit': 'False', 'N': '5994', 'S': '7', 'V': '18', 'Q': '957', 'S+V': '25'}, {'patient_id': 'p00290', 'record_id': 'p00290_s09', 'split': 'train', 'within_configured_limit': 'False', 'N': '5323', 'S': '2', 'V': '19', 'Q': '1152', 'S+V': '21'}, {'patient_id': 'p00290', 'record_id': 'p00290_s10', 'split': 'train', 'within_configured_limit': 'False', 'N': '5454', 'S': '1', 'V': '9', 'Q': '684', 'S+V': '10'}, {'patient_id': 'p00290', 'record_id': 'p00290_s11', 'split': 'train', 'within_configured_limit': 'False', 'N': '4967', 'S': '8', 'V': '3', 'Q': '449', 'S+V': '11'}, {'patient_id': 'p00290', 'record_id': 'p00290_s12', 'split': 'train', 'within_configured_limit': 'False', 'N': '5382', 'S': '0', 'V': '6', 'Q': '1163', 'S+V': '6'}, {'patient_id': 'p00290', 'record_id': 'p00290_s13', 'split': 'train', 'within_configured_limit': 'False', 'N': '6012', 'S': '1', 'V': '25', 'Q': '408', 'S+V': '26'}, {'patient_id': 'p00290', 'record_id': 'p00290_s14', 'split': 'train', 'within_configured_limit': 'False', 'N': '5383', 'S': '1', 'V': '11', 'Q': '535', 'S+V': '12'}, {'patient_id': 'p00290', 'record_id': 'p00290_s15', 'split': 'train', 'within_configured_limit': 'False', 'N': '4946', 'S': '1', 'V': '10', 'Q': '668', 'S+V': '11'}, {'patient_id': 'p00290', 'record_id': 'p00290_s16', 'split': 'train', 'within_configured_limit': 'False', 'N': '5560', 'S': '0', 'V': '0', 'Q': '38', 'S+V': '0'}, {'patient_id': 'p00290', 'record_id': 'p00290_s17', 'split': 'train', 'within_configured_limit': 'False', 'N': '6081', 'S': '0', 'V': '0', 'Q': '37', 'S+V': '0'}, {'patient_id': 'p00290', 'record_id': 'p00290_s18', 'split': 'train', 'within_configured_limit': 'False', 'N': '6023', 'S': '1', 'V': '3', 'Q': '41', 'S+V': '4'}, {'patient_id': 'p00290', 'record_id': 'p00290_s19', 'split': 'train', 'within_configured_limit': 'False', 'N': '5732', 'S': '0', 'V': '1', 'Q': '34', 'S+V': '1'}, {'patient_id': 'p00290', 'record_id': 'p00290_s20', 'split': 'train', 'within_configured_limit': 'False', 'N': '5643', 'S': '0', 'V': '1', 'Q': '35', 'S+V': '1'}, {'patient_id': 'p00290', 'record_id': 'p00290_s21', 'split': 'train', 'within_configured_limit': 'False', 'N': '4730', 'S': '0', 'V': '13', 'Q': '1820', 'S+V': '13'}, {'patient_id': 'p00290', 'record_id': 'p00290_s22', 'split': 'train', 'within_configured_limit': 'False', 'N': '5306', 'S': '0', 'V': '8', 'Q': '1565', 'S+V': '8'}, {'patient_id': 'p00290', 'record_id': 'p00290_s23', 'split': 'train', 'within_configured_limit': 'False', 'N': '5713', 'S': '0', 'V': '3', 'Q': '342', 'S+V': '3'}, {'patient_id': 'p00290', 'record_id': 'p00290_s24', 'split': 'train', 'within_configured_limit': 'False', 'N': '5652', 'S': '0', 'V': '11', 'Q': '824', 'S+V': '11'}, {'patient_id': 'p00290', 'record_id': 'p00290_s25', 'split': 'train', 'within_configured_limit': 'False', 'N': '5955', 'S': '0', 'V': '9', 'Q': '28', 'S+V': '9'}, {'patient_id': 'p00290', 'record_id': 'p00290_s26', 'split': 'train', 'within_configured_limit': 'False', 'N': '5514', 'S': '0', 'V': '0', 'Q': '49', 'S+V': '0'}, {'patient_id': 'p00290', 'record_id': 'p00290_s27', 'split': 'train', 'within_configured_limit': 'False', 'N': '5725', 'S': '0', 'V': '12', 'Q': '2304', 'S+V': '12'}, {'patient_id': 'p00290', 'record_id': 'p00290_s28', 'split': 'train', 'within_configured_limit': 'False', 'N': '4893', 'S': '0', 'V': '3', 'Q': '1541', 'S+V': '3'}, {'patient_id': 'p00290', 'record_id': 'p00290_s29', 'split': 'train', 'within_configured_limit': 'False', 'N': '6597', 'S': '24', 'V': '6', 'Q': '1037', 'S+V': '30'}, {'patient_id': 'p00290', 'record_id': 'p00290_s30', 'split': 'train', 'within_configured_limit': 'False', 'N': '5988', 'S': '1', 'V': '0', 'Q': '13', 'S+V': '1'}, {'patient_id': 'p00290', 'record_id': 'p00290_s31', 'split': 'train', 'within_configured_limit': 'False', 'N': '5431', 'S': '0', 'V': '0', 'Q': '95', 'S+V': '0'}, {'patient_id': 'p00290', 'record_id': 'p00290_s32', 'split': 'train', 'within_configured_limit': 'False', 'N': '6216', 'S': '4', 'V': '19', 'Q': '996', 'S+V': '23'}, {'patient_id': 'p00290', 'record_id': 'p00290_s33', 'split': 'train', 'within_configured_limit': 'False', 'N': '5430', 'S': '21', 'V': '5', 'Q': '774', 'S+V': '26'}, {'patient_id': 'p00290', 'record_id': 'p00290_s34', 'split': 'train', 'within_configured_limit': 'False', 'N': '5572', 'S': '2', 'V': '2', 'Q': '852', 'S+V': '4'}, {'patient_id': 'p00290', 'record_id': 'p00290_s35', 'split': 'train', 'within_configured_limit': 'False', 'N': '5266', 'S': '1', 'V': '1', 'Q': '28', 'S+V': '2'}, {'patient_id': 'p00290', 'record_id': 'p00290_s36', 'split': 'train', 'within_configured_limit': 'False', 'N': '5624', 'S': '2', 'V': '9', 'Q': '26', 'S+V': '11'}, {'patient_id': 'p00290', 'record_id': 'p00290_s37', 'split': 'train', 'within_configured_limit': 'False', 'N': '6063', 'S': '4', 'V': '14', 'Q': '1482', 'S+V': '18'}, {'patient_id': 'p00290', 'record_id': 'p00290_s38', 'split': 'train', 'within_configured_limit': 'False', 'N': '5667', 'S': '1', 'V': '14', 'Q': '1288', 'S+V': '15'}, {'patient_id': 'p00290', 'record_id': 'p00290_s39', 'split': 'train', 'within_configured_limit': 'False', 'N': '5759', 'S': '9', 'V': '21', 'Q': '1422', 'S+V': '30'}, {'patient_id': 'p00290', 'record_id': 'p00290_s40', 'split': 'train', 'within_configured_limit': 'False', 'N': '5395', 'S': '2', 'V': '17', 'Q': '1150', 'S+V': '19'}, {'patient_id': 'p00290', 'record_id': 'p00290_s41', 'split': 'train', 'within_configured_limit': 'False', 'N': '5501', 'S': '1', 'V': '7', 'Q': '763', 'S+V': '8'}, {'patient_id': 'p00290', 'record_id': 'p00290_s42', 'split': 'train', 'within_configured_limit': 'False', 'N': '5050', 'S': '0', 'V': '0', 'Q': '40', 'S+V': '0'}, {'patient_id': 'p00290', 'record_id': 'p00290_s43', 'split': 'train', 'within_configured_limit': 'False', 'N': '4892', 'S': '1', 'V': '5', 'Q': '72', 'S+V': '6'}, {'patient_id': 'p00290', 'record_id': 'p00290_s44', 'split': 'train', 'within_configured_limit': 'False', 'N': '4624', 'S': '1', 'V': '8', 'Q': '1800', 'S+V': '9'}, {'patient_id': 'p00290', 'record_id': 'p00290_s45', 'split': 'train', 'within_configured_limit': 'False', 'N': '5495', 'S': '4', 'V': '1', 'Q': '396', 'S+V': '5'}, {'patient_id': 'p00290', 'record_id': 'p00290_s46', 'split': 'train', 'within_configured_limit': 'False', 'N': '5488', 'S': '11', 'V': '8', 'Q': '868', 'S+V': '19'}, {'patient_id': 'p00290', 'record_id': 'p00290_s47', 'split': 'train', 'within_configured_limit': 'False', 'N': '5680', 'S': '4', 'V': '4', 'Q': '57', 'S+V': '8'}, {'patient_id': 'p00290', 'record_id': 'p00290_s48', 'split': 'train', 'within_configured_limit': 'False', 'N': '5891', 'S': '0', 'V': '0', 'Q': '22', 'S+V': '0'}, {'patient_id': 'p00290', 'record_id': 'p00290_s49', 'split': 'train', 'within_configured_limit': 'False', 'N': '0', 'S': '0', 'V': '0', 'Q': '912', 'S+V': '0'}, {'patient_id': 'p00877', 'record_id': 'p00877_s00', 'split': 'dev', 'within_configured_limit': 'True', 'N': '4510', 'S': '217', 'V': '1', 'Q': '504', 'S+V': '218'}, {'patient_id': 'p00877', 'record_id': 'p00877_s01', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4251', 'S': '200', 'V': '0', 'Q': '814', 'S+V': '200'}, {'patient_id': 'p00877', 'record_id': 'p00877_s02', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4374', 'S': '161', 'V': '0', 'Q': '22', 'S+V': '161'}, {'patient_id': 'p00877', 'record_id': 'p00877_s03', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4265', 'S': '151', 'V': '0', 'Q': '55', 'S+V': '151'}, {'patient_id': 'p00877', 'record_id': 'p00877_s04', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4369', 'S': '120', 'V': '0', 'Q': '147', 'S+V': '120'}, {'patient_id': 'p00877', 'record_id': 'p00877_s05', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4495', 'S': '65', 'V': '0', 'Q': '1137', 'S+V': '65'}, {'patient_id': 'p00877', 'record_id': 'p00877_s06', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4571', 'S': '76', 'V': '0', 'Q': '2855', 'S+V': '76'}, {'patient_id': 'p00877', 'record_id': 'p00877_s07', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4585', 'S': '290', 'V': '0', 'Q': '323', 'S+V': '290'}, {'patient_id': 'p00877', 'record_id': 'p00877_s08', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4411', 'S': '78', 'V': '0', 'Q': '398', 'S+V': '78'}, {'patient_id': 'p00877', 'record_id': 'p00877_s09', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4149', 'S': '116', 'V': '0', 'Q': '973', 'S+V': '116'}, {'patient_id': 'p00877', 'record_id': 'p00877_s10', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4154', 'S': '107', 'V': '0', 'Q': '538', 'S+V': '107'}, {'patient_id': 'p00877', 'record_id': 'p00877_s11', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4247', 'S': '60', 'V': '0', 'Q': '6', 'S+V': '60'}, {'patient_id': 'p00877', 'record_id': 'p00877_s12', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4188', 'S': '55', 'V': '0', 'Q': '35', 'S+V': '55'}, {'patient_id': 'p00877', 'record_id': 'p00877_s13', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4308', 'S': '208', 'V': '0', 'Q': '6', 'S+V': '208'}, {'patient_id': 'p00877', 'record_id': 'p00877_s14', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4216', 'S': '111', 'V': '0', 'Q': '16', 'S+V': '111'}, {'patient_id': 'p00877', 'record_id': 'p00877_s15', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4146', 'S': '63', 'V': '0', 'Q': '14', 'S+V': '63'}, {'patient_id': 'p00877', 'record_id': 'p00877_s16', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4239', 'S': '153', 'V': '0', 'Q': '1128', 'S+V': '153'}, {'patient_id': 'p00877', 'record_id': 'p00877_s17', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4327', 'S': '173', 'V': '0', 'Q': '1033', 'S+V': '173'}, {'patient_id': 'p00877', 'record_id': 'p00877_s18', 'split': 'dev', 'within_configured_limit': 'False', 'N': '2643', 'S': '79', 'V': '0', 'Q': '3083', 'S+V': '79'}, {'patient_id': 'p00877', 'record_id': 'p00877_s19', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4712', 'S': '171', 'V': '1', 'Q': '211', 'S+V': '172'}, {'patient_id': 'p00877', 'record_id': 'p00877_s20', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4700', 'S': '189', 'V': '0', 'Q': '279', 'S+V': '189'}, {'patient_id': 'p00877', 'record_id': 'p00877_s21', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4801', 'S': '148', 'V': '0', 'Q': '416', 'S+V': '148'}, {'patient_id': 'p00877', 'record_id': 'p00877_s22', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4563', 'S': '211', 'V': '0', 'Q': '288', 'S+V': '211'}, {'patient_id': 'p00877', 'record_id': 'p00877_s23', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4051', 'S': '60', 'V': '0', 'Q': '18', 'S+V': '60'}, {'patient_id': 'p00877', 'record_id': 'p00877_s24', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3900', 'S': '64', 'V': '0', 'Q': '39', 'S+V': '64'}, {'patient_id': 'p00877', 'record_id': 'p00877_s25', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4243', 'S': '82', 'V': '0', 'Q': '975', 'S+V': '82'}, {'patient_id': 'p00877', 'record_id': 'p00877_s26', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4096', 'S': '162', 'V': '0', 'Q': '1833', 'S+V': '162'}, {'patient_id': 'p00877', 'record_id': 'p00877_s27', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4640', 'S': '310', 'V': '0', 'Q': '469', 'S+V': '310'}, {'patient_id': 'p00877', 'record_id': 'p00877_s28', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3833', 'S': '57', 'V': '1', 'Q': '538', 'S+V': '58'}, {'patient_id': 'p00877', 'record_id': 'p00877_s29', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3956', 'S': '38', 'V': '0', 'Q': '27', 'S+V': '38'}, {'patient_id': 'p00877', 'record_id': 'p00877_s30', 'split': 'dev', 'within_configured_limit': 'False', 'N': '2968', 'S': '174', 'V': '1', 'Q': '2364', 'S+V': '175'}, {'patient_id': 'p00877', 'record_id': 'p00877_s31', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3340', 'S': '133', 'V': '0', 'Q': '1770', 'S+V': '133'}, {'patient_id': 'p00877', 'record_id': 'p00877_s32', 'split': 'dev', 'within_configured_limit': 'False', 'N': '2295', 'S': '123', 'V': '0', 'Q': '2991', 'S+V': '123'}, {'patient_id': 'p00877', 'record_id': 'p00877_s33', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4019', 'S': '58', 'V': '0', 'Q': '67', 'S+V': '58'}, {'patient_id': 'p00877', 'record_id': 'p00877_s34', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4169', 'S': '72', 'V': '0', 'Q': '28', 'S+V': '72'}, {'patient_id': 'p00877', 'record_id': 'p00877_s35', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3021', 'S': '112', 'V': '0', 'Q': '2126', 'S+V': '112'}, {'patient_id': 'p00877', 'record_id': 'p00877_s36', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3713', 'S': '193', 'V': '0', 'Q': '1669', 'S+V': '193'}, {'patient_id': 'p00877', 'record_id': 'p00877_s37', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4561', 'S': '308', 'V': '0', 'Q': '385', 'S+V': '308'}, {'patient_id': 'p00877', 'record_id': 'p00877_s38', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4439', 'S': '224', 'V': '0', 'Q': '310', 'S+V': '224'}, {'patient_id': 'p00877', 'record_id': 'p00877_s39', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4342', 'S': '170', 'V': '0', 'Q': '263', 'S+V': '170'}, {'patient_id': 'p00877', 'record_id': 'p00877_s40', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3464', 'S': '96', 'V': '0', 'Q': '934', 'S+V': '96'}, {'patient_id': 'p00877', 'record_id': 'p00877_s41', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3998', 'S': '54', 'V': '0', 'Q': '6', 'S+V': '54'}, {'patient_id': 'p00877', 'record_id': 'p00877_s42', 'split': 'dev', 'within_configured_limit': 'False', 'N': '2985', 'S': '100', 'V': '0', 'Q': '2705', 'S+V': '100'}, {'patient_id': 'p00877', 'record_id': 'p00877_s43', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3110', 'S': '98', 'V': '0', 'Q': '1936', 'S+V': '98'}, {'patient_id': 'p00877', 'record_id': 'p00877_s44', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4003', 'S': '272', 'V': '1', 'Q': '1190', 'S+V': '273'}, {'patient_id': 'p00877', 'record_id': 'p00877_s45', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4206', 'S': '69', 'V': '0', 'Q': '30', 'S+V': '69'}, {'patient_id': 'p00877', 'record_id': 'p00877_s46', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3601', 'S': '30', 'V': '0', 'Q': '453', 'S+V': '30'}, {'patient_id': 'p00877', 'record_id': 'p00877_s47', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3296', 'S': '83', 'V': '0', 'Q': '2015', 'S+V': '83'}, {'patient_id': 'p00877', 'record_id': 'p00877_s48', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3534', 'S': '164', 'V': '0', 'Q': '1362', 'S+V': '164'}, {'patient_id': 'p00877', 'record_id': 'p00877_s49', 'split': 'dev', 'within_configured_limit': 'False', 'N': '0', 'S': '0', 'V': '0', 'Q': '2289', 'S+V': '0'}, {'patient_id': 'p01083', 'record_id': 'p01083_s00', 'split': 'train', 'within_configured_limit': 'True', 'N': '3879', 'S': '5', 'V': '0', 'Q': '607', 'S+V': '5'}, {'patient_id': 'p01083', 'record_id': 'p01083_s01', 'split': 'train', 'within_configured_limit': 'False', 'N': '382', 'S': '3', 'V': '0', 'Q': '4726', 'S+V': '3'}, {'patient_id': 'p01083', 'record_id': 'p01083_s02', 'split': 'train', 'within_configured_limit': 'False', 'N': '2570', 'S': '38', 'V': '0', 'Q': '1799', 'S+V': '38'}, {'patient_id': 'p01083', 'record_id': 'p01083_s03', 'split': 'train', 'within_configured_limit': 'False', 'N': '3661', 'S': '12', 'V': '0', 'Q': '161', 'S+V': '12'}, {'patient_id': 'p01083', 'record_id': 'p01083_s04', 'split': 'train', 'within_configured_limit': 'False', 'N': '3465', 'S': '4', 'V': '0', 'Q': '511', 'S+V': '4'}, {'patient_id': 'p01083', 'record_id': 'p01083_s05', 'split': 'train', 'within_configured_limit': 'False', 'N': '2971', 'S': '6', 'V': '0', 'Q': '1529', 'S+V': '6'}, {'patient_id': 'p01083', 'record_id': 'p01083_s06', 'split': 'train', 'within_configured_limit': 'False', 'N': '1721', 'S': '5', 'V': '0', 'Q': '2772', 'S+V': '5'}, {'patient_id': 'p01083', 'record_id': 'p01083_s07', 'split': 'train', 'within_configured_limit': 'False', 'N': '2295', 'S': '5', 'V': '0', 'Q': '2184', 'S+V': '5'}, {'patient_id': 'p01083', 'record_id': 'p01083_s08', 'split': 'train', 'within_configured_limit': 'False', 'N': '3187', 'S': '7', 'V': '0', 'Q': '1264', 'S+V': '7'}, {'patient_id': 'p01083', 'record_id': 'p01083_s09', 'split': 'train', 'within_configured_limit': 'False', 'N': '735', 'S': '2', 'V': '0', 'Q': '4573', 'S+V': '2'}, {'patient_id': 'p01083', 'record_id': 'p01083_s10', 'split': 'train', 'within_configured_limit': 'False', 'N': '1993', 'S': '3', 'V': '0', 'Q': '2599', 'S+V': '3'}, {'patient_id': 'p01083', 'record_id': 'p01083_s11', 'split': 'train', 'within_configured_limit': 'False', 'N': '3768', 'S': '28', 'V': '1', 'Q': '105', 'S+V': '29'}, {'patient_id': 'p01083', 'record_id': 'p01083_s12', 'split': 'train', 'within_configured_limit': 'False', 'N': '2507', 'S': '14', 'V': '0', 'Q': '1878', 'S+V': '14'}, {'patient_id': 'p01083', 'record_id': 'p01083_s13', 'split': 'train', 'within_configured_limit': 'False', 'N': '2430', 'S': '45', 'V': '0', 'Q': '2624', 'S+V': '45'}, {'patient_id': 'p01083', 'record_id': 'p01083_s14', 'split': 'train', 'within_configured_limit': 'False', 'N': '3517', 'S': '13', 'V': '0', 'Q': '80', 'S+V': '13'}, {'patient_id': 'p01083', 'record_id': 'p01083_s15', 'split': 'train', 'within_configured_limit': 'False', 'N': '3315', 'S': '18', 'V': '0', 'Q': '32', 'S+V': '18'}, {'patient_id': 'p01083', 'record_id': 'p01083_s16', 'split': 'train', 'within_configured_limit': 'False', 'N': '3598', 'S': '9', 'V': '0', 'Q': '129', 'S+V': '9'}, {'patient_id': 'p01083', 'record_id': 'p01083_s17', 'split': 'train', 'within_configured_limit': 'False', 'N': '3549', 'S': '8', 'V': '0', 'Q': '205', 'S+V': '8'}, {'patient_id': 'p01083', 'record_id': 'p01083_s18', 'split': 'train', 'within_configured_limit': 'False', 'N': '3162', 'S': '6', 'V': '0', 'Q': '587', 'S+V': '6'}, {'patient_id': 'p01083', 'record_id': 'p01083_s19', 'split': 'train', 'within_configured_limit': 'False', 'N': '2858', 'S': '86', 'V': '0', 'Q': '1603', 'S+V': '86'}, {'patient_id': 'p01083', 'record_id': 'p01083_s20', 'split': 'train', 'within_configured_limit': 'False', 'N': '2215', 'S': '129', 'V': '0', 'Q': '1867', 'S+V': '129'}, {'patient_id': 'p01083', 'record_id': 'p01083_s21', 'split': 'train', 'within_configured_limit': 'False', 'N': '3305', 'S': '11', 'V': '0', 'Q': '1093', 'S+V': '11'}, {'patient_id': 'p01083', 'record_id': 'p01083_s22', 'split': 'train', 'within_configured_limit': 'False', 'N': '1724', 'S': '4', 'V': '0', 'Q': '2931', 'S+V': '4'}, {'patient_id': 'p01083', 'record_id': 'p01083_s23', 'split': 'train', 'within_configured_limit': 'False', 'N': '3348', 'S': '14', 'V': '0', 'Q': '19', 'S+V': '14'}, {'patient_id': 'p01083', 'record_id': 'p01083_s24', 'split': 'train', 'within_configured_limit': 'False', 'N': '3263', 'S': '7', 'V': '0', 'Q': '226', 'S+V': '7'}, {'patient_id': 'p01083', 'record_id': 'p01083_s25', 'split': 'train', 'within_configured_limit': 'False', 'N': '1796', 'S': '10', 'V': '1', 'Q': '2942', 'S+V': '11'}, {'patient_id': 'p01083', 'record_id': 'p01083_s26', 'split': 'train', 'within_configured_limit': 'False', 'N': '2086', 'S': '7', 'V': '0', 'Q': '2184', 'S+V': '7'}, {'patient_id': 'p01083', 'record_id': 'p01083_s27', 'split': 'train', 'within_configured_limit': 'False', 'N': '1651', 'S': '8', 'V': '0', 'Q': '2745', 'S+V': '8'}, {'patient_id': 'p01083', 'record_id': 'p01083_s28', 'split': 'train', 'within_configured_limit': 'False', 'N': '2518', 'S': '5', 'V': '0', 'Q': '1603', 'S+V': '5'}, {'patient_id': 'p01083', 'record_id': 'p01083_s29', 'split': 'train', 'within_configured_limit': 'False', 'N': '3157', 'S': '21', 'V': '0', 'Q': '465', 'S+V': '21'}, {'patient_id': 'p01083', 'record_id': 'p01083_s30', 'split': 'train', 'within_configured_limit': 'False', 'N': '1149', 'S': '27', 'V': '0', 'Q': '3896', 'S+V': '27'}, {'patient_id': 'p01083', 'record_id': 'p01083_s31', 'split': 'train', 'within_configured_limit': 'False', 'N': '2519', 'S': '44', 'V': '0', 'Q': '2146', 'S+V': '44'}, {'patient_id': 'p01083', 'record_id': 'p01083_s32', 'split': 'train', 'within_configured_limit': 'False', 'N': '1977', 'S': '2', 'V': '0', 'Q': '2262', 'S+V': '2'}, {'patient_id': 'p01083', 'record_id': 'p01083_s33', 'split': 'train', 'within_configured_limit': 'False', 'N': '3172', 'S': '14', 'V': '0', 'Q': '415', 'S+V': '14'}, {'patient_id': 'p01083', 'record_id': 'p01083_s34', 'split': 'train', 'within_configured_limit': 'False', 'N': '3357', 'S': '39', 'V': '1', 'Q': '179', 'S+V': '40'}, {'patient_id': 'p01083', 'record_id': 'p01083_s35', 'split': 'train', 'within_configured_limit': 'False', 'N': '2386', 'S': '25', 'V': '0', 'Q': '1969', 'S+V': '25'}, {'patient_id': 'p01083', 'record_id': 'p01083_s36', 'split': 'train', 'within_configured_limit': 'False', 'N': '3451', 'S': '10', 'V': '0', 'Q': '95', 'S+V': '10'}, {'patient_id': 'p01083', 'record_id': 'p01083_s37', 'split': 'train', 'within_configured_limit': 'False', 'N': '409', 'S': '2', 'V': '0', 'Q': '4532', 'S+V': '2'}, {'patient_id': 'p01083', 'record_id': 'p01083_s38', 'split': 'train', 'within_configured_limit': 'False', 'N': '686', 'S': '2', 'V': '0', 'Q': '4286', 'S+V': '2'}, {'patient_id': 'p01083', 'record_id': 'p01083_s39', 'split': 'train', 'within_configured_limit': 'False', 'N': '529', 'S': '1', 'V': '0', 'Q': '4347', 'S+V': '1'}, {'patient_id': 'p01083', 'record_id': 'p01083_s40', 'split': 'train', 'within_configured_limit': 'False', 'N': '309', 'S': '0', 'V': '0', 'Q': '4242', 'S+V': '0'}, {'patient_id': 'p01083', 'record_id': 'p01083_s41', 'split': 'train', 'within_configured_limit': 'False', 'N': '3449', 'S': '14', 'V': '0', 'Q': '450', 'S+V': '14'}, {'patient_id': 'p01083', 'record_id': 'p01083_s42', 'split': 'train', 'within_configured_limit': 'False', 'N': '1049', 'S': '24', 'V': '1', 'Q': '3604', 'S+V': '25'}, {'patient_id': 'p01083', 'record_id': 'p01083_s43', 'split': 'train', 'within_configured_limit': 'False', 'N': '3310', 'S': '44', 'V': '0', 'Q': '248', 'S+V': '44'}, {'patient_id': 'p01083', 'record_id': 'p01083_s44', 'split': 'train', 'within_configured_limit': 'False', 'N': '1680', 'S': '10', 'V': '0', 'Q': '3084', 'S+V': '10'}, {'patient_id': 'p01083', 'record_id': 'p01083_s45', 'split': 'train', 'within_configured_limit': 'False', 'N': '3061', 'S': '6', 'V': '0', 'Q': '500', 'S+V': '6'}, {'patient_id': 'p01083', 'record_id': 'p01083_s46', 'split': 'train', 'within_configured_limit': 'False', 'N': '2906', 'S': '22', 'V': '0', 'Q': '563', 'S+V': '22'}, {'patient_id': 'p01083', 'record_id': 'p01083_s47', 'split': 'train', 'within_configured_limit': 'False', 'N': '0', 'S': '0', 'V': '0', 'Q': '3716', 'S+V': '0'}, {'patient_id': 'p01083', 'record_id': 'p01083_s48', 'split': 'train', 'within_configured_limit': 'False', 'N': '0', 'S': '0', 'V': '0', 'Q': '3427', 'S+V': '0'}, {'patient_id': 'p01083', 'record_id': 'p01083_s49', 'split': 'train', 'within_configured_limit': 'False', 'N': '0', 'S': '0', 'V': '0', 'Q': '3601', 'S+V': '0'}, {'patient_id': 'p01950', 'record_id': 'p01950_s00', 'split': 'train', 'within_configured_limit': 'True', 'N': '4547', 'S': '0', 'V': '5', 'Q': '1821', 'S+V': '5'}, {'patient_id': 'p01950', 'record_id': 'p01950_s01', 'split': 'train', 'within_configured_limit': 'False', 'N': '5104', 'S': '0', 'V': '1', 'Q': '779', 'S+V': '1'}, {'patient_id': 'p01950', 'record_id': 'p01950_s02', 'split': 'train', 'within_configured_limit': 'False', 'N': '5396', 'S': '0', 'V': '0', 'Q': '275', 'S+V': '0'}, {'patient_id': 'p01950', 'record_id': 'p01950_s03', 'split': 'train', 'within_configured_limit': 'False', 'N': '4976', 'S': '0', 'V': '5', 'Q': '518', 'S+V': '5'}, {'patient_id': 'p01950', 'record_id': 'p01950_s04', 'split': 'train', 'within_configured_limit': 'False', 'N': '4327', 'S': '0', 'V': '24', 'Q': '550', 'S+V': '24'}, {'patient_id': 'p01950', 'record_id': 'p01950_s05', 'split': 'train', 'within_configured_limit': 'False', 'N': '2390', 'S': '0', 'V': '0', 'Q': '2469', 'S+V': '0'}, {'patient_id': 'p01950', 'record_id': 'p01950_s06', 'split': 'train', 'within_configured_limit': 'False', 'N': '4408', 'S': '0', 'V': '9', 'Q': '1821', 'S+V': '9'}, {'patient_id': 'p01950', 'record_id': 'p01950_s07', 'split': 'train', 'within_configured_limit': 'False', 'N': '6067', 'S': '0', 'V': '9', 'Q': '91', 'S+V': '9'}, {'patient_id': 'p01950', 'record_id': 'p01950_s08', 'split': 'train', 'within_configured_limit': 'False', 'N': '5335', 'S': '0', 'V': '1', 'Q': '503', 'S+V': '1'}, {'patient_id': 'p01950', 'record_id': 'p01950_s09', 'split': 'train', 'within_configured_limit': 'False', 'N': '6258', 'S': '0', 'V': '7', 'Q': '469', 'S+V': '7'}, {'patient_id': 'p01950', 'record_id': 'p01950_s10', 'split': 'train', 'within_configured_limit': 'False', 'N': '5272', 'S': '0', 'V': '2', 'Q': '594', 'S+V': '2'}, {'patient_id': 'p01950', 'record_id': 'p01950_s11', 'split': 'train', 'within_configured_limit': 'False', 'N': '5137', 'S': '0', 'V': '0', 'Q': '208', 'S+V': '0'}, {'patient_id': 'p01950', 'record_id': 'p01950_s12', 'split': 'train', 'within_configured_limit': 'False', 'N': '5258', 'S': '0', 'V': '1', 'Q': '215', 'S+V': '1'}, {'patient_id': 'p01950', 'record_id': 'p01950_s13', 'split': 'train', 'within_configured_limit': 'False', 'N': '4697', 'S': '0', 'V': '4', 'Q': '95', 'S+V': '4'}, {'patient_id': 'p01950', 'record_id': 'p01950_s14', 'split': 'train', 'within_configured_limit': 'False', 'N': '5125', 'S': '0', 'V': '8', 'Q': '181', 'S+V': '8'}, {'patient_id': 'p01950', 'record_id': 'p01950_s15', 'split': 'train', 'within_configured_limit': 'False', 'N': '2366', 'S': '0', 'V': '0', 'Q': '2415', 'S+V': '0'}, {'patient_id': 'p01950', 'record_id': 'p01950_s16', 'split': 'train', 'within_configured_limit': 'False', 'N': '4488', 'S': '0', 'V': '8', 'Q': '272', 'S+V': '8'}, {'patient_id': 'p01950', 'record_id': 'p01950_s17', 'split': 'train', 'within_configured_limit': 'False', 'N': '5911', 'S': '0', 'V': '1', 'Q': '281', 'S+V': '1'}, {'patient_id': 'p01950', 'record_id': 'p01950_s18', 'split': 'train', 'within_configured_limit': 'False', 'N': '6328', 'S': '0', 'V': '5', 'Q': '470', 'S+V': '5'}, {'patient_id': 'p01950', 'record_id': 'p01950_s19', 'split': 'train', 'within_configured_limit': 'False', 'N': '6155', 'S': '0', 'V': '4', 'Q': '400', 'S+V': '4'}, {'patient_id': 'p01950', 'record_id': 'p01950_s20', 'split': 'train', 'within_configured_limit': 'False', 'N': '6875', 'S': '0', 'V': '21', 'Q': '113', 'S+V': '21'}, {'patient_id': 'p01950', 'record_id': 'p01950_s21', 'split': 'train', 'within_configured_limit': 'False', 'N': '5298', 'S': '0', 'V': '7', 'Q': '585', 'S+V': '7'}, {'patient_id': 'p01950', 'record_id': 'p01950_s22', 'split': 'train', 'within_configured_limit': 'False', 'N': '5211', 'S': '0', 'V': '7', 'Q': '231', 'S+V': '7'}, {'patient_id': 'p01950', 'record_id': 'p01950_s23', 'split': 'train', 'within_configured_limit': 'False', 'N': '5050', 'S': '0', 'V': '10', 'Q': '76', 'S+V': '10'}, {'patient_id': 'p01950', 'record_id': 'p01950_s24', 'split': 'train', 'within_configured_limit': 'False', 'N': '4640', 'S': '0', 'V': '3', 'Q': '332', 'S+V': '3'}, {'patient_id': 'p01950', 'record_id': 'p01950_s25', 'split': 'train', 'within_configured_limit': 'False', 'N': '5403', 'S': '0', 'V': '0', 'Q': '197', 'S+V': '0'}, {'patient_id': 'p01950', 'record_id': 'p01950_s26', 'split': 'train', 'within_configured_limit': 'False', 'N': '5553', 'S': '0', 'V': '2', 'Q': '352', 'S+V': '2'}, {'patient_id': 'p01950', 'record_id': 'p01950_s27', 'split': 'train', 'within_configured_limit': 'False', 'N': '6891', 'S': '0', 'V': '14', 'Q': '338', 'S+V': '14'}, {'patient_id': 'p01950', 'record_id': 'p01950_s28', 'split': 'train', 'within_configured_limit': 'False', 'N': '6584', 'S': '0', 'V': '5', 'Q': '402', 'S+V': '5'}, {'patient_id': 'p01950', 'record_id': 'p01950_s29', 'split': 'train', 'within_configured_limit': 'False', 'N': '7218', 'S': '0', 'V': '31', 'Q': '188', 'S+V': '31'}, {'patient_id': 'p01950', 'record_id': 'p01950_s30', 'split': 'train', 'within_configured_limit': 'False', 'N': '5710', 'S': '0', 'V': '0', 'Q': '81', 'S+V': '0'}, {'patient_id': 'p01950', 'record_id': 'p01950_s31', 'split': 'train', 'within_configured_limit': 'False', 'N': '5353', 'S': '0', 'V': '6', 'Q': '269', 'S+V': '6'}, {'patient_id': 'p01950', 'record_id': 'p01950_s32', 'split': 'train', 'within_configured_limit': 'False', 'N': '4448', 'S': '0', 'V': '16', 'Q': '1726', 'S+V': '16'}, {'patient_id': 'p01950', 'record_id': 'p01950_s33', 'split': 'train', 'within_configured_limit': 'False', 'N': '5164', 'S': '0', 'V': '8', 'Q': '99', 'S+V': '8'}, {'patient_id': 'p01950', 'record_id': 'p01950_s34', 'split': 'train', 'within_configured_limit': 'False', 'N': '7408', 'S': '0', 'V': '13', 'Q': '175', 'S+V': '13'}, {'patient_id': 'p01950', 'record_id': 'p01950_s35', 'split': 'train', 'within_configured_limit': 'False', 'N': '5426', 'S': '0', 'V': '0', 'Q': '341', 'S+V': '0'}, {'patient_id': 'p01950', 'record_id': 'p01950_s36', 'split': 'train', 'within_configured_limit': 'False', 'N': '4476', 'S': '1', 'V': '3', 'Q': '49', 'S+V': '4'}, {'patient_id': 'p01950', 'record_id': 'p01950_s37', 'split': 'train', 'within_configured_limit': 'False', 'N': '5169', 'S': '0', 'V': '4', 'Q': '183', 'S+V': '4'}, {'patient_id': 'p01950', 'record_id': 'p01950_s38', 'split': 'train', 'within_configured_limit': 'False', 'N': '5273', 'S': '0', 'V': '1', 'Q': '75', 'S+V': '1'}, {'patient_id': 'p01950', 'record_id': 'p01950_s39', 'split': 'train', 'within_configured_limit': 'False', 'N': '5259', 'S': '0', 'V': '0', 'Q': '314', 'S+V': '0'}, {'patient_id': 'p01950', 'record_id': 'p01950_s40', 'split': 'train', 'within_configured_limit': 'False', 'N': '5032', 'S': '1', 'V': '0', 'Q': '134', 'S+V': '1'}, {'patient_id': 'p01950', 'record_id': 'p01950_s41', 'split': 'train', 'within_configured_limit': 'False', 'N': '5257', 'S': '0', 'V': '2', 'Q': '245', 'S+V': '2'}, {'patient_id': 'p01950', 'record_id': 'p01950_s42', 'split': 'train', 'within_configured_limit': 'False', 'N': '5843', 'S': '0', 'V': '2', 'Q': '487', 'S+V': '2'}, {'patient_id': 'p01950', 'record_id': 'p01950_s43', 'split': 'train', 'within_configured_limit': 'False', 'N': '6488', 'S': '0', 'V': '26', 'Q': '285', 'S+V': '26'}, {'patient_id': 'p01950', 'record_id': 'p01950_s44', 'split': 'train', 'within_configured_limit': 'False', 'N': '6050', 'S': '0', 'V': '17', 'Q': '424', 'S+V': '17'}, {'patient_id': 'p01950', 'record_id': 'p01950_s45', 'split': 'train', 'within_configured_limit': 'False', 'N': '5769', 'S': '0', 'V': '3', 'Q': '475', 'S+V': '3'}, {'patient_id': 'p01950', 'record_id': 'p01950_s46', 'split': 'train', 'within_configured_limit': 'False', 'N': '5012', 'S': '0', 'V': '3', 'Q': '758', 'S+V': '3'}, {'patient_id': 'p01950', 'record_id': 'p01950_s47', 'split': 'train', 'within_configured_limit': 'False', 'N': '4691', 'S': '1', 'V': '11', 'Q': '157', 'S+V': '12'}, {'patient_id': 'p01950', 'record_id': 'p01950_s48', 'split': 'train', 'within_configured_limit': 'False', 'N': '5378', 'S': '0', 'V': '15', 'Q': '1543', 'S+V': '15'}, {'patient_id': 'p01950', 'record_id': 'p01950_s49', 'split': 'train', 'within_configured_limit': 'False', 'N': '5966', 'S': '0', 'V': '25', 'Q': '175', 'S+V': '25'}, {'patient_id': 'p01965', 'record_id': 'p01965_s00', 'split': 'test', 'within_configured_limit': 'True', 'N': '5121', 'S': '1', 'V': '7', 'Q': '18', 'S+V': '8'}, {'patient_id': 'p01965', 'record_id': 'p01965_s01', 'split': 'test', 'within_configured_limit': 'False', 'N': '4853', 'S': '1', 'V': '0', 'Q': '251', 'S+V': '1'}, {'patient_id': 'p01965', 'record_id': 'p01965_s02', 'split': 'test', 'within_configured_limit': 'False', 'N': '4409', 'S': '0', 'V': '1', 'Q': '1168', 'S+V': '1'}, {'patient_id': 'p01965', 'record_id': 'p01965_s03', 'split': 'test', 'within_configured_limit': 'False', 'N': '4981', 'S': '2', 'V': '59', 'Q': '13', 'S+V': '61'}, {'patient_id': 'p01965', 'record_id': 'p01965_s04', 'split': 'test', 'within_configured_limit': 'False', 'N': '4656', 'S': '2', 'V': '0', 'Q': '192', 'S+V': '2'}, {'patient_id': 'p01965', 'record_id': 'p01965_s05', 'split': 'test', 'within_configured_limit': 'False', 'N': '5220', 'S': '4', 'V': '208', 'Q': '435', 'S+V': '212'}, {'patient_id': 'p01965', 'record_id': 'p01965_s06', 'split': 'test', 'within_configured_limit': 'False', 'N': '4881', 'S': '5', 'V': '64', 'Q': '6', 'S+V': '69'}, {'patient_id': 'p01965', 'record_id': 'p01965_s07', 'split': 'test', 'within_configured_limit': 'False', 'N': '4789', 'S': '2', 'V': '47', 'Q': '65', 'S+V': '49'}, {'patient_id': 'p01965', 'record_id': 'p01965_s08', 'split': 'test', 'within_configured_limit': 'False', 'N': '4830', 'S': '2', 'V': '42', 'Q': '463', 'S+V': '44'}, {'patient_id': 'p01965', 'record_id': 'p01965_s09', 'split': 'test', 'within_configured_limit': 'False', 'N': '4572', 'S': '3', 'V': '16', 'Q': '6', 'S+V': '19'}, {'patient_id': 'p01965', 'record_id': 'p01965_s10', 'split': 'test', 'within_configured_limit': 'False', 'N': '4621', 'S': '13', 'V': '72', 'Q': '220', 'S+V': '85'}, {'patient_id': 'p01965', 'record_id': 'p01965_s11', 'split': 'test', 'within_configured_limit': 'False', 'N': '4733', 'S': '6', 'V': '0', 'Q': '221', 'S+V': '6'}, {'patient_id': 'p01965', 'record_id': 'p01965_s12', 'split': 'test', 'within_configured_limit': 'False', 'N': '5034', 'S': '20', 'V': '0', 'Q': '9', 'S+V': '20'}, {'patient_id': 'p01965', 'record_id': 'p01965_s13', 'split': 'test', 'within_configured_limit': 'False', 'N': '4683', 'S': '5', 'V': '0', 'Q': '293', 'S+V': '5'}, {'patient_id': 'p01965', 'record_id': 'p01965_s14', 'split': 'test', 'within_configured_limit': 'False', 'N': '4820', 'S': '1', 'V': '0', 'Q': '174', 'S+V': '1'}, {'patient_id': 'p01965', 'record_id': 'p01965_s15', 'split': 'test', 'within_configured_limit': 'False', 'N': '4779', 'S': '5', 'V': '166', 'Q': '846', 'S+V': '171'}, {'patient_id': 'p01965', 'record_id': 'p01965_s16', 'split': 'test', 'within_configured_limit': 'False', 'N': '4956', 'S': '2', 'V': '184', 'Q': '9', 'S+V': '186'}, {'patient_id': 'p01965', 'record_id': 'p01965_s17', 'split': 'test', 'within_configured_limit': 'False', 'N': '4776', 'S': '6', 'V': '26', 'Q': '618', 'S+V': '32'}, {'patient_id': 'p01965', 'record_id': 'p01965_s18', 'split': 'test', 'within_configured_limit': 'False', 'N': '4813', 'S': '13', 'V': '0', 'Q': '288', 'S+V': '13'}, {'patient_id': 'p01965', 'record_id': 'p01965_s19', 'split': 'test', 'within_configured_limit': 'False', 'N': '4653', 'S': '0', 'V': '330', 'Q': '341', 'S+V': '330'}, {'patient_id': 'p01965', 'record_id': 'p01965_s20', 'split': 'test', 'within_configured_limit': 'False', 'N': '4233', 'S': '54', 'V': '40', 'Q': '1062', 'S+V': '94'}, {'patient_id': 'p01965', 'record_id': 'p01965_s21', 'split': 'test', 'within_configured_limit': 'False', 'N': '4535', 'S': '3', 'V': '0', 'Q': '66', 'S+V': '3'}, {'patient_id': 'p01965', 'record_id': 'p01965_s22', 'split': 'test', 'within_configured_limit': 'False', 'N': '4691', 'S': '2', 'V': '146', 'Q': '0', 'S+V': '148'}, {'patient_id': 'p01965', 'record_id': 'p01965_s23', 'split': 'test', 'within_configured_limit': 'False', 'N': '4520', 'S': '4', 'V': '140', 'Q': '199', 'S+V': '144'}, {'patient_id': 'p01965', 'record_id': 'p01965_s24', 'split': 'test', 'within_configured_limit': 'False', 'N': '4614', 'S': '10', 'V': '10', 'Q': '472', 'S+V': '20'}, {'patient_id': 'p01965', 'record_id': 'p01965_s25', 'split': 'test', 'within_configured_limit': 'False', 'N': '4589', 'S': '6', 'V': '6', 'Q': '196', 'S+V': '12'}, {'patient_id': 'p01965', 'record_id': 'p01965_s26', 'split': 'test', 'within_configured_limit': 'False', 'N': '4486', 'S': '3', 'V': '0', 'Q': '204', 'S+V': '3'}, {'patient_id': 'p01965', 'record_id': 'p01965_s27', 'split': 'test', 'within_configured_limit': 'False', 'N': '4844', 'S': '12', 'V': '165', 'Q': '279', 'S+V': '177'}, {'patient_id': 'p01965', 'record_id': 'p01965_s28', 'split': 'test', 'within_configured_limit': 'False', 'N': '4687', 'S': '0', 'V': '45', 'Q': '302', 'S+V': '45'}, {'patient_id': 'p01965', 'record_id': 'p01965_s29', 'split': 'test', 'within_configured_limit': 'False', 'N': '4888', 'S': '0', 'V': '294', 'Q': '0', 'S+V': '294'}, {'patient_id': 'p01965', 'record_id': 'p01965_s30', 'split': 'test', 'within_configured_limit': 'False', 'N': '4200', 'S': '0', 'V': '243', 'Q': '732', 'S+V': '243'}, {'patient_id': 'p01965', 'record_id': 'p01965_s31', 'split': 'test', 'within_configured_limit': 'False', 'N': '4345', 'S': '0', 'V': '567', 'Q': '475', 'S+V': '567'}, {'patient_id': 'p01965', 'record_id': 'p01965_s32', 'split': 'test', 'within_configured_limit': 'False', 'N': '4641', 'S': '7', 'V': '0', 'Q': '379', 'S+V': '7'}, {'patient_id': 'p01965', 'record_id': 'p01965_s33', 'split': 'test', 'within_configured_limit': 'False', 'N': '4386', 'S': '0', 'V': '755', 'Q': '80', 'S+V': '755'}, {'patient_id': 'p01965', 'record_id': 'p01965_s34', 'split': 'test', 'within_configured_limit': 'False', 'N': '4137', 'S': '0', 'V': '708', 'Q': '405', 'S+V': '708'}, {'patient_id': 'p01965', 'record_id': 'p01965_s35', 'split': 'test', 'within_configured_limit': 'False', 'N': '5027', 'S': '0', 'V': '227', 'Q': '59', 'S+V': '227'}, {'patient_id': 'p01965', 'record_id': 'p01965_s36', 'split': 'test', 'within_configured_limit': 'False', 'N': '4732', 'S': '0', 'V': '78', 'Q': '209', 'S+V': '78'}, {'patient_id': 'p01965', 'record_id': 'p01965_s37', 'split': 'test', 'within_configured_limit': 'False', 'N': '4565', 'S': '5', 'V': '0', 'Q': '103', 'S+V': '5'}, {'patient_id': 'p01965', 'record_id': 'p01965_s38', 'split': 'test', 'within_configured_limit': 'False', 'N': '4210', 'S': '3', 'V': '0', 'Q': '1239', 'S+V': '3'}, {'patient_id': 'p01965', 'record_id': 'p01965_s39', 'split': 'test', 'within_configured_limit': 'False', 'N': '4491', 'S': '2', 'V': '69', 'Q': '197', 'S+V': '71'}, {'patient_id': 'p01965', 'record_id': 'p01965_s40', 'split': 'test', 'within_configured_limit': 'False', 'N': '4758', 'S': '10', 'V': '75', 'Q': '205', 'S+V': '85'}, {'patient_id': 'p01965', 'record_id': 'p01965_s41', 'split': 'test', 'within_configured_limit': 'False', 'N': '4566', 'S': '0', 'V': '375', 'Q': '277', 'S+V': '375'}, {'patient_id': 'p01965', 'record_id': 'p01965_s42', 'split': 'test', 'within_configured_limit': 'False', 'N': '4856', 'S': '5', 'V': '0', 'Q': '510', 'S+V': '5'}, {'patient_id': 'p01965', 'record_id': 'p01965_s43', 'split': 'test', 'within_configured_limit': 'False', 'N': '5071', 'S': '16', 'V': '0', 'Q': '280', 'S+V': '16'}, {'patient_id': 'p01965', 'record_id': 'p01965_s44', 'split': 'test', 'within_configured_limit': 'False', 'N': '4542', 'S': '2', 'V': '0', 'Q': '544', 'S+V': '2'}, {'patient_id': 'p01965', 'record_id': 'p01965_s45', 'split': 'test', 'within_configured_limit': 'False', 'N': '2890', 'S': '0', 'V': '0', 'Q': '3303', 'S+V': '0'}, {'patient_id': 'p01965', 'record_id': 'p01965_s46', 'split': 'test', 'within_configured_limit': 'False', 'N': '4892', 'S': '12', 'V': '66', 'Q': '580', 'S+V': '78'}, {'patient_id': 'p01965', 'record_id': 'p01965_s47', 'split': 'test', 'within_configured_limit': 'False', 'N': '4524', 'S': '2', 'V': '2', 'Q': '1089', 'S+V': '4'}, {'patient_id': 'p01965', 'record_id': 'p01965_s48', 'split': 'test', 'within_configured_limit': 'False', 'N': '5122', 'S': '17', 'V': '7', 'Q': '84', 'S+V': '24'}, {'patient_id': 'p01965', 'record_id': 'p01965_s49', 'split': 'test', 'within_configured_limit': 'False', 'N': '4656', 'S': '3', 'V': '0', 'Q': '601', 'S+V': '3'}, {'patient_id': 'p03281', 'record_id': 'p03281_s00', 'split': 'train', 'within_configured_limit': 'True', 'N': '4763', 'S': '0', 'V': '0', 'Q': '759', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s01', 'split': 'train', 'within_configured_limit': 'False', 'N': '4043', 'S': '0', 'V': '0', 'Q': '1307', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s02', 'split': 'train', 'within_configured_limit': 'False', 'N': '3620', 'S': '0', 'V': '0', 'Q': '1996', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s03', 'split': 'train', 'within_configured_limit': 'False', 'N': '4667', 'S': '0', 'V': '0', 'Q': '180', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s04', 'split': 'train', 'within_configured_limit': 'False', 'N': '4687', 'S': '0', 'V': '0', 'Q': '115', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s05', 'split': 'train', 'within_configured_limit': 'False', 'N': '4890', 'S': '0', 'V': '0', 'Q': '66', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s06', 'split': 'train', 'within_configured_limit': 'False', 'N': '4736', 'S': '0', 'V': '0', 'Q': '49', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s07', 'split': 'train', 'within_configured_limit': 'False', 'N': '4801', 'S': '0', 'V': '0', 'Q': '6', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s08', 'split': 'train', 'within_configured_limit': 'False', 'N': '4899', 'S': '0', 'V': '0', 'Q': '68', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s09', 'split': 'train', 'within_configured_limit': 'False', 'N': '3669', 'S': '0', 'V': '0', 'Q': '2398', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s10', 'split': 'train', 'within_configured_limit': 'False', 'N': '4696', 'S': '0', 'V': '1', 'Q': '1667', 'S+V': '1'}, {'patient_id': 'p03281', 'record_id': 'p03281_s11', 'split': 'train', 'within_configured_limit': 'False', 'N': '4864', 'S': '1', 'V': '0', 'Q': '757', 'S+V': '1'}, {'patient_id': 'p03281', 'record_id': 'p03281_s12', 'split': 'train', 'within_configured_limit': 'False', 'N': '6700', 'S': '0', 'V': '0', 'Q': '294', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s13', 'split': 'train', 'within_configured_limit': 'False', 'N': '3211', 'S': '0', 'V': '0', 'Q': '2843', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s14', 'split': 'train', 'within_configured_limit': 'False', 'N': '4937', 'S': '0', 'V': '0', 'Q': '93', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s15', 'split': 'train', 'within_configured_limit': 'False', 'N': '5098', 'S': '0', 'V': '0', 'Q': '54', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s16', 'split': 'train', 'within_configured_limit': 'False', 'N': '4851', 'S': '0', 'V': '0', 'Q': '69', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s17', 'split': 'train', 'within_configured_limit': 'False', 'N': '4563', 'S': '0', 'V': '0', 'Q': '118', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s18', 'split': 'train', 'within_configured_limit': 'False', 'N': '4647', 'S': '0', 'V': '0', 'Q': '149', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s19', 'split': 'train', 'within_configured_limit': 'False', 'N': '5109', 'S': '0', 'V': '0', 'Q': '493', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s20', 'split': 'train', 'within_configured_limit': 'False', 'N': '5358', 'S': '0', 'V': '1', 'Q': '1637', 'S+V': '1'}, {'patient_id': 'p03281', 'record_id': 'p03281_s21', 'split': 'train', 'within_configured_limit': 'False', 'N': '5195', 'S': '0', 'V': '0', 'Q': '1034', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s22', 'split': 'train', 'within_configured_limit': 'False', 'N': '5447', 'S': '0', 'V': '0', 'Q': '27', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s23', 'split': 'train', 'within_configured_limit': 'False', 'N': '5270', 'S': '0', 'V': '0', 'Q': '34', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s24', 'split': 'train', 'within_configured_limit': 'False', 'N': '5217', 'S': '0', 'V': '1', 'Q': '151', 'S+V': '1'}, {'patient_id': 'p03281', 'record_id': 'p03281_s25', 'split': 'train', 'within_configured_limit': 'False', 'N': '4805', 'S': '0', 'V': '0', 'Q': '1649', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s26', 'split': 'train', 'within_configured_limit': 'False', 'N': '5522', 'S': '0', 'V': '0', 'Q': '560', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s27', 'split': 'train', 'within_configured_limit': 'False', 'N': '5738', 'S': '0', 'V': '0', 'Q': '977', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s28', 'split': 'train', 'within_configured_limit': 'False', 'N': '5485', 'S': '0', 'V': '0', 'Q': '690', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s29', 'split': 'train', 'within_configured_limit': 'False', 'N': '5286', 'S': '0', 'V': '0', 'Q': '742', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s30', 'split': 'train', 'within_configured_limit': 'False', 'N': '4973', 'S': '0', 'V': '0', 'Q': '536', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s31', 'split': 'train', 'within_configured_limit': 'False', 'N': '5031', 'S': '0', 'V': '0', 'Q': '141', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s32', 'split': 'train', 'within_configured_limit': 'False', 'N': '4725', 'S': '0', 'V': '0', 'Q': '64', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s33', 'split': 'train', 'within_configured_limit': 'False', 'N': '5156', 'S': '1', 'V': '0', 'Q': '499', 'S+V': '1'}, {'patient_id': 'p03281', 'record_id': 'p03281_s34', 'split': 'train', 'within_configured_limit': 'False', 'N': '5428', 'S': '0', 'V': '0', 'Q': '518', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s35', 'split': 'train', 'within_configured_limit': 'False', 'N': '5035', 'S': '0', 'V': '0', 'Q': '55', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s36', 'split': 'train', 'within_configured_limit': 'False', 'N': '4983', 'S': '0', 'V': '0', 'Q': '72', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s37', 'split': 'train', 'within_configured_limit': 'False', 'N': '4938', 'S': '0', 'V': '0', 'Q': '259', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s38', 'split': 'train', 'within_configured_limit': 'False', 'N': '5265', 'S': '0', 'V': '0', 'Q': '1082', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s39', 'split': 'train', 'within_configured_limit': 'False', 'N': '4722', 'S': '0', 'V': '0', 'Q': '27', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s40', 'split': 'train', 'within_configured_limit': 'False', 'N': '4761', 'S': '0', 'V': '0', 'Q': '161', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s41', 'split': 'train', 'within_configured_limit': 'False', 'N': '4539', 'S': '0', 'V': '0', 'Q': '1495', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s42', 'split': 'train', 'within_configured_limit': 'False', 'N': '6000', 'S': '0', 'V': '0', 'Q': '702', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s43', 'split': 'train', 'within_configured_limit': 'False', 'N': '4971', 'S': '0', 'V': '0', 'Q': '2147', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s44', 'split': 'train', 'within_configured_limit': 'False', 'N': '4738', 'S': '0', 'V': '0', 'Q': '2427', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s45', 'split': 'train', 'within_configured_limit': 'False', 'N': '5098', 'S': '0', 'V': '0', 'Q': '1113', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s46', 'split': 'train', 'within_configured_limit': 'False', 'N': '5646', 'S': '0', 'V': '0', 'Q': '395', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s47', 'split': 'train', 'within_configured_limit': 'False', 'N': '5126', 'S': '0', 'V': '0', 'Q': '539', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s48', 'split': 'train', 'within_configured_limit': 'False', 'N': '4231', 'S': '0', 'V': '0', 'Q': '886', 'S+V': '0'}, {'patient_id': 'p03281', 'record_id': 'p03281_s49', 'split': 'train', 'within_configured_limit': 'False', 'N': '4724', 'S': '0', 'V': '0', 'Q': '88', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s00', 'split': 'train', 'within_configured_limit': 'True', 'N': '4797', 'S': '5', 'V': '0', 'Q': '1351', 'S+V': '5'}, {'patient_id': 'p03900', 'record_id': 'p03900_s01', 'split': 'train', 'within_configured_limit': 'False', 'N': '4834', 'S': '0', 'V': '0', 'Q': '1067', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s02', 'split': 'train', 'within_configured_limit': 'False', 'N': '5633', 'S': '2', 'V': '1', 'Q': '393', 'S+V': '3'}, {'patient_id': 'p03900', 'record_id': 'p03900_s03', 'split': 'train', 'within_configured_limit': 'False', 'N': '5333', 'S': '1', 'V': '0', 'Q': '756', 'S+V': '1'}, {'patient_id': 'p03900', 'record_id': 'p03900_s04', 'split': 'train', 'within_configured_limit': 'False', 'N': '5286', 'S': '0', 'V': '2', 'Q': '90', 'S+V': '2'}, {'patient_id': 'p03900', 'record_id': 'p03900_s05', 'split': 'train', 'within_configured_limit': 'False', 'N': '5036', 'S': '0', 'V': '2', 'Q': '39', 'S+V': '2'}, {'patient_id': 'p03900', 'record_id': 'p03900_s06', 'split': 'train', 'within_configured_limit': 'False', 'N': '5396', 'S': '1', 'V': '0', 'Q': '241', 'S+V': '1'}, {'patient_id': 'p03900', 'record_id': 'p03900_s07', 'split': 'train', 'within_configured_limit': 'False', 'N': '4790', 'S': '0', 'V': '0', 'Q': '33', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s08', 'split': 'train', 'within_configured_limit': 'False', 'N': '4799', 'S': '2', 'V': '0', 'Q': '43', 'S+V': '2'}, {'patient_id': 'p03900', 'record_id': 'p03900_s09', 'split': 'train', 'within_configured_limit': 'False', 'N': '5616', 'S': '0', 'V': '0', 'Q': '342', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s10', 'split': 'train', 'within_configured_limit': 'False', 'N': '5825', 'S': '0', 'V': '1', 'Q': '877', 'S+V': '1'}, {'patient_id': 'p03900', 'record_id': 'p03900_s11', 'split': 'train', 'within_configured_limit': 'False', 'N': '5849', 'S': '0', 'V': '0', 'Q': '802', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s12', 'split': 'train', 'within_configured_limit': 'False', 'N': '6300', 'S': '0', 'V': '1', 'Q': '96', 'S+V': '1'}, {'patient_id': 'p03900', 'record_id': 'p03900_s13', 'split': 'train', 'within_configured_limit': 'False', 'N': '4477', 'S': '0', 'V': '0', 'Q': '1479', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s14', 'split': 'train', 'within_configured_limit': 'False', 'N': '5158', 'S': '0', 'V': '0', 'Q': '1225', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s15', 'split': 'train', 'within_configured_limit': 'False', 'N': '6735', 'S': '1', 'V': '0', 'Q': '24', 'S+V': '1'}, {'patient_id': 'p03900', 'record_id': 'p03900_s16', 'split': 'train', 'within_configured_limit': 'False', 'N': '2724', 'S': '2', 'V': '0', 'Q': '3104', 'S+V': '2'}, {'patient_id': 'p03900', 'record_id': 'p03900_s17', 'split': 'train', 'within_configured_limit': 'False', 'N': '0', 'S': '0', 'V': '0', 'Q': '5088', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s18', 'split': 'train', 'within_configured_limit': 'False', 'N': '0', 'S': '0', 'V': '0', 'Q': '4673', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s19', 'split': 'train', 'within_configured_limit': 'False', 'N': '4778', 'S': '0', 'V': '1', 'Q': '97', 'S+V': '1'}, {'patient_id': 'p03900', 'record_id': 'p03900_s20', 'split': 'train', 'within_configured_limit': 'False', 'N': '4451', 'S': '0', 'V': '0', 'Q': '216', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s21', 'split': 'train', 'within_configured_limit': 'False', 'N': '4750', 'S': '1', 'V': '0', 'Q': '78', 'S+V': '1'}, {'patient_id': 'p03900', 'record_id': 'p03900_s22', 'split': 'train', 'within_configured_limit': 'False', 'N': '4443', 'S': '0', 'V': '0', 'Q': '48', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s23', 'split': 'train', 'within_configured_limit': 'False', 'N': '2916', 'S': '0', 'V': '0', 'Q': '1832', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s24', 'split': 'train', 'within_configured_limit': 'False', 'N': '4474', 'S': '0', 'V': '0', 'Q': '1678', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s25', 'split': 'train', 'within_configured_limit': 'False', 'N': '5539', 'S': '1', 'V': '0', 'Q': '300', 'S+V': '1'}, {'patient_id': 'p03900', 'record_id': 'p03900_s26', 'split': 'train', 'within_configured_limit': 'False', 'N': '5349', 'S': '0', 'V': '0', 'Q': '766', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s27', 'split': 'train', 'within_configured_limit': 'False', 'N': '5678', 'S': '0', 'V': '0', 'Q': '222', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s28', 'split': 'train', 'within_configured_limit': 'False', 'N': '5610', 'S': '0', 'V': '1', 'Q': '249', 'S+V': '1'}, {'patient_id': 'p03900', 'record_id': 'p03900_s29', 'split': 'train', 'within_configured_limit': 'False', 'N': '5280', 'S': '0', 'V': '0', 'Q': '935', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s30', 'split': 'train', 'within_configured_limit': 'False', 'N': '5707', 'S': '0', 'V': '0', 'Q': '1065', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s31', 'split': 'train', 'within_configured_limit': 'False', 'N': '5751', 'S': '0', 'V': '0', 'Q': '593', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s32', 'split': 'train', 'within_configured_limit': 'False', 'N': '6959', 'S': '0', 'V': '0', 'Q': '93', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s33', 'split': 'train', 'within_configured_limit': 'False', 'N': '6708', 'S': '1', 'V': '0', 'Q': '568', 'S+V': '1'}, {'patient_id': 'p03900', 'record_id': 'p03900_s34', 'split': 'train', 'within_configured_limit': 'False', 'N': '5918', 'S': '0', 'V': '0', 'Q': '779', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s35', 'split': 'train', 'within_configured_limit': 'False', 'N': '5551', 'S': '1', 'V': '0', 'Q': '433', 'S+V': '1'}, {'patient_id': 'p03900', 'record_id': 'p03900_s36', 'split': 'train', 'within_configured_limit': 'False', 'N': '6045', 'S': '0', 'V': '0', 'Q': '80', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s37', 'split': 'train', 'within_configured_limit': 'False', 'N': '5588', 'S': '0', 'V': '0', 'Q': '81', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s38', 'split': 'train', 'within_configured_limit': 'False', 'N': '5566', 'S': '0', 'V': '0', 'Q': '123', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s39', 'split': 'train', 'within_configured_limit': 'False', 'N': '5078', 'S': '1', 'V': '0', 'Q': '51', 'S+V': '1'}, {'patient_id': 'p03900', 'record_id': 'p03900_s40', 'split': 'train', 'within_configured_limit': 'False', 'N': '5122', 'S': '0', 'V': '0', 'Q': '240', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s41', 'split': 'train', 'within_configured_limit': 'False', 'N': '5358', 'S': '1', 'V': '0', 'Q': '115', 'S+V': '1'}, {'patient_id': 'p03900', 'record_id': 'p03900_s42', 'split': 'train', 'within_configured_limit': 'False', 'N': '4902', 'S': '0', 'V': '0', 'Q': '652', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s43', 'split': 'train', 'within_configured_limit': 'False', 'N': '5488', 'S': '0', 'V': '0', 'Q': '600', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s44', 'split': 'train', 'within_configured_limit': 'False', 'N': '5324', 'S': '0', 'V': '0', 'Q': '804', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s45', 'split': 'train', 'within_configured_limit': 'False', 'N': '5129', 'S': '0', 'V': '0', 'Q': '954', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s46', 'split': 'train', 'within_configured_limit': 'False', 'N': '5967', 'S': '0', 'V': '1', 'Q': '453', 'S+V': '1'}, {'patient_id': 'p03900', 'record_id': 'p03900_s47', 'split': 'train', 'within_configured_limit': 'False', 'N': '5820', 'S': '0', 'V': '0', 'Q': '656', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s48', 'split': 'train', 'within_configured_limit': 'False', 'N': '6079', 'S': '0', 'V': '0', 'Q': '456', 'S+V': '0'}, {'patient_id': 'p03900', 'record_id': 'p03900_s49', 'split': 'train', 'within_configured_limit': 'False', 'N': '5486', 'S': '0', 'V': '0', 'Q': '948', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s00', 'split': 'train', 'within_configured_limit': 'True', 'N': '1029', 'S': '0', 'V': '0', 'Q': '5113', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s01', 'split': 'train', 'within_configured_limit': 'False', 'N': '202', 'S': '0', 'V': '0', 'Q': '5437', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s02', 'split': 'train', 'within_configured_limit': 'False', 'N': '468', 'S': '1', 'V': '0', 'Q': '5035', 'S+V': '1'}, {'patient_id': 'p04014', 'record_id': 'p04014_s03', 'split': 'train', 'within_configured_limit': 'False', 'N': '4956', 'S': '0', 'V': '0', 'Q': '560', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s04', 'split': 'train', 'within_configured_limit': 'False', 'N': '88', 'S': '0', 'V': '0', 'Q': '5714', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s05', 'split': 'train', 'within_configured_limit': 'False', 'N': '5489', 'S': '0', 'V': '0', 'Q': '195', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s06', 'split': 'train', 'within_configured_limit': 'False', 'N': '4857', 'S': '0', 'V': '0', 'Q': '300', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s07', 'split': 'train', 'within_configured_limit': 'False', 'N': '2361', 'S': '1', 'V': '0', 'Q': '2610', 'S+V': '1'}, {'patient_id': 'p04014', 'record_id': 'p04014_s08', 'split': 'train', 'within_configured_limit': 'False', 'N': '2029', 'S': '0', 'V': '0', 'Q': '3465', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s09', 'split': 'train', 'within_configured_limit': 'False', 'N': '1073', 'S': '0', 'V': '0', 'Q': '5073', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s10', 'split': 'train', 'within_configured_limit': 'False', 'N': '818', 'S': '0', 'V': '0', 'Q': '5014', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s11', 'split': 'train', 'within_configured_limit': 'False', 'N': '268', 'S': '0', 'V': '0', 'Q': '5250', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s12', 'split': 'train', 'within_configured_limit': 'False', 'N': '4468', 'S': '0', 'V': '0', 'Q': '302', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s13', 'split': 'train', 'within_configured_limit': 'False', 'N': '2920', 'S': '1', 'V': '0', 'Q': '1004', 'S+V': '1'}, {'patient_id': 'p04014', 'record_id': 'p04014_s14', 'split': 'train', 'within_configured_limit': 'False', 'N': '1062', 'S': '0', 'V': '0', 'Q': '4923', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s15', 'split': 'train', 'within_configured_limit': 'False', 'N': '420', 'S': '0', 'V': '0', 'Q': '5452', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s16', 'split': 'train', 'within_configured_limit': 'False', 'N': '473', 'S': '0', 'V': '0', 'Q': '5023', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s17', 'split': 'train', 'within_configured_limit': 'False', 'N': '5498', 'S': '1', 'V': '0', 'Q': '235', 'S+V': '1'}, {'patient_id': 'p04014', 'record_id': 'p04014_s18', 'split': 'train', 'within_configured_limit': 'False', 'N': '5847', 'S': '1', 'V': '0', 'Q': '589', 'S+V': '1'}, {'patient_id': 'p04014', 'record_id': 'p04014_s19', 'split': 'train', 'within_configured_limit': 'False', 'N': '4991', 'S': '0', 'V': '0', 'Q': '355', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s20', 'split': 'train', 'within_configured_limit': 'False', 'N': '178', 'S': '1', 'V': '0', 'Q': '5616', 'S+V': '1'}, {'patient_id': 'p04014', 'record_id': 'p04014_s21', 'split': 'train', 'within_configured_limit': 'False', 'N': '1365', 'S': '0', 'V': '0', 'Q': '4188', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s22', 'split': 'train', 'within_configured_limit': 'False', 'N': '561', 'S': '0', 'V': '0', 'Q': '4772', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s23', 'split': 'train', 'within_configured_limit': 'False', 'N': '578', 'S': '0', 'V': '0', 'Q': '4893', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s24', 'split': 'train', 'within_configured_limit': 'False', 'N': '3431', 'S': '1', 'V': '0', 'Q': '2411', 'S+V': '1'}, {'patient_id': 'p04014', 'record_id': 'p04014_s25', 'split': 'train', 'within_configured_limit': 'False', 'N': '2688', 'S': '0', 'V': '0', 'Q': '2727', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s26', 'split': 'train', 'within_configured_limit': 'False', 'N': '50', 'S': '0', 'V': '0', 'Q': '5495', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s27', 'split': 'train', 'within_configured_limit': 'False', 'N': '5385', 'S': '0', 'V': '0', 'Q': '161', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s28', 'split': 'train', 'within_configured_limit': 'False', 'N': '156', 'S': '0', 'V': '0', 'Q': '5539', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s29', 'split': 'train', 'within_configured_limit': 'False', 'N': '1077', 'S': '0', 'V': '0', 'Q': '4554', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s30', 'split': 'train', 'within_configured_limit': 'False', 'N': '770', 'S': '0', 'V': '0', 'Q': '5502', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s31', 'split': 'train', 'within_configured_limit': 'False', 'N': '249', 'S': '0', 'V': '0', 'Q': '5623', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s32', 'split': 'train', 'within_configured_limit': 'False', 'N': '3085', 'S': '0', 'V': '0', 'Q': '2384', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s33', 'split': 'train', 'within_configured_limit': 'False', 'N': '764', 'S': '1', 'V': '0', 'Q': '3899', 'S+V': '1'}, {'patient_id': 'p04014', 'record_id': 'p04014_s34', 'split': 'train', 'within_configured_limit': 'False', 'N': '609', 'S': '0', 'V': '0', 'Q': '4446', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s35', 'split': 'train', 'within_configured_limit': 'False', 'N': '5416', 'S': '0', 'V': '0', 'Q': '215', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s36', 'split': 'train', 'within_configured_limit': 'False', 'N': '1106', 'S': '0', 'V': '0', 'Q': '4743', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s37', 'split': 'train', 'within_configured_limit': 'False', 'N': '1012', 'S': '0', 'V': '0', 'Q': '5147', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s38', 'split': 'train', 'within_configured_limit': 'False', 'N': '319', 'S': '0', 'V': '0', 'Q': '5491', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s39', 'split': 'train', 'within_configured_limit': 'False', 'N': '253', 'S': '0', 'V': '0', 'Q': '4660', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s40', 'split': 'train', 'within_configured_limit': 'False', 'N': '3865', 'S': '0', 'V': '0', 'Q': '1310', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s41', 'split': 'train', 'within_configured_limit': 'False', 'N': '5052', 'S': '0', 'V': '0', 'Q': '728', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s42', 'split': 'train', 'within_configured_limit': 'False', 'N': '59', 'S': '0', 'V': '0', 'Q': '5326', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s43', 'split': 'train', 'within_configured_limit': 'False', 'N': '1126', 'S': '0', 'V': '0', 'Q': '4539', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s44', 'split': 'train', 'within_configured_limit': 'False', 'N': '2735', 'S': '0', 'V': '0', 'Q': '2418', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s45', 'split': 'train', 'within_configured_limit': 'False', 'N': '5679', 'S': '2', 'V': '0', 'Q': '15', 'S+V': '2'}, {'patient_id': 'p04014', 'record_id': 'p04014_s46', 'split': 'train', 'within_configured_limit': 'False', 'N': '746', 'S': '0', 'V': '0', 'Q': '4983', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s47', 'split': 'train', 'within_configured_limit': 'False', 'N': '0', 'S': '0', 'V': '0', 'Q': '4295', 'S+V': '0'}, {'patient_id': 'p04014', 'record_id': 'p04014_s48', 'split': 'train', 'within_configured_limit': 'False', 'N': '4030', 'S': '1', 'V': '0', 'Q': '825', 'S+V': '1'}, {'patient_id': 'p04014', 'record_id': 'p04014_s49', 'split': 'train', 'within_configured_limit': 'False', 'N': '3707', 'S': '0', 'V': '0', 'Q': '1353', 'S+V': '0'}, {'patient_id': 'p04071', 'record_id': 'p04071_s00', 'split': 'test', 'within_configured_limit': 'True', 'N': '2427', 'S': '1029', 'V': '3', 'Q': '2300', 'S+V': '1032'}, {'patient_id': 'p04071', 'record_id': 'p04071_s01', 'split': 'test', 'within_configured_limit': 'False', 'N': '2349', 'S': '714', 'V': '2', 'Q': '2866', 'S+V': '716'}, {'patient_id': 'p04071', 'record_id': 'p04071_s02', 'split': 'test', 'within_configured_limit': 'False', 'N': '3344', 'S': '653', 'V': '2', 'Q': '16', 'S+V': '655'}, {'patient_id': 'p04071', 'record_id': 'p04071_s03', 'split': 'test', 'within_configured_limit': 'False', 'N': '3280', 'S': '701', 'V': '12', 'Q': '60', 'S+V': '713'}, {'patient_id': 'p04071', 'record_id': 'p04071_s04', 'split': 'test', 'within_configured_limit': 'False', 'N': '3078', 'S': '647', 'V': '7', 'Q': '42', 'S+V': '654'}, {'patient_id': 'p04071', 'record_id': 'p04071_s05', 'split': 'test', 'within_configured_limit': 'False', 'N': '2962', 'S': '585', 'V': '10', 'Q': '219', 'S+V': '595'}, {'patient_id': 'p04071', 'record_id': 'p04071_s06', 'split': 'test', 'within_configured_limit': 'False', 'N': '2825', 'S': '598', 'V': '2', 'Q': '1140', 'S+V': '600'}, {'patient_id': 'p04071', 'record_id': 'p04071_s07', 'split': 'test', 'within_configured_limit': 'False', 'N': '2383', 'S': '1035', 'V': '2', 'Q': '1836', 'S+V': '1037'}, {'patient_id': 'p04071', 'record_id': 'p04071_s08', 'split': 'test', 'within_configured_limit': 'False', 'N': '1782', 'S': '447', 'V': '0', 'Q': '2914', 'S+V': '447'}, {'patient_id': 'p04071', 'record_id': 'p04071_s09', 'split': 'test', 'within_configured_limit': 'False', 'N': '3141', 'S': '530', 'V': '5', 'Q': '766', 'S+V': '535'}, {'patient_id': 'p04071', 'record_id': 'p04071_s10', 'split': 'test', 'within_configured_limit': 'False', 'N': '1850', 'S': '455', 'V': '2', 'Q': '3037', 'S+V': '457'}, {'patient_id': 'p04071', 'record_id': 'p04071_s11', 'split': 'test', 'within_configured_limit': 'False', 'N': '3338', 'S': '481', 'V': '3', 'Q': '572', 'S+V': '484'}, {'patient_id': 'p04071', 'record_id': 'p04071_s12', 'split': 'test', 'within_configured_limit': 'False', 'N': '2987', 'S': '258', 'V': '6', 'Q': '854', 'S+V': '264'}, {'patient_id': 'p04071', 'record_id': 'p04071_s13', 'split': 'test', 'within_configured_limit': 'False', 'N': '3226', 'S': '283', 'V': '2', 'Q': '479', 'S+V': '285'}, {'patient_id': 'p04071', 'record_id': 'p04071_s14', 'split': 'test', 'within_configured_limit': 'False', 'N': '3466', 'S': '151', 'V': '15', 'Q': '219', 'S+V': '166'}, {'patient_id': 'p04071', 'record_id': 'p04071_s15', 'split': 'test', 'within_configured_limit': 'False', 'N': '3355', 'S': '186', 'V': '10', 'Q': '277', 'S+V': '196'}, {'patient_id': 'p04071', 'record_id': 'p04071_s16', 'split': 'test', 'within_configured_limit': 'False', 'N': '3125', 'S': '312', 'V': '10', 'Q': '198', 'S+V': '322'}, {'patient_id': 'p04071', 'record_id': 'p04071_s17', 'split': 'test', 'within_configured_limit': 'False', 'N': '3072', 'S': '412', 'V': '4', 'Q': '55', 'S+V': '416'}, {'patient_id': 'p04071', 'record_id': 'p04071_s18', 'split': 'test', 'within_configured_limit': 'False', 'N': '2996', 'S': '387', 'V': '4', 'Q': '127', 'S+V': '391'}, {'patient_id': 'p04071', 'record_id': 'p04071_s19', 'split': 'test', 'within_configured_limit': 'False', 'N': '1359', 'S': '415', 'V': '2', 'Q': '3967', 'S+V': '417'}, {'patient_id': 'p04071', 'record_id': 'p04071_s20', 'split': 'test', 'within_configured_limit': 'False', 'N': '2324', 'S': '946', 'V': '3', 'Q': '2193', 'S+V': '949'}, {'patient_id': 'p04071', 'record_id': 'p04071_s21', 'split': 'test', 'within_configured_limit': 'False', 'N': '3153', 'S': '295', 'V': '2', 'Q': '588', 'S+V': '297'}, {'patient_id': 'p04071', 'record_id': 'p04071_s22', 'split': 'test', 'within_configured_limit': 'False', 'N': '2485', 'S': '345', 'V': '2', 'Q': '1874', 'S+V': '347'}, {'patient_id': 'p04071', 'record_id': 'p04071_s23', 'split': 'test', 'within_configured_limit': 'False', 'N': '3305', 'S': '668', 'V': '2', 'Q': '37', 'S+V': '670'}, {'patient_id': 'p04071', 'record_id': 'p04071_s24', 'split': 'test', 'within_configured_limit': 'False', 'N': '3081', 'S': '604', 'V': '3', 'Q': '528', 'S+V': '607'}, {'patient_id': 'p04071', 'record_id': 'p04071_s25', 'split': 'test', 'within_configured_limit': 'False', 'N': '3460', 'S': '641', 'V': '23', 'Q': '23', 'S+V': '664'}, {'patient_id': 'p04071', 'record_id': 'p04071_s26', 'split': 'test', 'within_configured_limit': 'False', 'N': '2823', 'S': '1163', 'V': '2', 'Q': '1546', 'S+V': '1165'}, {'patient_id': 'p04071', 'record_id': 'p04071_s27', 'split': 'test', 'within_configured_limit': 'False', 'N': '3112', 'S': '1124', 'V': '4', 'Q': '1277', 'S+V': '1128'}, {'patient_id': 'p04071', 'record_id': 'p04071_s28', 'split': 'test', 'within_configured_limit': 'False', 'N': '2867', 'S': '939', 'V': '10', 'Q': '1529', 'S+V': '949'}, {'patient_id': 'p04071', 'record_id': 'p04071_s29', 'split': 'test', 'within_configured_limit': 'False', 'N': '3254', 'S': '507', 'V': '11', 'Q': '664', 'S+V': '518'}, {'patient_id': 'p04071', 'record_id': 'p04071_s30', 'split': 'test', 'within_configured_limit': 'False', 'N': '2021', 'S': '452', 'V': '1', 'Q': '2781', 'S+V': '453'}, {'patient_id': 'p04071', 'record_id': 'p04071_s31', 'split': 'test', 'within_configured_limit': 'False', 'N': '2851', 'S': '1192', 'V': '1', 'Q': '1814', 'S+V': '1193'}, {'patient_id': 'p04071', 'record_id': 'p04071_s32', 'split': 'test', 'within_configured_limit': 'False', 'N': '3089', 'S': '1093', 'V': '1', 'Q': '1378', 'S+V': '1094'}, {'patient_id': 'p04071', 'record_id': 'p04071_s33', 'split': 'test', 'within_configured_limit': 'False', 'N': '3210', 'S': '858', 'V': '6', 'Q': '972', 'S+V': '864'}, {'patient_id': 'p04071', 'record_id': 'p04071_s34', 'split': 'test', 'within_configured_limit': 'False', 'N': '3390', 'S': '1122', 'V': '35', 'Q': '20', 'S+V': '1157'}, {'patient_id': 'p04071', 'record_id': 'p04071_s35', 'split': 'test', 'within_configured_limit': 'False', 'N': '3248', 'S': '993', 'V': '0', 'Q': '20', 'S+V': '993'}, {'patient_id': 'p04071', 'record_id': 'p04071_s36', 'split': 'test', 'within_configured_limit': 'False', 'N': '3774', 'S': '810', 'V': '0', 'Q': '44', 'S+V': '810'}, {'patient_id': 'p04071', 'record_id': 'p04071_s37', 'split': 'test', 'within_configured_limit': 'False', 'N': '3522', 'S': '1430', 'V': '7', 'Q': '892', 'S+V': '1437'}, {'patient_id': 'p04071', 'record_id': 'p04071_s38', 'split': 'test', 'within_configured_limit': 'False', 'N': '3481', 'S': '975', 'V': '9', 'Q': '1117', 'S+V': '984'}, {'patient_id': 'p04071', 'record_id': 'p04071_s39', 'split': 'test', 'within_configured_limit': 'False', 'N': '3480', 'S': '938', 'V': '8', 'Q': '987', 'S+V': '946'}, {'patient_id': 'p04071', 'record_id': 'p04071_s40', 'split': 'test', 'within_configured_limit': 'False', 'N': '2109', 'S': '465', 'V': '7', 'Q': '3062', 'S+V': '472'}, {'patient_id': 'p04071', 'record_id': 'p04071_s41', 'split': 'test', 'within_configured_limit': 'False', 'N': '3059', 'S': '629', 'V': '10', 'Q': '451', 'S+V': '639'}, {'patient_id': 'p04071', 'record_id': 'p04071_s42', 'split': 'test', 'within_configured_limit': 'False', 'N': '3222', 'S': '781', 'V': '2', 'Q': '538', 'S+V': '783'}, {'patient_id': 'p04071', 'record_id': 'p04071_s43', 'split': 'test', 'within_configured_limit': 'False', 'N': '3097', 'S': '965', 'V': '1', 'Q': '1815', 'S+V': '966'}, {'patient_id': 'p04071', 'record_id': 'p04071_s44', 'split': 'test', 'within_configured_limit': 'False', 'N': '2127', 'S': '521', 'V': '2', 'Q': '2775', 'S+V': '523'}, {'patient_id': 'p04071', 'record_id': 'p04071_s45', 'split': 'test', 'within_configured_limit': 'False', 'N': '3141', 'S': '1458', 'V': '0', 'Q': '199', 'S+V': '1458'}, {'patient_id': 'p04071', 'record_id': 'p04071_s46', 'split': 'test', 'within_configured_limit': 'False', 'N': '3051', 'S': '993', 'V': '6', 'Q': '477', 'S+V': '999'}, {'patient_id': 'p04071', 'record_id': 'p04071_s47', 'split': 'test', 'within_configured_limit': 'False', 'N': '3243', 'S': '894', 'V': '4', 'Q': '286', 'S+V': '898'}, {'patient_id': 'p04071', 'record_id': 'p04071_s48', 'split': 'test', 'within_configured_limit': 'False', 'N': '2904', 'S': '1099', 'V': '1', 'Q': '1538', 'S+V': '1100'}, {'patient_id': 'p04071', 'record_id': 'p04071_s49', 'split': 'test', 'within_configured_limit': 'False', 'N': '3168', 'S': '936', 'V': '1', 'Q': '1039', 'S+V': '937'}, {'patient_id': 'p05133', 'record_id': 'p05133_s00', 'split': 'train', 'within_configured_limit': 'True', 'N': '1818', 'S': '4', 'V': '0', 'Q': '4041', 'S+V': '4'}, {'patient_id': 'p05133', 'record_id': 'p05133_s01', 'split': 'train', 'within_configured_limit': 'False', 'N': '1630', 'S': '3', 'V': '0', 'Q': '2976', 'S+V': '3'}, {'patient_id': 'p05133', 'record_id': 'p05133_s02', 'split': 'train', 'within_configured_limit': 'False', 'N': '2365', 'S': '0', 'V': '0', 'Q': '2368', 'S+V': '0'}, {'patient_id': 'p05133', 'record_id': 'p05133_s03', 'split': 'train', 'within_configured_limit': 'False', 'N': '2151', 'S': '2', 'V': '0', 'Q': '2409', 'S+V': '2'}, {'patient_id': 'p05133', 'record_id': 'p05133_s04', 'split': 'train', 'within_configured_limit': 'False', 'N': '3076', 'S': '2', 'V': '0', 'Q': '1396', 'S+V': '2'}, {'patient_id': 'p05133', 'record_id': 'p05133_s05', 'split': 'train', 'within_configured_limit': 'False', 'N': '4146', 'S': '1', 'V': '2', 'Q': '69', 'S+V': '3'}, {'patient_id': 'p05133', 'record_id': 'p05133_s06', 'split': 'train', 'within_configured_limit': 'False', 'N': '4158', 'S': '1', 'V': '0', 'Q': '60', 'S+V': '1'}, {'patient_id': 'p05133', 'record_id': 'p05133_s07', 'split': 'train', 'within_configured_limit': 'False', 'N': '4117', 'S': '1', 'V': '5', 'Q': '105', 'S+V': '6'}, {'patient_id': 'p05133', 'record_id': 'p05133_s08', 'split': 'train', 'within_configured_limit': 'False', 'N': '4147', 'S': '1', 'V': '0', 'Q': '81', 'S+V': '1'}, {'patient_id': 'p05133', 'record_id': 'p05133_s09', 'split': 'train', 'within_configured_limit': 'False', 'N': '2434', 'S': '2', 'V': '0', 'Q': '2351', 'S+V': '2'}, {'patient_id': 'p05133', 'record_id': 'p05133_s10', 'split': 'train', 'within_configured_limit': 'False', 'N': '2534', 'S': '0', 'V': '0', 'Q': '1883', 'S+V': '0'}, {'patient_id': 'p05133', 'record_id': 'p05133_s11', 'split': 'train', 'within_configured_limit': 'False', 'N': '3148', 'S': '2', 'V': '0', 'Q': '1295', 'S+V': '2'}, {'patient_id': 'p05133', 'record_id': 'p05133_s12', 'split': 'train', 'within_configured_limit': 'False', 'N': '1544', 'S': '3', 'V': '0', 'Q': '2970', 'S+V': '3'}, {'patient_id': 'p05133', 'record_id': 'p05133_s13', 'split': 'train', 'within_configured_limit': 'False', 'N': '3522', 'S': '16', 'V': '2', 'Q': '750', 'S+V': '18'}, {'patient_id': 'p05133', 'record_id': 'p05133_s14', 'split': 'train', 'within_configured_limit': 'False', 'N': '3848', 'S': '0', 'V': '0', 'Q': '520', 'S+V': '0'}, {'patient_id': 'p05133', 'record_id': 'p05133_s15', 'split': 'train', 'within_configured_limit': 'False', 'N': '4146', 'S': '2', 'V': '0', 'Q': '71', 'S+V': '2'}, {'patient_id': 'p05133', 'record_id': 'p05133_s16', 'split': 'train', 'within_configured_limit': 'False', 'N': '4077', 'S': '1', 'V': '0', 'Q': '168', 'S+V': '1'}, {'patient_id': 'p05133', 'record_id': 'p05133_s17', 'split': 'train', 'within_configured_limit': 'False', 'N': '4132', 'S': '2', 'V': '0', 'Q': '103', 'S+V': '2'}, {'patient_id': 'p05133', 'record_id': 'p05133_s18', 'split': 'train', 'within_configured_limit': 'False', 'N': '4146', 'S': '2', 'V': '0', 'Q': '78', 'S+V': '2'}, {'patient_id': 'p05133', 'record_id': 'p05133_s19', 'split': 'train', 'within_configured_limit': 'False', 'N': '2046', 'S': '2', 'V': '0', 'Q': '2490', 'S+V': '2'}, {'patient_id': 'p05133', 'record_id': 'p05133_s20', 'split': 'train', 'within_configured_limit': 'False', 'N': '2126', 'S': '0', 'V': '0', 'Q': '2171', 'S+V': '0'}, {'patient_id': 'p05133', 'record_id': 'p05133_s21', 'split': 'train', 'within_configured_limit': 'False', 'N': '3364', 'S': '2', 'V': '0', 'Q': '1039', 'S+V': '2'}, {'patient_id': 'p05133', 'record_id': 'p05133_s22', 'split': 'train', 'within_configured_limit': 'False', 'N': '4087', 'S': '2', 'V': '1', 'Q': '135', 'S+V': '3'}, {'patient_id': 'p05133', 'record_id': 'p05133_s23', 'split': 'train', 'within_configured_limit': 'False', 'N': '4168', 'S': '0', 'V': '0', 'Q': '55', 'S+V': '0'}, {'patient_id': 'p05133', 'record_id': 'p05133_s24', 'split': 'train', 'within_configured_limit': 'False', 'N': '4045', 'S': '2', 'V': '0', 'Q': '181', 'S+V': '2'}, {'patient_id': 'p05133', 'record_id': 'p05133_s25', 'split': 'train', 'within_configured_limit': 'False', 'N': '2186', 'S': '2', 'V': '0', 'Q': '2366', 'S+V': '2'}, {'patient_id': 'p05133', 'record_id': 'p05133_s26', 'split': 'train', 'within_configured_limit': 'False', 'N': '1556', 'S': '3', 'V': '0', 'Q': '3786', 'S+V': '3'}, {'patient_id': 'p05133', 'record_id': 'p05133_s27', 'split': 'train', 'within_configured_limit': 'False', 'N': '2703', 'S': '0', 'V': '0', 'Q': '1629', 'S+V': '0'}, {'patient_id': 'p05133', 'record_id': 'p05133_s28', 'split': 'train', 'within_configured_limit': 'False', 'N': '2456', 'S': '1', 'V': '1', 'Q': '2042', 'S+V': '2'}, {'patient_id': 'p05133', 'record_id': 'p05133_s29', 'split': 'train', 'within_configured_limit': 'False', 'N': '1793', 'S': '1', 'V': '0', 'Q': '2618', 'S+V': '1'}, {'patient_id': 'p05133', 'record_id': 'p05133_s30', 'split': 'train', 'within_configured_limit': 'False', 'N': '2273', 'S': '3', 'V': '0', 'Q': '2333', 'S+V': '3'}, {'patient_id': 'p05133', 'record_id': 'p05133_s31', 'split': 'train', 'within_configured_limit': 'False', 'N': '4156', 'S': '1', 'V': '0', 'Q': '61', 'S+V': '1'}, {'patient_id': 'p05133', 'record_id': 'p05133_s32', 'split': 'train', 'within_configured_limit': 'False', 'N': '4003', 'S': '0', 'V': '1', 'Q': '263', 'S+V': '1'}, {'patient_id': 'p05133', 'record_id': 'p05133_s33', 'split': 'train', 'within_configured_limit': 'False', 'N': '2505', 'S': '1', 'V': '0', 'Q': '1946', 'S+V': '1'}, {'patient_id': 'p05133', 'record_id': 'p05133_s34', 'split': 'train', 'within_configured_limit': 'False', 'N': '932', 'S': '0', 'V': '0', 'Q': '3858', 'S+V': '0'}, {'patient_id': 'p05133', 'record_id': 'p05133_s35', 'split': 'train', 'within_configured_limit': 'False', 'N': '4128', 'S': '1', 'V': '0', 'Q': '94', 'S+V': '1'}, {'patient_id': 'p05133', 'record_id': 'p05133_s36', 'split': 'train', 'within_configured_limit': 'False', 'N': '4062', 'S': '2', 'V': '0', 'Q': '171', 'S+V': '2'}, {'patient_id': 'p05133', 'record_id': 'p05133_s37', 'split': 'train', 'within_configured_limit': 'False', 'N': '3247', 'S': '1', 'V': '1', 'Q': '1167', 'S+V': '2'}, {'patient_id': 'p05133', 'record_id': 'p05133_s38', 'split': 'train', 'within_configured_limit': 'False', 'N': '411', 'S': '0', 'V': '0', 'Q': '3852', 'S+V': '0'}, {'patient_id': 'p05133', 'record_id': 'p05133_s39', 'split': 'train', 'within_configured_limit': 'False', 'N': '4155', 'S': '4', 'V': '0', 'Q': '75', 'S+V': '4'}, {'patient_id': 'p05133', 'record_id': 'p05133_s40', 'split': 'train', 'within_configured_limit': 'False', 'N': '2060', 'S': '0', 'V': '0', 'Q': '2786', 'S+V': '0'}, {'patient_id': 'p05133', 'record_id': 'p05133_s41', 'split': 'train', 'within_configured_limit': 'False', 'N': '2707', 'S': '1', 'V': '0', 'Q': '2577', 'S+V': '1'}, {'patient_id': 'p05133', 'record_id': 'p05133_s42', 'split': 'train', 'within_configured_limit': 'False', 'N': '1382', 'S': '0', 'V': '0', 'Q': '4125', 'S+V': '0'}, {'patient_id': 'p05133', 'record_id': 'p05133_s43', 'split': 'train', 'within_configured_limit': 'False', 'N': '1834', 'S': '1', 'V': '0', 'Q': '3309', 'S+V': '1'}, {'patient_id': 'p05133', 'record_id': 'p05133_s44', 'split': 'train', 'within_configured_limit': 'False', 'N': '2178', 'S': '1', 'V': '0', 'Q': '3638', 'S+V': '1'}, {'patient_id': 'p05133', 'record_id': 'p05133_s45', 'split': 'train', 'within_configured_limit': 'False', 'N': '2224', 'S': '1', 'V': '0', 'Q': '2270', 'S+V': '1'}, {'patient_id': 'p05133', 'record_id': 'p05133_s46', 'split': 'train', 'within_configured_limit': 'False', 'N': '3085', 'S': '2', 'V': '0', 'Q': '1277', 'S+V': '2'}, {'patient_id': 'p05133', 'record_id': 'p05133_s47', 'split': 'train', 'within_configured_limit': 'False', 'N': '3772', 'S': '0', 'V': '0', 'Q': '498', 'S+V': '0'}, {'patient_id': 'p05133', 'record_id': 'p05133_s48', 'split': 'train', 'within_configured_limit': 'False', 'N': '1860', 'S': '2', 'V': '0', 'Q': '2484', 'S+V': '2'}, {'patient_id': 'p05133', 'record_id': 'p05133_s49', 'split': 'train', 'within_configured_limit': 'False', 'N': '4086', 'S': '2', 'V': '0', 'Q': '134', 'S+V': '2'}, {'patient_id': 'p07028', 'record_id': 'p07028_s00', 'split': 'train', 'within_configured_limit': 'True', 'N': '4315', 'S': '24', 'V': '2', 'Q': '139', 'S+V': '26'}, {'patient_id': 'p07028', 'record_id': 'p07028_s01', 'split': 'train', 'within_configured_limit': 'False', 'N': '5007', 'S': '59', 'V': '21', 'Q': '165', 'S+V': '80'}, {'patient_id': 'p07028', 'record_id': 'p07028_s02', 'split': 'train', 'within_configured_limit': 'False', 'N': '4195', 'S': '148', 'V': '7', 'Q': '124', 'S+V': '155'}, {'patient_id': 'p07028', 'record_id': 'p07028_s03', 'split': 'train', 'within_configured_limit': 'False', 'N': '3708', 'S': '32', 'V': '8', 'Q': '38', 'S+V': '40'}, {'patient_id': 'p07028', 'record_id': 'p07028_s04', 'split': 'train', 'within_configured_limit': 'False', 'N': '4137', 'S': '48', 'V': '4', 'Q': '381', 'S+V': '52'}, {'patient_id': 'p07028', 'record_id': 'p07028_s05', 'split': 'train', 'within_configured_limit': 'False', 'N': '4002', 'S': '56', 'V': '7', 'Q': '127', 'S+V': '63'}, {'patient_id': 'p07028', 'record_id': 'p07028_s06', 'split': 'train', 'within_configured_limit': 'False', 'N': '3911', 'S': '41', 'V': '9', 'Q': '177', 'S+V': '50'}, {'patient_id': 'p07028', 'record_id': 'p07028_s07', 'split': 'train', 'within_configured_limit': 'False', 'N': '3692', 'S': '29', 'V': '6', 'Q': '70', 'S+V': '35'}, {'patient_id': 'p07028', 'record_id': 'p07028_s08', 'split': 'train', 'within_configured_limit': 'False', 'N': '3728', 'S': '25', 'V': '9', 'Q': '46', 'S+V': '34'}, {'patient_id': 'p07028', 'record_id': 'p07028_s09', 'split': 'train', 'within_configured_limit': 'False', 'N': '4402', 'S': '27', 'V': '2', 'Q': '782', 'S+V': '29'}, {'patient_id': 'p07028', 'record_id': 'p07028_s10', 'split': 'train', 'within_configured_limit': 'False', 'N': '1225', 'S': '10', 'V': '1', 'Q': '7712', 'S+V': '11'}, {'patient_id': 'p07028', 'record_id': 'p07028_s11', 'split': 'train', 'within_configured_limit': 'False', 'N': '3760', 'S': '29', 'V': '2', 'Q': '820', 'S+V': '31'}, {'patient_id': 'p07028', 'record_id': 'p07028_s12', 'split': 'train', 'within_configured_limit': 'False', 'N': '4005', 'S': '56', 'V': '8', 'Q': '163', 'S+V': '64'}, {'patient_id': 'p07028', 'record_id': 'p07028_s13', 'split': 'train', 'within_configured_limit': 'False', 'N': '3999', 'S': '58', 'V': '12', 'Q': '35', 'S+V': '70'}, {'patient_id': 'p07028', 'record_id': 'p07028_s14', 'split': 'train', 'within_configured_limit': 'False', 'N': '4212', 'S': '48', 'V': '2', 'Q': '689', 'S+V': '50'}, {'patient_id': 'p07028', 'record_id': 'p07028_s15', 'split': 'train', 'within_configured_limit': 'False', 'N': '3815', 'S': '44', 'V': '7', 'Q': '2988', 'S+V': '51'}, {'patient_id': 'p07028', 'record_id': 'p07028_s16', 'split': 'train', 'within_configured_limit': 'False', 'N': '4705', 'S': '13', 'V': '10', 'Q': '2252', 'S+V': '23'}, {'patient_id': 'p07028', 'record_id': 'p07028_s17', 'split': 'train', 'within_configured_limit': 'False', 'N': '4341', 'S': '87', 'V': '13', 'Q': '646', 'S+V': '100'}, {'patient_id': 'p07028', 'record_id': 'p07028_s18', 'split': 'train', 'within_configured_limit': 'False', 'N': '4526', 'S': '34', 'V': '18', 'Q': '74', 'S+V': '52'}, {'patient_id': 'p07028', 'record_id': 'p07028_s19', 'split': 'train', 'within_configured_limit': 'False', 'N': '3792', 'S': '126', 'V': '20', 'Q': '38', 'S+V': '146'}, {'patient_id': 'p07028', 'record_id': 'p07028_s20', 'split': 'train', 'within_configured_limit': 'False', 'N': '3910', 'S': '83', 'V': '9', 'Q': '73', 'S+V': '92'}, {'patient_id': 'p07028', 'record_id': 'p07028_s21', 'split': 'train', 'within_configured_limit': 'False', 'N': '3814', 'S': '51', 'V': '6', 'Q': '646', 'S+V': '57'}, {'patient_id': 'p07028', 'record_id': 'p07028_s22', 'split': 'train', 'within_configured_limit': 'False', 'N': '3489', 'S': '8', 'V': '6', 'Q': '2577', 'S+V': '14'}, {'patient_id': 'p07028', 'record_id': 'p07028_s23', 'split': 'train', 'within_configured_limit': 'False', 'N': '3937', 'S': '34', 'V': '8', 'Q': '2522', 'S+V': '42'}, {'patient_id': 'p07028', 'record_id': 'p07028_s24', 'split': 'train', 'within_configured_limit': 'False', 'N': '4257', 'S': '64', 'V': '16', 'Q': '405', 'S+V': '80'}, {'patient_id': 'p07028', 'record_id': 'p07028_s25', 'split': 'train', 'within_configured_limit': 'False', 'N': '3835', 'S': '35', 'V': '13', 'Q': '1287', 'S+V': '48'}, {'patient_id': 'p07028', 'record_id': 'p07028_s26', 'split': 'train', 'within_configured_limit': 'False', 'N': '4238', 'S': '38', 'V': '11', 'Q': '21', 'S+V': '49'}, {'patient_id': 'p07028', 'record_id': 'p07028_s27', 'split': 'train', 'within_configured_limit': 'False', 'N': '4054', 'S': '52', 'V': '11', 'Q': '171', 'S+V': '63'}, {'patient_id': 'p07028', 'record_id': 'p07028_s28', 'split': 'train', 'within_configured_limit': 'False', 'N': '3886', 'S': '25', 'V': '5', 'Q': '41', 'S+V': '30'}, {'patient_id': 'p07028', 'record_id': 'p07028_s29', 'split': 'train', 'within_configured_limit': 'False', 'N': '4353', 'S': '19', 'V': '7', 'Q': '550', 'S+V': '26'}, {'patient_id': 'p07028', 'record_id': 'p07028_s30', 'split': 'train', 'within_configured_limit': 'False', 'N': '4341', 'S': '22', 'V': '14', 'Q': '50', 'S+V': '36'}, {'patient_id': 'p07028', 'record_id': 'p07028_s31', 'split': 'train', 'within_configured_limit': 'False', 'N': '3656', 'S': '34', 'V': '12', 'Q': '1214', 'S+V': '46'}, {'patient_id': 'p07028', 'record_id': 'p07028_s32', 'split': 'train', 'within_configured_limit': 'False', 'N': '4896', 'S': '14', 'V': '8', 'Q': '221', 'S+V': '22'}, {'patient_id': 'p07028', 'record_id': 'p07028_s33', 'split': 'train', 'within_configured_limit': 'False', 'N': '4134', 'S': '25', 'V': '7', 'Q': '442', 'S+V': '32'}, {'patient_id': 'p07028', 'record_id': 'p07028_s34', 'split': 'train', 'within_configured_limit': 'False', 'N': '4624', 'S': '47', 'V': '6', 'Q': '380', 'S+V': '53'}, {'patient_id': 'p07028', 'record_id': 'p07028_s35', 'split': 'train', 'within_configured_limit': 'False', 'N': '3776', 'S': '27', 'V': '2', 'Q': '1369', 'S+V': '29'}, {'patient_id': 'p07028', 'record_id': 'p07028_s36', 'split': 'train', 'within_configured_limit': 'False', 'N': '3910', 'S': '55', 'V': '13', 'Q': '47', 'S+V': '68'}, {'patient_id': 'p07028', 'record_id': 'p07028_s37', 'split': 'train', 'within_configured_limit': 'False', 'N': '4649', 'S': '28', 'V': '6', 'Q': '69', 'S+V': '34'}, {'patient_id': 'p07028', 'record_id': 'p07028_s38', 'split': 'train', 'within_configured_limit': 'False', 'N': '4856', 'S': '44', 'V': '4', 'Q': '12', 'S+V': '48'}, {'patient_id': 'p07028', 'record_id': 'p07028_s39', 'split': 'train', 'within_configured_limit': 'False', 'N': '4566', 'S': '47', 'V': '4', 'Q': '26', 'S+V': '51'}, {'patient_id': 'p07028', 'record_id': 'p07028_s40', 'split': 'train', 'within_configured_limit': 'False', 'N': '4017', 'S': '63', 'V': '6', 'Q': '653', 'S+V': '69'}, {'patient_id': 'p07028', 'record_id': 'p07028_s41', 'split': 'train', 'within_configured_limit': 'False', 'N': '4244', 'S': '34', 'V': '6', 'Q': '362', 'S+V': '40'}, {'patient_id': 'p07028', 'record_id': 'p07028_s42', 'split': 'train', 'within_configured_limit': 'False', 'N': '2878', 'S': '12', 'V': '12', 'Q': '3171', 'S+V': '24'}, {'patient_id': 'p07028', 'record_id': 'p07028_s43', 'split': 'train', 'within_configured_limit': 'False', 'N': '4828', 'S': '22', 'V': '1', 'Q': '50', 'S+V': '23'}, {'patient_id': 'p07028', 'record_id': 'p07028_s44', 'split': 'train', 'within_configured_limit': 'False', 'N': '4467', 'S': '60', 'V': '7', 'Q': '299', 'S+V': '67'}, {'patient_id': 'p07028', 'record_id': 'p07028_s45', 'split': 'train', 'within_configured_limit': 'False', 'N': '4330', 'S': '66', 'V': '3', 'Q': '313', 'S+V': '69'}, {'patient_id': 'p07028', 'record_id': 'p07028_s46', 'split': 'train', 'within_configured_limit': 'False', 'N': '3780', 'S': '36', 'V': '6', 'Q': '43', 'S+V': '42'}, {'patient_id': 'p07028', 'record_id': 'p07028_s47', 'split': 'train', 'within_configured_limit': 'False', 'N': '4241', 'S': '35', 'V': '8', 'Q': '300', 'S+V': '43'}, {'patient_id': 'p07028', 'record_id': 'p07028_s48', 'split': 'train', 'within_configured_limit': 'False', 'N': '3621', 'S': '31', 'V': '9', 'Q': '50', 'S+V': '40'}, {'patient_id': 'p07028', 'record_id': 'p07028_s49', 'split': 'train', 'within_configured_limit': 'False', 'N': '3583', 'S': '20', 'V': '8', 'Q': '26', 'S+V': '28'}, {'patient_id': 'p07070', 'record_id': 'p07070_s00', 'split': 'train', 'within_configured_limit': 'True', 'N': '3667', 'S': '25', 'V': '0', 'Q': '2016', 'S+V': '25'}, {'patient_id': 'p07070', 'record_id': 'p07070_s01', 'split': 'train', 'within_configured_limit': 'False', 'N': '2020', 'S': '18', 'V': '0', 'Q': '3601', 'S+V': '18'}, {'patient_id': 'p07070', 'record_id': 'p07070_s02', 'split': 'train', 'within_configured_limit': 'False', 'N': '3332', 'S': '21', 'V': '0', 'Q': '1784', 'S+V': '21'}, {'patient_id': 'p07070', 'record_id': 'p07070_s03', 'split': 'train', 'within_configured_limit': 'False', 'N': '4251', 'S': '20', 'V': '0', 'Q': '53', 'S+V': '20'}, {'patient_id': 'p07070', 'record_id': 'p07070_s04', 'split': 'train', 'within_configured_limit': 'False', 'N': '3952', 'S': '37', 'V': '0', 'Q': '122', 'S+V': '37'}, {'patient_id': 'p07070', 'record_id': 'p07070_s05', 'split': 'train', 'within_configured_limit': 'False', 'N': '3951', 'S': '34', 'V': '0', 'Q': '72', 'S+V': '34'}, {'patient_id': 'p07070', 'record_id': 'p07070_s06', 'split': 'train', 'within_configured_limit': 'False', 'N': '3341', 'S': '18', 'V': '0', 'Q': '1044', 'S+V': '18'}, {'patient_id': 'p07070', 'record_id': 'p07070_s07', 'split': 'train', 'within_configured_limit': 'False', 'N': '3114', 'S': '40', 'V': '0', 'Q': '756', 'S+V': '40'}, {'patient_id': 'p07070', 'record_id': 'p07070_s08', 'split': 'train', 'within_configured_limit': 'False', 'N': '3502', 'S': '29', 'V': '0', 'Q': '294', 'S+V': '29'}, {'patient_id': 'p07070', 'record_id': 'p07070_s09', 'split': 'train', 'within_configured_limit': 'False', 'N': '3700', 'S': '10', 'V': '0', 'Q': '1914', 'S+V': '10'}, {'patient_id': 'p07070', 'record_id': 'p07070_s10', 'split': 'train', 'within_configured_limit': 'False', 'N': '2915', 'S': '9', 'V': '0', 'Q': '2265', 'S+V': '9'}, {'patient_id': 'p07070', 'record_id': 'p07070_s11', 'split': 'train', 'within_configured_limit': 'False', 'N': '2863', 'S': '13', 'V': '0', 'Q': '3004', 'S+V': '13'}, {'patient_id': 'p07070', 'record_id': 'p07070_s12', 'split': 'train', 'within_configured_limit': 'False', 'N': '2643', 'S': '13', 'V': '0', 'Q': '2572', 'S+V': '13'}, {'patient_id': 'p07070', 'record_id': 'p07070_s13', 'split': 'train', 'within_configured_limit': 'False', 'N': '4401', 'S': '19', 'V': '0', 'Q': '21', 'S+V': '19'}, {'patient_id': 'p07070', 'record_id': 'p07070_s14', 'split': 'train', 'within_configured_limit': 'False', 'N': '4150', 'S': '24', 'V': '0', 'Q': '40', 'S+V': '24'}, {'patient_id': 'p07070', 'record_id': 'p07070_s15', 'split': 'train', 'within_configured_limit': 'False', 'N': '3910', 'S': '41', 'V': '0', 'Q': '50', 'S+V': '41'}, {'patient_id': 'p07070', 'record_id': 'p07070_s16', 'split': 'train', 'within_configured_limit': 'False', 'N': '2529', 'S': '42', 'V': '0', 'Q': '2474', 'S+V': '42'}, {'patient_id': 'p07070', 'record_id': 'p07070_s17', 'split': 'train', 'within_configured_limit': 'False', 'N': '2429', 'S': '42', 'V': '0', 'Q': '3521', 'S+V': '42'}, {'patient_id': 'p07070', 'record_id': 'p07070_s18', 'split': 'train', 'within_configured_limit': 'False', 'N': '2611', 'S': '164', 'V': '0', 'Q': '3168', 'S+V': '164'}, {'patient_id': 'p07070', 'record_id': 'p07070_s19', 'split': 'train', 'within_configured_limit': 'False', 'N': '2918', 'S': '87', 'V': '0', 'Q': '2134', 'S+V': '87'}, {'patient_id': 'p07070', 'record_id': 'p07070_s20', 'split': 'train', 'within_configured_limit': 'False', 'N': '2119', 'S': '93', 'V': '0', 'Q': '3103', 'S+V': '93'}, {'patient_id': 'p07070', 'record_id': 'p07070_s21', 'split': 'train', 'within_configured_limit': 'False', 'N': '3122', 'S': '87', 'V': '0', 'Q': '1029', 'S+V': '87'}, {'patient_id': 'p07070', 'record_id': 'p07070_s22', 'split': 'train', 'within_configured_limit': 'False', 'N': '3788', 'S': '28', 'V': '0', 'Q': '129', 'S+V': '28'}, {'patient_id': 'p07070', 'record_id': 'p07070_s23', 'split': 'train', 'within_configured_limit': 'False', 'N': '3789', 'S': '77', 'V': '0', 'Q': '71', 'S+V': '77'}, {'patient_id': 'p07070', 'record_id': 'p07070_s24', 'split': 'train', 'within_configured_limit': 'False', 'N': '2576', 'S': '16', 'V': '1', 'Q': '3172', 'S+V': '17'}, {'patient_id': 'p07070', 'record_id': 'p07070_s25', 'split': 'train', 'within_configured_limit': 'False', 'N': '2871', 'S': '175', 'V': '0', 'Q': '2351', 'S+V': '175'}, {'patient_id': 'p07070', 'record_id': 'p07070_s26', 'split': 'train', 'within_configured_limit': 'False', 'N': '3336', 'S': '74', 'V': '0', 'Q': '2106', 'S+V': '74'}, {'patient_id': 'p07070', 'record_id': 'p07070_s27', 'split': 'train', 'within_configured_limit': 'False', 'N': '4075', 'S': '20', 'V': '0', 'Q': '118', 'S+V': '20'}, {'patient_id': 'p07070', 'record_id': 'p07070_s28', 'split': 'train', 'within_configured_limit': 'False', 'N': '2766', 'S': '13', 'V': '0', 'Q': '1816', 'S+V': '13'}, {'patient_id': 'p07070', 'record_id': 'p07070_s29', 'split': 'train', 'within_configured_limit': 'False', 'N': '2749', 'S': '13', 'V': '0', 'Q': '2409', 'S+V': '13'}, {'patient_id': 'p07070', 'record_id': 'p07070_s30', 'split': 'train', 'within_configured_limit': 'False', 'N': '3289', 'S': '36', 'V': '0', 'Q': '1020', 'S+V': '36'}, {'patient_id': 'p07070', 'record_id': 'p07070_s31', 'split': 'train', 'within_configured_limit': 'False', 'N': '3797', 'S': '27', 'V': '0', 'Q': '103', 'S+V': '27'}, {'patient_id': 'p07070', 'record_id': 'p07070_s32', 'split': 'train', 'within_configured_limit': 'False', 'N': '3817', 'S': '43', 'V': '0', 'Q': '548', 'S+V': '43'}, {'patient_id': 'p07070', 'record_id': 'p07070_s33', 'split': 'train', 'within_configured_limit': 'False', 'N': '3865', 'S': '6', 'V': '0', 'Q': '1938', 'S+V': '6'}, {'patient_id': 'p07070', 'record_id': 'p07070_s34', 'split': 'train', 'within_configured_limit': 'False', 'N': '3540', 'S': '19', 'V': '0', 'Q': '1432', 'S+V': '19'}, {'patient_id': 'p07070', 'record_id': 'p07070_s35', 'split': 'train', 'within_configured_limit': 'False', 'N': '3543', 'S': '22', 'V': '0', 'Q': '13', 'S+V': '22'}, {'patient_id': 'p07070', 'record_id': 'p07070_s36', 'split': 'train', 'within_configured_limit': 'False', 'N': '3407', 'S': '35', 'V': '0', 'Q': '94', 'S+V': '35'}, {'patient_id': 'p07070', 'record_id': 'p07070_s37', 'split': 'train', 'within_configured_limit': 'False', 'N': '2986', 'S': '4', 'V': '0', 'Q': '2373', 'S+V': '4'}, {'patient_id': 'p07070', 'record_id': 'p07070_s38', 'split': 'train', 'within_configured_limit': 'False', 'N': '2856', 'S': '27', 'V': '0', 'Q': '875', 'S+V': '27'}, {'patient_id': 'p07070', 'record_id': 'p07070_s39', 'split': 'train', 'within_configured_limit': 'False', 'N': '3455', 'S': '36', 'V': '0', 'Q': '388', 'S+V': '36'}, {'patient_id': 'p07070', 'record_id': 'p07070_s40', 'split': 'train', 'within_configured_limit': 'False', 'N': '2848', 'S': '18', 'V': '0', 'Q': '2170', 'S+V': '18'}, {'patient_id': 'p07070', 'record_id': 'p07070_s41', 'split': 'train', 'within_configured_limit': 'False', 'N': '1839', 'S': '23', 'V': '0', 'Q': '3041', 'S+V': '23'}, {'patient_id': 'p07070', 'record_id': 'p07070_s42', 'split': 'train', 'within_configured_limit': 'False', 'N': '3083', 'S': '8', 'V': '0', 'Q': '1798', 'S+V': '8'}, {'patient_id': 'p07070', 'record_id': 'p07070_s43', 'split': 'train', 'within_configured_limit': 'False', 'N': '3026', 'S': '14', 'V': '0', 'Q': '2015', 'S+V': '14'}, {'patient_id': 'p07070', 'record_id': 'p07070_s44', 'split': 'train', 'within_configured_limit': 'False', 'N': '3249', 'S': '7', 'V': '0', 'Q': '1380', 'S+V': '7'}, {'patient_id': 'p07070', 'record_id': 'p07070_s45', 'split': 'train', 'within_configured_limit': 'False', 'N': '3115', 'S': '8', 'V': '0', 'Q': '2516', 'S+V': '8'}, {'patient_id': 'p07070', 'record_id': 'p07070_s46', 'split': 'train', 'within_configured_limit': 'False', 'N': '3872', 'S': '18', 'V': '0', 'Q': '1453', 'S+V': '18'}, {'patient_id': 'p07070', 'record_id': 'p07070_s47', 'split': 'train', 'within_configured_limit': 'False', 'N': '2826', 'S': '15', 'V': '0', 'Q': '2494', 'S+V': '15'}, {'patient_id': 'p07070', 'record_id': 'p07070_s48', 'split': 'train', 'within_configured_limit': 'False', 'N': '3905', 'S': '20', 'V': '0', 'Q': '12', 'S+V': '20'}, {'patient_id': 'p07070', 'record_id': 'p07070_s49', 'split': 'train', 'within_configured_limit': 'False', 'N': '3229', 'S': '24', 'V': '2', 'Q': '2207', 'S+V': '26'}, {'patient_id': 'p07179', 'record_id': 'p07179_s00', 'split': 'train', 'within_configured_limit': 'True', 'N': '3934', 'S': '0', 'V': '0', 'Q': '2683', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s01', 'split': 'train', 'within_configured_limit': 'False', 'N': '6660', 'S': '0', 'V': '0', 'Q': '550', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s02', 'split': 'train', 'within_configured_limit': 'False', 'N': '2214', 'S': '0', 'V': '0', 'Q': '3155', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s03', 'split': 'train', 'within_configured_limit': 'False', 'N': '4811', 'S': '0', 'V': '0', 'Q': '291', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s04', 'split': 'train', 'within_configured_limit': 'False', 'N': '4393', 'S': '0', 'V': '0', 'Q': '32', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s05', 'split': 'train', 'within_configured_limit': 'False', 'N': '4352', 'S': '0', 'V': '0', 'Q': '22', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s06', 'split': 'train', 'within_configured_limit': 'False', 'N': '4634', 'S': '0', 'V': '0', 'Q': '0', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s07', 'split': 'train', 'within_configured_limit': 'False', 'N': '4435', 'S': '2', 'V': '0', 'Q': '13', 'S+V': '2'}, {'patient_id': 'p07179', 'record_id': 'p07179_s08', 'split': 'train', 'within_configured_limit': 'False', 'N': '3125', 'S': '1', 'V': '0', 'Q': '769', 'S+V': '1'}, {'patient_id': 'p07179', 'record_id': 'p07179_s09', 'split': 'train', 'within_configured_limit': 'False', 'N': '0', 'S': '0', 'V': '0', 'Q': '3287', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s10', 'split': 'train', 'within_configured_limit': 'False', 'N': '0', 'S': '0', 'V': '0', 'Q': '4192', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s11', 'split': 'train', 'within_configured_limit': 'False', 'N': '5366', 'S': '0', 'V': '0', 'Q': '851', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s12', 'split': 'train', 'within_configured_limit': 'False', 'N': '6479', 'S': '0', 'V': '0', 'Q': '313', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s13', 'split': 'train', 'within_configured_limit': 'False', 'N': '6621', 'S': '0', 'V': '0', 'Q': '616', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s14', 'split': 'train', 'within_configured_limit': 'False', 'N': '6341', 'S': '0', 'V': '0', 'Q': '523', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s15', 'split': 'train', 'within_configured_limit': 'False', 'N': '6463', 'S': '1', 'V': '0', 'Q': '598', 'S+V': '1'}, {'patient_id': 'p07179', 'record_id': 'p07179_s16', 'split': 'train', 'within_configured_limit': 'False', 'N': '6641', 'S': '0', 'V': '0', 'Q': '337', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s17', 'split': 'train', 'within_configured_limit': 'False', 'N': '6277', 'S': '0', 'V': '0', 'Q': '715', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s18', 'split': 'train', 'within_configured_limit': 'False', 'N': '7137', 'S': '1', 'V': '0', 'Q': '271', 'S+V': '1'}, {'patient_id': 'p07179', 'record_id': 'p07179_s19', 'split': 'train', 'within_configured_limit': 'False', 'N': '6363', 'S': '0', 'V': '0', 'Q': '1382', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s20', 'split': 'train', 'within_configured_limit': 'False', 'N': '6635', 'S': '0', 'V': '0', 'Q': '187', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s21', 'split': 'train', 'within_configured_limit': 'False', 'N': '6390', 'S': '1', 'V': '0', 'Q': '935', 'S+V': '1'}, {'patient_id': 'p07179', 'record_id': 'p07179_s22', 'split': 'train', 'within_configured_limit': 'False', 'N': '7093', 'S': '0', 'V': '0', 'Q': '375', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s23', 'split': 'train', 'within_configured_limit': 'False', 'N': '5969', 'S': '0', 'V': '0', 'Q': '334', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s24', 'split': 'train', 'within_configured_limit': 'False', 'N': '2830', 'S': '0', 'V': '0', 'Q': '2480', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s25', 'split': 'train', 'within_configured_limit': 'False', 'N': '58', 'S': '0', 'V': '0', 'Q': '5248', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s26', 'split': 'train', 'within_configured_limit': 'False', 'N': '0', 'S': '0', 'V': '0', 'Q': '6641', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s27', 'split': 'train', 'within_configured_limit': 'False', 'N': '0', 'S': '0', 'V': '0', 'Q': '5598', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s28', 'split': 'train', 'within_configured_limit': 'False', 'N': '0', 'S': '0', 'V': '0', 'Q': '6099', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s29', 'split': 'train', 'within_configured_limit': 'False', 'N': '0', 'S': '0', 'V': '0', 'Q': '5607', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s30', 'split': 'train', 'within_configured_limit': 'False', 'N': '0', 'S': '0', 'V': '0', 'Q': '5948', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s31', 'split': 'train', 'within_configured_limit': 'False', 'N': '0', 'S': '0', 'V': '0', 'Q': '5840', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s32', 'split': 'train', 'within_configured_limit': 'False', 'N': '3380', 'S': '0', 'V': '0', 'Q': '2189', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s33', 'split': 'train', 'within_configured_limit': 'False', 'N': '6304', 'S': '0', 'V': '0', 'Q': '219', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s34', 'split': 'train', 'within_configured_limit': 'False', 'N': '6213', 'S': '0', 'V': '0', 'Q': '299', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s35', 'split': 'train', 'within_configured_limit': 'False', 'N': '5703', 'S': '0', 'V': '0', 'Q': '410', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s36', 'split': 'train', 'within_configured_limit': 'False', 'N': '5786', 'S': '0', 'V': '0', 'Q': '562', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s37', 'split': 'train', 'within_configured_limit': 'False', 'N': '6031', 'S': '1', 'V': '0', 'Q': '510', 'S+V': '1'}, {'patient_id': 'p07179', 'record_id': 'p07179_s38', 'split': 'train', 'within_configured_limit': 'False', 'N': '5773', 'S': '0', 'V': '0', 'Q': '638', 'S+V': '0'}, {'patient_id': 'p07179', 'record_id': 'p07179_s39', 'split': 'train', 'within_configured_limit': 'False', 'N': '6202', 'S': '0', 'V': '0', 'Q': '1199', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s00', 'split': 'train', 'within_configured_limit': 'True', 'N': '2125', 'S': '0', 'V': '0', 'Q': '4107', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s01', 'split': 'train', 'within_configured_limit': 'False', 'N': '3231', 'S': '0', 'V': '0', 'Q': '2643', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s02', 'split': 'train', 'within_configured_limit': 'False', 'N': '4855', 'S': '1', 'V': '0', 'Q': '271', 'S+V': '1'}, {'patient_id': 'p07743', 'record_id': 'p07743_s03', 'split': 'train', 'within_configured_limit': 'False', 'N': '4040', 'S': '0', 'V': '0', 'Q': '1985', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s04', 'split': 'train', 'within_configured_limit': 'False', 'N': '4075', 'S': '0', 'V': '0', 'Q': '3958', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s05', 'split': 'train', 'within_configured_limit': 'False', 'N': '3979', 'S': '0', 'V': '0', 'Q': '2751', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s06', 'split': 'train', 'within_configured_limit': 'False', 'N': '3422', 'S': '0', 'V': '0', 'Q': '2351', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s07', 'split': 'train', 'within_configured_limit': 'False', 'N': '3358', 'S': '0', 'V': '0', 'Q': '2020', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s08', 'split': 'train', 'within_configured_limit': 'False', 'N': '4564', 'S': '0', 'V': '0', 'Q': '104', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s09', 'split': 'train', 'within_configured_limit': 'False', 'N': '5272', 'S': '0', 'V': '0', 'Q': '0', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s10', 'split': 'train', 'within_configured_limit': 'False', 'N': '4751', 'S': '0', 'V': '0', 'Q': '93', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s11', 'split': 'train', 'within_configured_limit': 'False', 'N': '4517', 'S': '0', 'V': '0', 'Q': '104', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s12', 'split': 'train', 'within_configured_limit': 'False', 'N': '4567', 'S': '0', 'V': '0', 'Q': '124', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s13', 'split': 'train', 'within_configured_limit': 'False', 'N': '4146', 'S': '0', 'V': '0', 'Q': '616', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s14', 'split': 'train', 'within_configured_limit': 'False', 'N': '4927', 'S': '0', 'V': '0', 'Q': '2709', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s15', 'split': 'train', 'within_configured_limit': 'False', 'N': '5187', 'S': '1', 'V': '0', 'Q': '2587', 'S+V': '1'}, {'patient_id': 'p07743', 'record_id': 'p07743_s16', 'split': 'train', 'within_configured_limit': 'False', 'N': '5422', 'S': '0', 'V': '0', 'Q': '2856', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s17', 'split': 'train', 'within_configured_limit': 'False', 'N': '5141', 'S': '1', 'V': '0', 'Q': '1084', 'S+V': '1'}, {'patient_id': 'p07743', 'record_id': 'p07743_s18', 'split': 'train', 'within_configured_limit': 'False', 'N': '4515', 'S': '1', 'V': '0', 'Q': '1022', 'S+V': '1'}, {'patient_id': 'p07743', 'record_id': 'p07743_s19', 'split': 'train', 'within_configured_limit': 'False', 'N': '4705', 'S': '0', 'V': '0', 'Q': '1076', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s20', 'split': 'train', 'within_configured_limit': 'False', 'N': '4195', 'S': '3', 'V': '0', 'Q': '2357', 'S+V': '3'}, {'patient_id': 'p07743', 'record_id': 'p07743_s21', 'split': 'train', 'within_configured_limit': 'False', 'N': '4601', 'S': '1', 'V': '0', 'Q': '393', 'S+V': '1'}, {'patient_id': 'p07743', 'record_id': 'p07743_s22', 'split': 'train', 'within_configured_limit': 'False', 'N': '4545', 'S': '0', 'V': '0', 'Q': '6', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s23', 'split': 'train', 'within_configured_limit': 'False', 'N': '4335', 'S': '0', 'V': '1', 'Q': '52', 'S+V': '1'}, {'patient_id': 'p07743', 'record_id': 'p07743_s24', 'split': 'train', 'within_configured_limit': 'False', 'N': '4201', 'S': '0', 'V': '0', 'Q': '0', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s25', 'split': 'train', 'within_configured_limit': 'False', 'N': '4109', 'S': '0', 'V': '0', 'Q': '98', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s26', 'split': 'train', 'within_configured_limit': 'False', 'N': '3930', 'S': '0', 'V': '0', 'Q': '57', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s27', 'split': 'train', 'within_configured_limit': 'False', 'N': '4202', 'S': '0', 'V': '0', 'Q': '117', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s28', 'split': 'train', 'within_configured_limit': 'False', 'N': '4101', 'S': '1', 'V': '0', 'Q': '245', 'S+V': '1'}, {'patient_id': 'p07743', 'record_id': 'p07743_s29', 'split': 'train', 'within_configured_limit': 'False', 'N': '4171', 'S': '0', 'V': '0', 'Q': '162', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s30', 'split': 'train', 'within_configured_limit': 'False', 'N': '4234', 'S': '0', 'V': '0', 'Q': '127', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s31', 'split': 'train', 'within_configured_limit': 'False', 'N': '4540', 'S': '0', 'V': '0', 'Q': '1297', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s32', 'split': 'train', 'within_configured_limit': 'False', 'N': '3769', 'S': '0', 'V': '0', 'Q': '2578', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s33', 'split': 'train', 'within_configured_limit': 'False', 'N': '4362', 'S': '0', 'V': '0', 'Q': '1354', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s34', 'split': 'train', 'within_configured_limit': 'False', 'N': '4413', 'S': '0', 'V': '0', 'Q': '1403', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s35', 'split': 'train', 'within_configured_limit': 'False', 'N': '3504', 'S': '0', 'V': '0', 'Q': '2624', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s36', 'split': 'train', 'within_configured_limit': 'False', 'N': '4178', 'S': '0', 'V': '0', 'Q': '1517', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s37', 'split': 'train', 'within_configured_limit': 'False', 'N': '3522', 'S': '0', 'V': '0', 'Q': '2794', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s38', 'split': 'train', 'within_configured_limit': 'False', 'N': '3505', 'S': '0', 'V': '0', 'Q': '3466', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s39', 'split': 'train', 'within_configured_limit': 'False', 'N': '4415', 'S': '0', 'V': '0', 'Q': '2642', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s40', 'split': 'train', 'within_configured_limit': 'False', 'N': '3699', 'S': '0', 'V': '0', 'Q': '1011', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s41', 'split': 'train', 'within_configured_limit': 'False', 'N': '3675', 'S': '0', 'V': '0', 'Q': '1992', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s42', 'split': 'train', 'within_configured_limit': 'False', 'N': '4274', 'S': '1', 'V': '0', 'Q': '714', 'S+V': '1'}, {'patient_id': 'p07743', 'record_id': 'p07743_s43', 'split': 'train', 'within_configured_limit': 'False', 'N': '4591', 'S': '0', 'V': '0', 'Q': '524', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s44', 'split': 'train', 'within_configured_limit': 'False', 'N': '4424', 'S': '0', 'V': '0', 'Q': '169', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s45', 'split': 'train', 'within_configured_limit': 'False', 'N': '4232', 'S': '0', 'V': '0', 'Q': '107', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s46', 'split': 'train', 'within_configured_limit': 'False', 'N': '4096', 'S': '0', 'V': '0', 'Q': '78', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s47', 'split': 'train', 'within_configured_limit': 'False', 'N': '3901', 'S': '0', 'V': '0', 'Q': '137', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s48', 'split': 'train', 'within_configured_limit': 'False', 'N': '3266', 'S': '0', 'V': '0', 'Q': '1967', 'S+V': '0'}, {'patient_id': 'p07743', 'record_id': 'p07743_s49', 'split': 'train', 'within_configured_limit': 'False', 'N': '2927', 'S': '0', 'V': '0', 'Q': '2960', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s00', 'split': 'train', 'within_configured_limit': 'True', 'N': '3918', 'S': '0', 'V': '0', 'Q': '1025', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s01', 'split': 'train', 'within_configured_limit': 'False', 'N': '4077', 'S': '1', 'V': '0', 'Q': '137', 'S+V': '1'}, {'patient_id': 'p07929', 'record_id': 'p07929_s02', 'split': 'train', 'within_configured_limit': 'False', 'N': '3667', 'S': '2', 'V': '0', 'Q': '856', 'S+V': '2'}, {'patient_id': 'p07929', 'record_id': 'p07929_s03', 'split': 'train', 'within_configured_limit': 'False', 'N': '4261', 'S': '0', 'V': '0', 'Q': '25', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s04', 'split': 'train', 'within_configured_limit': 'False', 'N': '4241', 'S': '0', 'V': '0', 'Q': '76', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s05', 'split': 'train', 'within_configured_limit': 'False', 'N': '4103', 'S': '0', 'V': '0', 'Q': '36', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s06', 'split': 'train', 'within_configured_limit': 'False', 'N': '4142', 'S': '0', 'V': '0', 'Q': '95', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s07', 'split': 'train', 'within_configured_limit': 'False', 'N': '4331', 'S': '0', 'V': '0', 'Q': '277', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s08', 'split': 'train', 'within_configured_limit': 'False', 'N': '3613', 'S': '0', 'V': '0', 'Q': '1376', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s09', 'split': 'train', 'within_configured_limit': 'False', 'N': '4144', 'S': '0', 'V': '0', 'Q': '1648', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s10', 'split': 'train', 'within_configured_limit': 'False', 'N': '3773', 'S': '0', 'V': '0', 'Q': '1632', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s11', 'split': 'train', 'within_configured_limit': 'False', 'N': '4295', 'S': '1', 'V': '0', 'Q': '161', 'S+V': '1'}, {'patient_id': 'p07929', 'record_id': 'p07929_s12', 'split': 'train', 'within_configured_limit': 'False', 'N': '4416', 'S': '2', 'V': '0', 'Q': '57', 'S+V': '2'}, {'patient_id': 'p07929', 'record_id': 'p07929_s13', 'split': 'train', 'within_configured_limit': 'False', 'N': '4232', 'S': '1', 'V': '0', 'Q': '248', 'S+V': '1'}, {'patient_id': 'p07929', 'record_id': 'p07929_s14', 'split': 'train', 'within_configured_limit': 'False', 'N': '4555', 'S': '1', 'V': '0', 'Q': '66', 'S+V': '1'}, {'patient_id': 'p07929', 'record_id': 'p07929_s15', 'split': 'train', 'within_configured_limit': 'False', 'N': '4336', 'S': '4', 'V': '0', 'Q': '110', 'S+V': '4'}, {'patient_id': 'p07929', 'record_id': 'p07929_s16', 'split': 'train', 'within_configured_limit': 'False', 'N': '4355', 'S': '1', 'V': '0', 'Q': '206', 'S+V': '1'}, {'patient_id': 'p07929', 'record_id': 'p07929_s17', 'split': 'train', 'within_configured_limit': 'False', 'N': '3732', 'S': '0', 'V': '0', 'Q': '1501', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s18', 'split': 'train', 'within_configured_limit': 'False', 'N': '4423', 'S': '0', 'V': '0', 'Q': '1526', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s19', 'split': 'train', 'within_configured_limit': 'False', 'N': '4625', 'S': '0', 'V': '0', 'Q': '412', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s20', 'split': 'train', 'within_configured_limit': 'False', 'N': '4293', 'S': '0', 'V': '0', 'Q': '242', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s21', 'split': 'train', 'within_configured_limit': 'False', 'N': '3228', 'S': '1', 'V': '0', 'Q': '2695', 'S+V': '1'}, {'patient_id': 'p07929', 'record_id': 'p07929_s22', 'split': 'train', 'within_configured_limit': 'False', 'N': '4800', 'S': '1', 'V': '0', 'Q': '134', 'S+V': '1'}, {'patient_id': 'p07929', 'record_id': 'p07929_s23', 'split': 'train', 'within_configured_limit': 'False', 'N': '4310', 'S': '0', 'V': '0', 'Q': '54', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s24', 'split': 'train', 'within_configured_limit': 'False', 'N': '4285', 'S': '1', 'V': '0', 'Q': '591', 'S+V': '1'}, {'patient_id': 'p07929', 'record_id': 'p07929_s25', 'split': 'train', 'within_configured_limit': 'False', 'N': '4207', 'S': '1', 'V': '0', 'Q': '454', 'S+V': '1'}, {'patient_id': 'p07929', 'record_id': 'p07929_s26', 'split': 'train', 'within_configured_limit': 'False', 'N': '3143', 'S': '0', 'V': '0', 'Q': '2739', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s27', 'split': 'train', 'within_configured_limit': 'False', 'N': '3842', 'S': '1', 'V': '0', 'Q': '1743', 'S+V': '1'}, {'patient_id': 'p07929', 'record_id': 'p07929_s28', 'split': 'train', 'within_configured_limit': 'False', 'N': '4001', 'S': '0', 'V': '0', 'Q': '1168', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s29', 'split': 'train', 'within_configured_limit': 'False', 'N': '4587', 'S': '0', 'V': '0', 'Q': '33', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s30', 'split': 'train', 'within_configured_limit': 'False', 'N': '4388', 'S': '1', 'V': '0', 'Q': '53', 'S+V': '1'}, {'patient_id': 'p07929', 'record_id': 'p07929_s31', 'split': 'train', 'within_configured_limit': 'False', 'N': '4132', 'S': '0', 'V': '0', 'Q': '30', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s32', 'split': 'train', 'within_configured_limit': 'False', 'N': '4023', 'S': '1', 'V': '0', 'Q': '42', 'S+V': '1'}, {'patient_id': 'p07929', 'record_id': 'p07929_s33', 'split': 'train', 'within_configured_limit': 'False', 'N': '4389', 'S': '1', 'V': '0', 'Q': '533', 'S+V': '1'}, {'patient_id': 'p07929', 'record_id': 'p07929_s34', 'split': 'train', 'within_configured_limit': 'False', 'N': '4449', 'S': '0', 'V': '0', 'Q': '336', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s35', 'split': 'train', 'within_configured_limit': 'False', 'N': '3973', 'S': '0', 'V': '0', 'Q': '63', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s36', 'split': 'train', 'within_configured_limit': 'False', 'N': '3654', 'S': '0', 'V': '0', 'Q': '970', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s37', 'split': 'train', 'within_configured_limit': 'False', 'N': '3661', 'S': '0', 'V': '0', 'Q': '199', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s38', 'split': 'train', 'within_configured_limit': 'False', 'N': '3962', 'S': '1', 'V': '0', 'Q': '216', 'S+V': '1'}, {'patient_id': 'p07929', 'record_id': 'p07929_s39', 'split': 'train', 'within_configured_limit': 'False', 'N': '4314', 'S': '1', 'V': '0', 'Q': '29', 'S+V': '1'}, {'patient_id': 'p07929', 'record_id': 'p07929_s40', 'split': 'train', 'within_configured_limit': 'False', 'N': '4510', 'S': '0', 'V': '0', 'Q': '155', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s41', 'split': 'train', 'within_configured_limit': 'False', 'N': '4117', 'S': '0', 'V': '0', 'Q': '931', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s42', 'split': 'train', 'within_configured_limit': 'False', 'N': '4363', 'S': '0', 'V': '0', 'Q': '451', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s43', 'split': 'train', 'within_configured_limit': 'False', 'N': '4087', 'S': '0', 'V': '0', 'Q': '368', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s44', 'split': 'train', 'within_configured_limit': 'False', 'N': '3943', 'S': '0', 'V': '0', 'Q': '627', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s45', 'split': 'train', 'within_configured_limit': 'False', 'N': '4723', 'S': '0', 'V': '0', 'Q': '1109', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s46', 'split': 'train', 'within_configured_limit': 'False', 'N': '4272', 'S': '1', 'V': '0', 'Q': '486', 'S+V': '1'}, {'patient_id': 'p07929', 'record_id': 'p07929_s47', 'split': 'train', 'within_configured_limit': 'False', 'N': '4385', 'S': '1', 'V': '1', 'Q': '442', 'S+V': '2'}, {'patient_id': 'p07929', 'record_id': 'p07929_s48', 'split': 'train', 'within_configured_limit': 'False', 'N': '3964', 'S': '0', 'V': '0', 'Q': '858', 'S+V': '0'}, {'patient_id': 'p07929', 'record_id': 'p07929_s49', 'split': 'train', 'within_configured_limit': 'False', 'N': '4249', 'S': '2', 'V': '0', 'Q': '816', 'S+V': '2'}, {'patient_id': 'p08689', 'record_id': 'p08689_s00', 'split': 'train', 'within_configured_limit': 'True', 'N': '2103', 'S': '0', 'V': '169', 'Q': '2608', 'S+V': '169'}, {'patient_id': 'p08689', 'record_id': 'p08689_s01', 'split': 'train', 'within_configured_limit': 'False', 'N': '2675', 'S': '0', 'V': '343', 'Q': '2227', 'S+V': '343'}, {'patient_id': 'p08689', 'record_id': 'p08689_s02', 'split': 'train', 'within_configured_limit': 'False', 'N': '2355', 'S': '0', 'V': '356', 'Q': '2231', 'S+V': '356'}, {'patient_id': 'p08689', 'record_id': 'p08689_s03', 'split': 'train', 'within_configured_limit': 'False', 'N': '2702', 'S': '0', 'V': '331', 'Q': '1736', 'S+V': '331'}, {'patient_id': 'p08689', 'record_id': 'p08689_s04', 'split': 'train', 'within_configured_limit': 'False', 'N': '2589', 'S': '0', 'V': '338', 'Q': '2269', 'S+V': '338'}, {'patient_id': 'p08689', 'record_id': 'p08689_s05', 'split': 'train', 'within_configured_limit': 'False', 'N': '2816', 'S': '0', 'V': '398', 'Q': '1551', 'S+V': '398'}, {'patient_id': 'p08689', 'record_id': 'p08689_s06', 'split': 'train', 'within_configured_limit': 'False', 'N': '3352', 'S': '0', 'V': '72', 'Q': '57', 'S+V': '72'}, {'patient_id': 'p08689', 'record_id': 'p08689_s07', 'split': 'train', 'within_configured_limit': 'False', 'N': '1851', 'S': '0', 'V': '51', 'Q': '1895', 'S+V': '51'}, {'patient_id': 'p08689', 'record_id': 'p08689_s08', 'split': 'train', 'within_configured_limit': 'False', 'N': '3288', 'S': '0', 'V': '110', 'Q': '235', 'S+V': '110'}, {'patient_id': 'p08689', 'record_id': 'p08689_s09', 'split': 'train', 'within_configured_limit': 'False', 'N': '3314', 'S': '0', 'V': '81', 'Q': '48', 'S+V': '81'}, {'patient_id': 'p08689', 'record_id': 'p08689_s10', 'split': 'train', 'within_configured_limit': 'False', 'N': '3065', 'S': '0', 'V': '42', 'Q': '33', 'S+V': '42'}, {'patient_id': 'p08689', 'record_id': 'p08689_s11', 'split': 'train', 'within_configured_limit': 'False', 'N': '3063', 'S': '0', 'V': '52', 'Q': '70', 'S+V': '52'}, {'patient_id': 'p08689', 'record_id': 'p08689_s12', 'split': 'train', 'within_configured_limit': 'False', 'N': '2932', 'S': '0', 'V': '53', 'Q': '1172', 'S+V': '53'}, {'patient_id': 'p08689', 'record_id': 'p08689_s13', 'split': 'train', 'within_configured_limit': 'False', 'N': '3979', 'S': '0', 'V': '105', 'Q': '436', 'S+V': '105'}, {'patient_id': 'p08689', 'record_id': 'p08689_s14', 'split': 'train', 'within_configured_limit': 'False', 'N': '2758', 'S': '0', 'V': '56', 'Q': '2449', 'S+V': '56'}, {'patient_id': 'p08689', 'record_id': 'p08689_s15', 'split': 'train', 'within_configured_limit': 'False', 'N': '3384', 'S': '0', 'V': '228', 'Q': '1268', 'S+V': '228'}, {'patient_id': 'p08689', 'record_id': 'p08689_s16', 'split': 'train', 'within_configured_limit': 'False', 'N': '2772', 'S': '0', 'V': '206', 'Q': '2456', 'S+V': '206'}, {'patient_id': 'p08689', 'record_id': 'p08689_s17', 'split': 'train', 'within_configured_limit': 'False', 'N': '3724', 'S': '0', 'V': '442', 'Q': '397', 'S+V': '442'}, {'patient_id': 'p08689', 'record_id': 'p08689_s18', 'split': 'train', 'within_configured_limit': 'False', 'N': '3781', 'S': '0', 'V': '255', 'Q': '292', 'S+V': '255'}, {'patient_id': 'p08689', 'record_id': 'p08689_s19', 'split': 'train', 'within_configured_limit': 'False', 'N': '3645', 'S': '0', 'V': '98', 'Q': '226', 'S+V': '98'}, {'patient_id': 'p08689', 'record_id': 'p08689_s20', 'split': 'train', 'within_configured_limit': 'False', 'N': '3383', 'S': '0', 'V': '64', 'Q': '204', 'S+V': '64'}, {'patient_id': 'p08689', 'record_id': 'p08689_s21', 'split': 'train', 'within_configured_limit': 'False', 'N': '3565', 'S': '0', 'V': '141', 'Q': '263', 'S+V': '141'}, {'patient_id': 'p08689', 'record_id': 'p08689_s22', 'split': 'train', 'within_configured_limit': 'False', 'N': '3546', 'S': '0', 'V': '93', 'Q': '56', 'S+V': '93'}, {'patient_id': 'p08689', 'record_id': 'p08689_s23', 'split': 'train', 'within_configured_limit': 'False', 'N': '3445', 'S': '0', 'V': '89', 'Q': '26', 'S+V': '89'}, {'patient_id': 'p08689', 'record_id': 'p08689_s24', 'split': 'train', 'within_configured_limit': 'False', 'N': '3315', 'S': '0', 'V': '74', 'Q': '108', 'S+V': '74'}, {'patient_id': 'p08689', 'record_id': 'p08689_s25', 'split': 'train', 'within_configured_limit': 'False', 'N': '3008', 'S': '0', 'V': '141', 'Q': '1249', 'S+V': '141'}, {'patient_id': 'p08689', 'record_id': 'p08689_s26', 'split': 'train', 'within_configured_limit': 'False', 'N': '2761', 'S': '0', 'V': '114', 'Q': '2277', 'S+V': '114'}, {'patient_id': 'p08689', 'record_id': 'p08689_s27', 'split': 'train', 'within_configured_limit': 'False', 'N': '2909', 'S': '0', 'V': '92', 'Q': '2340', 'S+V': '92'}, {'patient_id': 'p08689', 'record_id': 'p08689_s28', 'split': 'train', 'within_configured_limit': 'False', 'N': '2851', 'S': '0', 'V': '103', 'Q': '1905', 'S+V': '103'}, {'patient_id': 'p08689', 'record_id': 'p08689_s29', 'split': 'train', 'within_configured_limit': 'False', 'N': '2411', 'S': '0', 'V': '199', 'Q': '2344', 'S+V': '199'}, {'patient_id': 'p08689', 'record_id': 'p08689_s30', 'split': 'train', 'within_configured_limit': 'False', 'N': '1267', 'S': '0', 'V': '207', 'Q': '3918', 'S+V': '207'}, {'patient_id': 'p08689', 'record_id': 'p08689_s31', 'split': 'train', 'within_configured_limit': 'False', 'N': '2133', 'S': '0', 'V': '323', 'Q': '2655', 'S+V': '323'}, {'patient_id': 'p08689', 'record_id': 'p08689_s32', 'split': 'train', 'within_configured_limit': 'False', 'N': '2789', 'S': '0', 'V': '215', 'Q': '1797', 'S+V': '215'}, {'patient_id': 'p08689', 'record_id': 'p08689_s33', 'split': 'train', 'within_configured_limit': 'False', 'N': '3136', 'S': '0', 'V': '295', 'Q': '1107', 'S+V': '295'}, {'patient_id': 'p08689', 'record_id': 'p08689_s34', 'split': 'train', 'within_configured_limit': 'False', 'N': '3308', 'S': '0', 'V': '159', 'Q': '562', 'S+V': '159'}, {'patient_id': 'p08689', 'record_id': 'p08689_s35', 'split': 'train', 'within_configured_limit': 'False', 'N': '3407', 'S': '0', 'V': '151', 'Q': '488', 'S+V': '151'}, {'patient_id': 'p08689', 'record_id': 'p08689_s36', 'split': 'train', 'within_configured_limit': 'False', 'N': '3682', 'S': '0', 'V': '335', 'Q': '289', 'S+V': '335'}, {'patient_id': 'p08689', 'record_id': 'p08689_s37', 'split': 'train', 'within_configured_limit': 'False', 'N': '3381', 'S': '0', 'V': '163', 'Q': '678', 'S+V': '163'}, {'patient_id': 'p08689', 'record_id': 'p08689_s38', 'split': 'train', 'within_configured_limit': 'False', 'N': '3394', 'S': '0', 'V': '52', 'Q': '14', 'S+V': '52'}, {'patient_id': 'p08689', 'record_id': 'p08689_s39', 'split': 'train', 'within_configured_limit': 'False', 'N': '3338', 'S': '0', 'V': '38', 'Q': '24', 'S+V': '38'}, {'patient_id': 'p08689', 'record_id': 'p08689_s40', 'split': 'train', 'within_configured_limit': 'False', 'N': '3460', 'S': '0', 'V': '72', 'Q': '208', 'S+V': '72'}, {'patient_id': 'p08689', 'record_id': 'p08689_s41', 'split': 'train', 'within_configured_limit': 'False', 'N': '3327', 'S': '0', 'V': '113', 'Q': '101', 'S+V': '113'}, {'patient_id': 'p08689', 'record_id': 'p08689_s42', 'split': 'train', 'within_configured_limit': 'False', 'N': '2978', 'S': '0', 'V': '37', 'Q': '451', 'S+V': '37'}, {'patient_id': 'p08689', 'record_id': 'p08689_s43', 'split': 'train', 'within_configured_limit': 'False', 'N': '3124', 'S': '0', 'V': '112', 'Q': '718', 'S+V': '112'}, {'patient_id': 'p08689', 'record_id': 'p08689_s44', 'split': 'train', 'within_configured_limit': 'False', 'N': '3294', 'S': '0', 'V': '67', 'Q': '632', 'S+V': '67'}, {'patient_id': 'p08689', 'record_id': 'p08689_s45', 'split': 'train', 'within_configured_limit': 'False', 'N': '2826', 'S': '0', 'V': '70', 'Q': '1828', 'S+V': '70'}, {'patient_id': 'p08689', 'record_id': 'p08689_s46', 'split': 'train', 'within_configured_limit': 'False', 'N': '1497', 'S': '0', 'V': '48', 'Q': '4086', 'S+V': '48'}, {'patient_id': 'p08689', 'record_id': 'p08689_s47', 'split': 'train', 'within_configured_limit': 'False', 'N': '2031', 'S': '0', 'V': '148', 'Q': '3394', 'S+V': '148'}, {'patient_id': 'p08689', 'record_id': 'p08689_s48', 'split': 'train', 'within_configured_limit': 'False', 'N': '1622', 'S': '0', 'V': '147', 'Q': '3847', 'S+V': '147'}, {'patient_id': 'p08689', 'record_id': 'p08689_s49', 'split': 'train', 'within_configured_limit': 'False', 'N': '2619', 'S': '0', 'V': '315', 'Q': '2595', 'S+V': '315'}, {'patient_id': 'p09134', 'record_id': 'p09134_s00', 'split': 'dev', 'within_configured_limit': 'True', 'N': '3555', 'S': '0', 'V': '0', 'Q': '1790', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s01', 'split': 'dev', 'within_configured_limit': 'False', 'N': '885', 'S': '0', 'V': '0', 'Q': '4891', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s02', 'split': 'dev', 'within_configured_limit': 'False', 'N': '1461', 'S': '0', 'V': '0', 'Q': '3647', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s03', 'split': 'dev', 'within_configured_limit': 'False', 'N': '2446', 'S': '0', 'V': '0', 'Q': '2478', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s04', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4781', 'S': '1', 'V': '0', 'Q': '176', 'S+V': '1'}, {'patient_id': 'p09134', 'record_id': 'p09134_s05', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4321', 'S': '0', 'V': '0', 'Q': '60', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s06', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3869', 'S': '0', 'V': '0', 'Q': '359', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s07', 'split': 'dev', 'within_configured_limit': 'False', 'N': '1854', 'S': '0', 'V': '0', 'Q': '2756', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s08', 'split': 'dev', 'within_configured_limit': 'False', 'N': '841', 'S': '0', 'V': '0', 'Q': '4808', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s09', 'split': 'dev', 'within_configured_limit': 'False', 'N': '1474', 'S': '0', 'V': '7', 'Q': '4302', 'S+V': '7'}, {'patient_id': 'p09134', 'record_id': 'p09134_s10', 'split': 'dev', 'within_configured_limit': 'False', 'N': '2278', 'S': '0', 'V': '0', 'Q': '3071', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s11', 'split': 'dev', 'within_configured_limit': 'False', 'N': '1403', 'S': '0', 'V': '0', 'Q': '3573', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s12', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3756', 'S': '0', 'V': '0', 'Q': '727', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s13', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4523', 'S': '0', 'V': '0', 'Q': '59', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s14', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3898', 'S': '0', 'V': '0', 'Q': '355', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s15', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3873', 'S': '0', 'V': '0', 'Q': '67', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s16', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3886', 'S': '0', 'V': '0', 'Q': '41', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s17', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3866', 'S': '0', 'V': '0', 'Q': '66', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s18', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3624', 'S': '0', 'V': '0', 'Q': '253', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s19', 'split': 'dev', 'within_configured_limit': 'False', 'N': '338', 'S': '0', 'V': '0', 'Q': '5484', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s20', 'split': 'dev', 'within_configured_limit': 'False', 'N': '637', 'S': '0', 'V': '0', 'Q': '4701', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s21', 'split': 'dev', 'within_configured_limit': 'False', 'N': '899', 'S': '0', 'V': '0', 'Q': '4785', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s22', 'split': 'dev', 'within_configured_limit': 'False', 'N': '919', 'S': '0', 'V': '0', 'Q': '4424', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s23', 'split': 'dev', 'within_configured_limit': 'False', 'N': '2086', 'S': '0', 'V': '0', 'Q': '2827', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s24', 'split': 'dev', 'within_configured_limit': 'False', 'N': '1890', 'S': '0', 'V': '0', 'Q': '3263', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s25', 'split': 'dev', 'within_configured_limit': 'False', 'N': '5010', 'S': '0', 'V': '0', 'Q': '25', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s26', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4208', 'S': '0', 'V': '0', 'Q': '316', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s27', 'split': 'dev', 'within_configured_limit': 'False', 'N': '2050', 'S': '0', 'V': '0', 'Q': '3370', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s28', 'split': 'dev', 'within_configured_limit': 'False', 'N': '1877', 'S': '0', 'V': '0', 'Q': '3712', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s29', 'split': 'dev', 'within_configured_limit': 'False', 'N': '494', 'S': '0', 'V': '0', 'Q': '5311', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s30', 'split': 'dev', 'within_configured_limit': 'False', 'N': '1035', 'S': '0', 'V': '0', 'Q': '4281', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s31', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3166', 'S': '0', 'V': '0', 'Q': '1238', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s32', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3992', 'S': '0', 'V': '0', 'Q': '33', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s33', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3911', 'S': '0', 'V': '0', 'Q': '51', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s34', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3895', 'S': '0', 'V': '0', 'Q': '33', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s35', 'split': 'dev', 'within_configured_limit': 'False', 'N': '1385', 'S': '0', 'V': '0', 'Q': '3818', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s36', 'split': 'dev', 'within_configured_limit': 'False', 'N': '2900', 'S': '0', 'V': '0', 'Q': '2150', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s37', 'split': 'dev', 'within_configured_limit': 'False', 'N': '5158', 'S': '0', 'V': '0', 'Q': '92', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s38', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4872', 'S': '0', 'V': '0', 'Q': '379', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s39', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4695', 'S': '0', 'V': '0', 'Q': '86', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s40', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4463', 'S': '0', 'V': '0', 'Q': '310', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s41', 'split': 'dev', 'within_configured_limit': 'False', 'N': '4511', 'S': '1', 'V': '0', 'Q': '206', 'S+V': '1'}, {'patient_id': 'p09134', 'record_id': 'p09134_s42', 'split': 'dev', 'within_configured_limit': 'False', 'N': '1758', 'S': '0', 'V': '0', 'Q': '3817', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s43', 'split': 'dev', 'within_configured_limit': 'False', 'N': '1439', 'S': '0', 'V': '0', 'Q': '3281', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s44', 'split': 'dev', 'within_configured_limit': 'False', 'N': '1997', 'S': '0', 'V': '0', 'Q': '3549', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s45', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3232', 'S': '0', 'V': '0', 'Q': '1595', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s46', 'split': 'dev', 'within_configured_limit': 'False', 'N': '1233', 'S': '0', 'V': '0', 'Q': '4241', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s47', 'split': 'dev', 'within_configured_limit': 'False', 'N': '2952', 'S': '0', 'V': '0', 'Q': '1620', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s48', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3814', 'S': '0', 'V': '0', 'Q': '55', 'S+V': '0'}, {'patient_id': 'p09134', 'record_id': 'p09134_s49', 'split': 'dev', 'within_configured_limit': 'False', 'N': '3587', 'S': '0', 'V': '0', 'Q': '307', 'S+V': '0'}, {'patient_id': 'p09354', 'record_id': 'p09354_s00', 'split': 'train', 'within_configured_limit': 'True', 'N': '1448', 'S': '0', 'V': '1', 'Q': '4449', 'S+V': '1'}, {'patient_id': 'p09354', 'record_id': 'p09354_s01', 'split': 'train', 'within_configured_limit': 'False', 'N': '1892', 'S': '0', 'V': '0', 'Q': '3885', 'S+V': '0'}, {'patient_id': 'p09354', 'record_id': 'p09354_s02', 'split': 'train', 'within_configured_limit': 'False', 'N': '3980', 'S': '0', 'V': '1', 'Q': '1171', 'S+V': '1'}, {'patient_id': 'p09354', 'record_id': 'p09354_s03', 'split': 'train', 'within_configured_limit': 'False', 'N': '3753', 'S': '0', 'V': '1', 'Q': '1207', 'S+V': '1'}, {'patient_id': 'p09354', 'record_id': 'p09354_s04', 'split': 'train', 'within_configured_limit': 'False', 'N': '1783', 'S': '0', 'V': '4', 'Q': '4574', 'S+V': '4'}, {'patient_id': 'p09354', 'record_id': 'p09354_s05', 'split': 'train', 'within_configured_limit': 'False', 'N': '4042', 'S': '0', 'V': '9', 'Q': '2297', 'S+V': '9'}, {'patient_id': 'p09354', 'record_id': 'p09354_s06', 'split': 'train', 'within_configured_limit': 'False', 'N': '4167', 'S': '2', 'V': '3', 'Q': '1307', 'S+V': '5'}, {'patient_id': 'p09354', 'record_id': 'p09354_s07', 'split': 'train', 'within_configured_limit': 'False', 'N': '2745', 'S': '0', 'V': '1', 'Q': '2793', 'S+V': '1'}, {'patient_id': 'p09354', 'record_id': 'p09354_s08', 'split': 'train', 'within_configured_limit': 'False', 'N': '1544', 'S': '1', 'V': '0', 'Q': '5260', 'S+V': '1'}, {'patient_id': 'p09354', 'record_id': 'p09354_s09', 'split': 'train', 'within_configured_limit': 'False', 'N': '3983', 'S': '0', 'V': '3', 'Q': '1000', 'S+V': '3'}, {'patient_id': 'p09354', 'record_id': 'p09354_s10', 'split': 'train', 'within_configured_limit': 'False', 'N': '2772', 'S': '1', 'V': '0', 'Q': '1919', 'S+V': '1'}, {'patient_id': 'p09354', 'record_id': 'p09354_s11', 'split': 'train', 'within_configured_limit': 'False', 'N': '4116', 'S': '4', 'V': '1', 'Q': '96', 'S+V': '5'}, {'patient_id': 'p09354', 'record_id': 'p09354_s12', 'split': 'train', 'within_configured_limit': 'False', 'N': '4726', 'S': '0', 'V': '1', 'Q': '43', 'S+V': '1'}, {'patient_id': 'p09354', 'record_id': 'p09354_s13', 'split': 'train', 'within_configured_limit': 'False', 'N': '4913', 'S': '1', 'V': '0', 'Q': '0', 'S+V': '1'}, {'patient_id': 'p09354', 'record_id': 'p09354_s14', 'split': 'train', 'within_configured_limit': 'False', 'N': '4863', 'S': '1', 'V': '0', 'Q': '8', 'S+V': '1'}, {'patient_id': 'p09354', 'record_id': 'p09354_s15', 'split': 'train', 'within_configured_limit': 'False', 'N': '4629', 'S': '1', 'V': '0', 'Q': '19', 'S+V': '1'}, {'patient_id': 'p09354', 'record_id': 'p09354_s16', 'split': 'train', 'within_configured_limit': 'False', 'N': '4347', 'S': '1', 'V': '0', 'Q': '16', 'S+V': '1'}, {'patient_id': 'p09354', 'record_id': 'p09354_s17', 'split': 'train', 'within_configured_limit': 'False', 'N': '4354', 'S': '1', 'V': '0', 'Q': '13', 'S+V': '1'}, {'patient_id': 'p09354', 'record_id': 'p09354_s18', 'split': 'train', 'within_configured_limit': 'False', 'N': '4831', 'S': '0', 'V': '1', 'Q': '243', 'S+V': '1'}, {'patient_id': 'p09354', 'record_id': 'p09354_s19', 'split': 'train', 'within_configured_limit': 'False', 'N': '4348', 'S': '1', 'V': '2', 'Q': '15', 'S+V': '3'}, {'patient_id': 'p09354', 'record_id': 'p09354_s20', 'split': 'train', 'within_configured_limit': 'False', 'N': '5052', 'S': '0', 'V': '0', 'Q': '349', 'S+V': '0'}, {'patient_id': 'p09354', 'record_id': 'p09354_s21', 'split': 'train', 'within_configured_limit': 'False', 'N': '2683', 'S': '1', 'V': '0', 'Q': '2443', 'S+V': '1'}, {'patient_id': 'p09354', 'record_id': 'p09354_s22', 'split': 'train', 'within_configured_limit': 'False', 'N': '1486', 'S': '0', 'V': '0', 'Q': '3481', 'S+V': '0'}, {'patient_id': 'p09354', 'record_id': 'p09354_s23', 'split': 'train', 'within_configured_limit': 'False', 'N': '3042', 'S': '0', 'V': '1', 'Q': '2106', 'S+V': '1'}, {'patient_id': 'p09354', 'record_id': 'p09354_s24', 'split': 'train', 'within_configured_limit': 'False', 'N': '4826', 'S': '0', 'V': '0', 'Q': '429', 'S+V': '0'}, {'patient_id': 'p09354', 'record_id': 'p09354_s25', 'split': 'train', 'within_configured_limit': 'False', 'N': '5037', 'S': '0', 'V': '0', 'Q': '173', 'S+V': '0'}, {'patient_id': 'p09354', 'record_id': 'p09354_s26', 'split': 'train', 'within_configured_limit': 'False', 'N': '4877', 'S': '1', 'V': '3', 'Q': '274', 'S+V': '4'}, {'patient_id': 'p09354', 'record_id': 'p09354_s27', 'split': 'train', 'within_configured_limit': 'False', 'N': '4832', 'S': '2', 'V': '7', 'Q': '415', 'S+V': '9'}, {'patient_id': 'p09354', 'record_id': 'p09354_s28', 'split': 'train', 'within_configured_limit': 'False', 'N': '5145', 'S': '1', 'V': '8', 'Q': '514', 'S+V': '9'}, {'patient_id': 'p09354', 'record_id': 'p09354_s29', 'split': 'train', 'within_configured_limit': 'False', 'N': '4012', 'S': '1', 'V': '1', 'Q': '517', 'S+V': '2'}, {'patient_id': 'p09354', 'record_id': 'p09354_s30', 'split': 'train', 'within_configured_limit': 'False', 'N': '3701', 'S': '1', 'V': '0', 'Q': '752', 'S+V': '1'}, {'patient_id': 'p09354', 'record_id': 'p09354_s31', 'split': 'train', 'within_configured_limit': 'False', 'N': '3362', 'S': '1', 'V': '1', 'Q': '967', 'S+V': '2'}, {'patient_id': 'p09354', 'record_id': 'p09354_s32', 'split': 'train', 'within_configured_limit': 'False', 'N': '3821', 'S': '1', 'V': '2', 'Q': '70', 'S+V': '3'}, {'patient_id': 'p09354', 'record_id': 'p09354_s33', 'split': 'train', 'within_configured_limit': 'False', 'N': '3917', 'S': '0', 'V': '0', 'Q': '20', 'S+V': '0'}, {'patient_id': 'p09354', 'record_id': 'p09354_s34', 'split': 'train', 'within_configured_limit': 'False', 'N': '4281', 'S': '1', 'V': '3', 'Q': '60', 'S+V': '4'}, {'patient_id': 'p09354', 'record_id': 'p09354_s35', 'split': 'train', 'within_configured_limit': 'False', 'N': '4271', 'S': '0', 'V': '0', 'Q': '12', 'S+V': '0'}, {'patient_id': 'p09354', 'record_id': 'p09354_s36', 'split': 'train', 'within_configured_limit': 'False', 'N': '4138', 'S': '1', 'V': '0', 'Q': '17', 'S+V': '1'}, {'patient_id': 'p09354', 'record_id': 'p09354_s37', 'split': 'train', 'within_configured_limit': 'False', 'N': '4527', 'S': '1', 'V': '0', 'Q': '143', 'S+V': '1'}, {'patient_id': 'p09354', 'record_id': 'p09354_s38', 'split': 'train', 'within_configured_limit': 'False', 'N': '4440', 'S': '0', 'V': '0', 'Q': '55', 'S+V': '0'}, {'patient_id': 'p09354', 'record_id': 'p09354_s39', 'split': 'train', 'within_configured_limit': 'False', 'N': '4722', 'S': '0', 'V': '0', 'Q': '792', 'S+V': '0'}, {'patient_id': 'p09354', 'record_id': 'p09354_s40', 'split': 'train', 'within_configured_limit': 'False', 'N': '3661', 'S': '0', 'V': '1', 'Q': '1775', 'S+V': '1'}, {'patient_id': 'p09440', 'record_id': 'p09440_s00', 'split': 'train', 'within_configured_limit': 'True', 'N': '1992', 'S': '1', 'V': '0', 'Q': '4140', 'S+V': '1'}, {'patient_id': 'p09440', 'record_id': 'p09440_s01', 'split': 'train', 'within_configured_limit': 'False', 'N': '4717', 'S': '0', 'V': '0', 'Q': '1066', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s02', 'split': 'train', 'within_configured_limit': 'False', 'N': '5552', 'S': '0', 'V': '0', 'Q': '67', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s03', 'split': 'train', 'within_configured_limit': 'False', 'N': '5411', 'S': '0', 'V': '0', 'Q': '440', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s04', 'split': 'train', 'within_configured_limit': 'False', 'N': '5298', 'S': '2', 'V': '0', 'Q': '1136', 'S+V': '2'}, {'patient_id': 'p09440', 'record_id': 'p09440_s05', 'split': 'train', 'within_configured_limit': 'False', 'N': '5405', 'S': '3', 'V': '0', 'Q': '94', 'S+V': '3'}, {'patient_id': 'p09440', 'record_id': 'p09440_s06', 'split': 'train', 'within_configured_limit': 'False', 'N': '5379', 'S': '0', 'V': '0', 'Q': '147', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s07', 'split': 'train', 'within_configured_limit': 'False', 'N': '5095', 'S': '0', 'V': '0', 'Q': '48', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s08', 'split': 'train', 'within_configured_limit': 'False', 'N': '4783', 'S': '2', 'V': '0', 'Q': '554', 'S+V': '2'}, {'patient_id': 'p09440', 'record_id': 'p09440_s09', 'split': 'train', 'within_configured_limit': 'False', 'N': '4124', 'S': '0', 'V': '0', 'Q': '2087', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s10', 'split': 'train', 'within_configured_limit': 'False', 'N': '3873', 'S': '2', 'V': '0', 'Q': '1520', 'S+V': '2'}, {'patient_id': 'p09440', 'record_id': 'p09440_s11', 'split': 'train', 'within_configured_limit': 'False', 'N': '6453', 'S': '0', 'V': '0', 'Q': '766', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s12', 'split': 'train', 'within_configured_limit': 'False', 'N': '0', 'S': '0', 'V': '0', 'Q': '5078', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s13', 'split': 'train', 'within_configured_limit': 'False', 'N': '0', 'S': '0', 'V': '0', 'Q': '5308', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s14', 'split': 'train', 'within_configured_limit': 'False', 'N': '6360', 'S': '0', 'V': '0', 'Q': '1861', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s15', 'split': 'train', 'within_configured_limit': 'False', 'N': '7655', 'S': '0', 'V': '0', 'Q': '1488', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s16', 'split': 'train', 'within_configured_limit': 'False', 'N': '6849', 'S': '0', 'V': '0', 'Q': '1145', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s17', 'split': 'train', 'within_configured_limit': 'False', 'N': '2597', 'S': '0', 'V': '0', 'Q': '3006', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s18', 'split': 'train', 'within_configured_limit': 'False', 'N': '4380', 'S': '0', 'V': '0', 'Q': '888', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s19', 'split': 'train', 'within_configured_limit': 'False', 'N': '3274', 'S': '2', 'V': '0', 'Q': '1357', 'S+V': '2'}, {'patient_id': 'p09440', 'record_id': 'p09440_s20', 'split': 'train', 'within_configured_limit': 'False', 'N': '5365', 'S': '0', 'V': '0', 'Q': '393', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s21', 'split': 'train', 'within_configured_limit': 'False', 'N': '5266', 'S': '2', 'V': '0', 'Q': '175', 'S+V': '2'}, {'patient_id': 'p09440', 'record_id': 'p09440_s22', 'split': 'train', 'within_configured_limit': 'False', 'N': '5262', 'S': '0', 'V': '0', 'Q': '275', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s23', 'split': 'train', 'within_configured_limit': 'False', 'N': '5136', 'S': '0', 'V': '0', 'Q': '213', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s24', 'split': 'train', 'within_configured_limit': 'False', 'N': '5550', 'S': '0', 'V': '0', 'Q': '470', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s25', 'split': 'train', 'within_configured_limit': 'False', 'N': '5605', 'S': '0', 'V': '0', 'Q': '141', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s26', 'split': 'train', 'within_configured_limit': 'False', 'N': '2740', 'S': '0', 'V': '0', 'Q': '2236', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s27', 'split': 'train', 'within_configured_limit': 'False', 'N': '1072', 'S': '0', 'V': '0', 'Q': '4179', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s28', 'split': 'train', 'within_configured_limit': 'False', 'N': '7014', 'S': '0', 'V': '0', 'Q': '638', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s29', 'split': 'train', 'within_configured_limit': 'False', 'N': '5447', 'S': '0', 'V': '0', 'Q': '729', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s30', 'split': 'train', 'within_configured_limit': 'False', 'N': '4958', 'S': '0', 'V': '0', 'Q': '445', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s31', 'split': 'train', 'within_configured_limit': 'False', 'N': '5551', 'S': '2', 'V': '0', 'Q': '175', 'S+V': '2'}, {'patient_id': 'p09440', 'record_id': 'p09440_s32', 'split': 'train', 'within_configured_limit': 'False', 'N': '5377', 'S': '0', 'V': '0', 'Q': '801', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s33', 'split': 'train', 'within_configured_limit': 'False', 'N': '5313', 'S': '0', 'V': '0', 'Q': '621', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s34', 'split': 'train', 'within_configured_limit': 'False', 'N': '5174', 'S': '0', 'V': '0', 'Q': '232', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s35', 'split': 'train', 'within_configured_limit': 'False', 'N': '4627', 'S': '0', 'V': '0', 'Q': '155', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s36', 'split': 'train', 'within_configured_limit': 'False', 'N': '6005', 'S': '0', 'V': '0', 'Q': '791', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s37', 'split': 'train', 'within_configured_limit': 'False', 'N': '5330', 'S': '0', 'V': '0', 'Q': '648', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s38', 'split': 'train', 'within_configured_limit': 'False', 'N': '5850', 'S': '0', 'V': '0', 'Q': '1464', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s39', 'split': 'train', 'within_configured_limit': 'False', 'N': '5470', 'S': '1', 'V': '0', 'Q': '1964', 'S+V': '1'}, {'patient_id': 'p09440', 'record_id': 'p09440_s40', 'split': 'train', 'within_configured_limit': 'False', 'N': '2350', 'S': '0', 'V': '0', 'Q': '2575', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s41', 'split': 'train', 'within_configured_limit': 'False', 'N': '308', 'S': '0', 'V': '0', 'Q': '5260', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s42', 'split': 'train', 'within_configured_limit': 'False', 'N': '5745', 'S': '0', 'V': '0', 'Q': '298', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s43', 'split': 'train', 'within_configured_limit': 'False', 'N': '3342', 'S': '1', 'V': '0', 'Q': '2613', 'S+V': '1'}, {'patient_id': 'p09440', 'record_id': 'p09440_s44', 'split': 'train', 'within_configured_limit': 'False', 'N': '5194', 'S': '0', 'V': '0', 'Q': '467', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s45', 'split': 'train', 'within_configured_limit': 'False', 'N': '4508', 'S': '1', 'V': '0', 'Q': '646', 'S+V': '1'}, {'patient_id': 'p09440', 'record_id': 'p09440_s46', 'split': 'train', 'within_configured_limit': 'False', 'N': '922', 'S': '0', 'V': '0', 'Q': '4275', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s47', 'split': 'train', 'within_configured_limit': 'False', 'N': '4889', 'S': '0', 'V': '0', 'Q': '494', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s48', 'split': 'train', 'within_configured_limit': 'False', 'N': '5004', 'S': '0', 'V': '0', 'Q': '20', 'S+V': '0'}, {'patient_id': 'p09440', 'record_id': 'p09440_s49', 'split': 'train', 'within_configured_limit': 'False', 'N': '4734', 'S': '0', 'V': '0', 'Q': '2157', 'S+V': '0'}, {'patient_id': 'p09951', 'record_id': 'p09951_s00', 'split': 'train', 'within_configured_limit': 'True', 'N': '2901', 'S': '0', 'V': '92', 'Q': '1771', 'S+V': '92'}, {'patient_id': 'p09951', 'record_id': 'p09951_s01', 'split': 'train', 'within_configured_limit': 'False', 'N': '2816', 'S': '0', 'V': '252', 'Q': '2307', 'S+V': '252'}, {'patient_id': 'p09951', 'record_id': 'p09951_s02', 'split': 'train', 'within_configured_limit': 'False', 'N': '3718', 'S': '0', 'V': '348', 'Q': '1164', 'S+V': '348'}, {'patient_id': 'p09951', 'record_id': 'p09951_s03', 'split': 'train', 'within_configured_limit': 'False', 'N': '2954', 'S': '0', 'V': '182', 'Q': '1992', 'S+V': '182'}, {'patient_id': 'p09951', 'record_id': 'p09951_s04', 'split': 'train', 'within_configured_limit': 'False', 'N': '3946', 'S': '0', 'V': '90', 'Q': '425', 'S+V': '90'}, {'patient_id': 'p09951', 'record_id': 'p09951_s05', 'split': 'train', 'within_configured_limit': 'False', 'N': '3202', 'S': '0', 'V': '22', 'Q': '1356', 'S+V': '22'}, {'patient_id': 'p09951', 'record_id': 'p09951_s06', 'split': 'train', 'within_configured_limit': 'False', 'N': '4261', 'S': '0', 'V': '184', 'Q': '364', 'S+V': '184'}, {'patient_id': 'p09951', 'record_id': 'p09951_s07', 'split': 'train', 'within_configured_limit': 'False', 'N': '4007', 'S': '0', 'V': '193', 'Q': '438', 'S+V': '193'}, {'patient_id': 'p09951', 'record_id': 'p09951_s08', 'split': 'train', 'within_configured_limit': 'False', 'N': '3747', 'S': '0', 'V': '6', 'Q': '129', 'S+V': '6'}, {'patient_id': 'p09951', 'record_id': 'p09951_s09', 'split': 'train', 'within_configured_limit': 'False', 'N': '3793', 'S': '0', 'V': '13', 'Q': '68', 'S+V': '13'}, {'patient_id': 'p09951', 'record_id': 'p09951_s10', 'split': 'train', 'within_configured_limit': 'False', 'N': '3706', 'S': '0', 'V': '17', 'Q': '163', 'S+V': '17'}, {'patient_id': 'p09951', 'record_id': 'p09951_s11', 'split': 'train', 'within_configured_limit': 'False', 'N': '3599', 'S': '0', 'V': '12', 'Q': '265', 'S+V': '12'}, {'patient_id': 'p09951', 'record_id': 'p09951_s12', 'split': 'train', 'within_configured_limit': 'False', 'N': '3608', 'S': '0', 'V': '8', 'Q': '228', 'S+V': '8'}, {'patient_id': 'p09951', 'record_id': 'p09951_s13', 'split': 'train', 'within_configured_limit': 'False', 'N': '3284', 'S': '2', 'V': '6', 'Q': '983', 'S+V': '8'}, {'patient_id': 'p09951', 'record_id': 'p09951_s14', 'split': 'train', 'within_configured_limit': 'False', 'N': '1822', 'S': '4', 'V': '102', 'Q': '2649', 'S+V': '106'}, {'patient_id': 'p09951', 'record_id': 'p09951_s15', 'split': 'train', 'within_configured_limit': 'False', 'N': '2895', 'S': '0', 'V': '1084', 'Q': '785', 'S+V': '1084'}, {'patient_id': 'p09951', 'record_id': 'p09951_s16', 'split': 'train', 'within_configured_limit': 'False', 'N': '2779', 'S': '0', 'V': '687', 'Q': '1199', 'S+V': '687'}, {'patient_id': 'p09951', 'record_id': 'p09951_s17', 'split': 'train', 'within_configured_limit': 'False', 'N': '3659', 'S': '0', 'V': '575', 'Q': '222', 'S+V': '575'}, {'patient_id': 'p09951', 'record_id': 'p09951_s18', 'split': 'train', 'within_configured_limit': 'False', 'N': '2900', 'S': '0', 'V': '1118', 'Q': '870', 'S+V': '1118'}, {'patient_id': 'p09951', 'record_id': 'p09951_s19', 'split': 'train', 'within_configured_limit': 'False', 'N': '3269', 'S': '0', 'V': '991', 'Q': '787', 'S+V': '991'}, {'patient_id': 'p09951', 'record_id': 'p09951_s20', 'split': 'train', 'within_configured_limit': 'False', 'N': '3361', 'S': '0', 'V': '1288', 'Q': '545', 'S+V': '1288'}, {'patient_id': 'p09951', 'record_id': 'p09951_s21', 'split': 'train', 'within_configured_limit': 'False', 'N': '3538', 'S': '0', 'V': '1189', 'Q': '299', 'S+V': '1189'}, {'patient_id': 'p09951', 'record_id': 'p09951_s22', 'split': 'train', 'within_configured_limit': 'False', 'N': '3846', 'S': '0', 'V': '1248', 'Q': '35', 'S+V': '1248'}, {'patient_id': 'p09951', 'record_id': 'p09951_s23', 'split': 'train', 'within_configured_limit': 'False', 'N': '3728', 'S': '0', 'V': '1078', 'Q': '171', 'S+V': '1078'}, {'patient_id': 'p09951', 'record_id': 'p09951_s24', 'split': 'train', 'within_configured_limit': 'False', 'N': '3752', 'S': '0', 'V': '708', 'Q': '48', 'S+V': '708'}, {'patient_id': 'p09951', 'record_id': 'p09951_s25', 'split': 'train', 'within_configured_limit': 'False', 'N': '3612', 'S': '1', 'V': '93', 'Q': '203', 'S+V': '94'}, {'patient_id': 'p09951', 'record_id': 'p09951_s26', 'split': 'train', 'within_configured_limit': 'False', 'N': '3567', 'S': '0', 'V': '81', 'Q': '192', 'S+V': '81'}, {'patient_id': 'p09951', 'record_id': 'p09951_s27', 'split': 'train', 'within_configured_limit': 'False', 'N': '3195', 'S': '0', 'V': '6', 'Q': '621', 'S+V': '6'}, {'patient_id': 'p09951', 'record_id': 'p09951_s28', 'split': 'train', 'within_configured_limit': 'False', 'N': '2020', 'S': '0', 'V': '678', 'Q': '2438', 'S+V': '678'}, {'patient_id': 'p09951', 'record_id': 'p09951_s29', 'split': 'train', 'within_configured_limit': 'False', 'N': '2694', 'S': '0', 'V': '1062', 'Q': '1092', 'S+V': '1062'}, {'patient_id': 'p09951', 'record_id': 'p09951_s30', 'split': 'train', 'within_configured_limit': 'False', 'N': '2108', 'S': '0', 'V': '1380', 'Q': '1342', 'S+V': '1380'}, {'patient_id': 'p09951', 'record_id': 'p09951_s31', 'split': 'train', 'within_configured_limit': 'False', 'N': '2579', 'S': '0', 'V': '1028', 'Q': '1480', 'S+V': '1028'}, {'patient_id': 'p09951', 'record_id': 'p09951_s32', 'split': 'train', 'within_configured_limit': 'False', 'N': '2499', 'S': '0', 'V': '1026', 'Q': '1347', 'S+V': '1026'}, {'patient_id': 'p09951', 'record_id': 'p09951_s33', 'split': 'train', 'within_configured_limit': 'False', 'N': '3352', 'S': '0', 'V': '1295', 'Q': '296', 'S+V': '1295'}, {'patient_id': 'p09951', 'record_id': 'p09951_s34', 'split': 'train', 'within_configured_limit': 'False', 'N': '2936', 'S': '0', 'V': '933', 'Q': '1091', 'S+V': '933'}, {'patient_id': 'p09951', 'record_id': 'p09951_s35', 'split': 'train', 'within_configured_limit': 'False', 'N': '2633', 'S': '0', 'V': '608', 'Q': '1874', 'S+V': '608'}, {'patient_id': 'p09951', 'record_id': 'p09951_s36', 'split': 'train', 'within_configured_limit': 'False', 'N': '2752', 'S': '0', 'V': '1004', 'Q': '1256', 'S+V': '1004'}, {'patient_id': 'p09951', 'record_id': 'p09951_s37', 'split': 'train', 'within_configured_limit': 'False', 'N': '2443', 'S': '0', 'V': '985', 'Q': '1506', 'S+V': '985'}, {'patient_id': 'p09951', 'record_id': 'p09951_s38', 'split': 'train', 'within_configured_limit': 'False', 'N': '2272', 'S': '0', 'V': '922', 'Q': '1616', 'S+V': '922'}, {'patient_id': 'p09951', 'record_id': 'p09951_s39', 'split': 'train', 'within_configured_limit': 'False', 'N': '2034', 'S': '0', 'V': '734', 'Q': '1783', 'S+V': '734'}, {'patient_id': 'p09951', 'record_id': 'p09951_s40', 'split': 'train', 'within_configured_limit': 'False', 'N': '3514', 'S': '0', 'V': '903', 'Q': '246', 'S+V': '903'}, {'patient_id': 'p09951', 'record_id': 'p09951_s41', 'split': 'train', 'within_configured_limit': 'False', 'N': '3585', 'S': '0', 'V': '1072', 'Q': '162', 'S+V': '1072'}, {'patient_id': 'p09951', 'record_id': 'p09951_s42', 'split': 'train', 'within_configured_limit': 'False', 'N': '3507', 'S': '0', 'V': '1024', 'Q': '144', 'S+V': '1024'}, {'patient_id': 'p09951', 'record_id': 'p09951_s43', 'split': 'train', 'within_configured_limit': 'False', 'N': '3592', 'S': '0', 'V': '919', 'Q': '124', 'S+V': '919'}, {'patient_id': 'p09951', 'record_id': 'p09951_s44', 'split': 'train', 'within_configured_limit': 'False', 'N': '3669', 'S': '1', 'V': '345', 'Q': '48', 'S+V': '346'}, {'patient_id': 'p09951', 'record_id': 'p09951_s45', 'split': 'train', 'within_configured_limit': 'False', 'N': '2627', 'S': '1', 'V': '323', 'Q': '1276', 'S+V': '324'}, {'patient_id': 'p09951', 'record_id': 'p09951_s46', 'split': 'train', 'within_configured_limit': 'False', 'N': '1238', 'S': '0', 'V': '517', 'Q': '3259', 'S+V': '517'}, {'patient_id': 'p09951', 'record_id': 'p09951_s47', 'split': 'train', 'within_configured_limit': 'False', 'N': '1234', 'S': '0', 'V': '970', 'Q': '2824', 'S+V': '970'}, {'patient_id': 'p09951', 'record_id': 'p09951_s48', 'split': 'train', 'within_configured_limit': 'False', 'N': '1184', 'S': '0', 'V': '678', 'Q': '3131', 'S+V': '678'}, {'patient_id': 'p09951', 'record_id': 'p09951_s49', 'split': 'train', 'within_configured_limit': 'False', 'N': '2955', 'S': '0', 'V': '1201', 'Q': '444', 'S+V': '1201'}], 'annotation_counts_configured_limit': [{'split': 'train', 'N': '52433', 'S': '65', 'V': '280', 'Q': '37858', 'S+V': '345', 'n_records': '16'}, {'split': 'dev', 'N': '8065', 'S': '217', 'V': '1', 'Q': '2294', 'S+V': '218', 'n_records': '2'}, {'split': 'test', 'N': '7548', 'S': '1030', 'V': '10', 'Q': '2318', 'S+V': '1040', 'n_records': '2'}]}
for name, rows in annotation_count_snapshot.items():
    table = pd.DataFrame(rows)
    display(HTML(f"<h3>{name}</h3><div style='max-height:450px;overflow:auto'>" + table.to_html(index=False) + "</div>"))


split,N,S,V,Q,S+V,n_records
train,3049803,4993,40429,976126,45422,781
dev,338004,6510,12,145912,6522,100
test,380274,36386,5507,74175,41893,100


split,patient_id,N,S,V,Q,S+V
dev,p00877,197007,6508,5,43073,6513
dev,p09134,140997,2,7,102839,9
test,p01965,232878,271,5240,19767,5511
test,p04071,147396,36115,267,54408,36382
train,p00290,271348,120,345,32576,465
train,p01083,116025,819,4,93605,823
train,p01950,267440,3,349,25236,352
train,p03281,246864,2,3,34188,5
train,p03900,255402,20,10,37888,30
train,p04014,105368,11,0,169907,11


patient_id,record_id,split,within_configured_limit,N,S,V,Q,S+V
p00290,p00290_s00,train,True,5197,1,11,1228,12
p00290,p00290_s01,train,False,4980,1,5,593,6
p00290,p00290_s02,train,False,4720,0,12,1129,12
p00290,p00290_s03,train,False,5627,0,2,335,2
p00290,p00290_s04,train,False,5528,2,4,99,6
p00290,p00290_s05,train,False,5555,1,2,82,3
p00290,p00290_s06,train,False,5985,0,3,272,3
p00290,p00290_s07,train,False,6139,0,0,71,0
p00290,p00290_s08,train,False,5994,7,18,957,25
p00290,p00290_s09,train,False,5323,2,19,1152,21


split,N,S,V,Q,S+V,n_records
train,52433,65,280,37858,345,16
dev,8065,217,1,2294,218,2
test,7548,1030,10,2318,1040,2


## Dataset preparado — resultado final

Preparação de todos os **981 registos** concluída em `datasets/icentia11_prepared/`: **4 865 610 batimentos**. Todos os arrays foram verificados (forma, float32, valores finitos, normalização, labels e SQI), bem como a consistência do manifesto e a separação por paciente. Os 16 clientes de treino têm dados; dev/test não recebem cliente de treino.

| Split | N | S | V | Q | Total |
| --- | ---: | ---: | ---: | ---: | ---: |
| train | 3 046 595 | 4 986 | 40 388 | 812 253 | 3 904 222 |
| dev | 337 404 | 6 461 | 12 | 126 123 | 470 000 |
| test | 379 868 | 36 355 | 5 499 | 69 666 | 491 388 |

Estas contagens são posteriores às rejeições, ao contrário das contagens de anotações originais acima. Q significa batimento não classificável e foi mantida separadamente de N/S/V.

A saída contém `manifest.csv`, os NPZ por batimento, os manifestos dos clientes e de avaliação, as estatísticas de processamento, as tabelas de classes, `preparation_config.json` e `validation_report.json`. Os checkpoints permitem retomar com a mesma configuração e os mesmos dados; para alterar a pipeline, usar uma nova pasta de saída, sem reutilizar os checkpoints anteriores.

A execução automática mantém-se desligada para evitar repetir a preparação ao abrir o notebook. O dataset já está gerado.


In [ ]:
# Resumo da preparação concluída, sem voltar a processar os ECG.
prepared_summary = pd.read_csv(OUTPUT_ROOT / "split_class_counts.csv").set_index("split")
prepared_summary["total"] = prepared_summary.sum(axis=1)
display(prepared_summary)
print((OUTPUT_ROOT / "validation_report.json").read_text())


split,N,S,V,Q,total
train,3046595,4986,40388,812253,3904222
dev,337404,6461,12,126123,470000
test,379868,36355,5499,69666,491388


{
  "status": "complete",
  "records": 981,
  "validated_arrays": 4865610,
  "all_arrays_checked": true,
  "manifest_consistency_checked": true,
  "elapsed_seconds": 414,
  "output_directory": "datasets/icentia11_prepared",
  "client_partition": "one_client_per_patient",
  "patient_client_bijection_checked": true,
  "split_client_counts": {
    "train": 16,
    "dev": 2,
    "test": 2
  },
  "array_validation_note": "Signals unchanged; prior complete array validation remains applicable. All manifest rows checked during patient reassignment."
}

## Resumo dos splits: um cliente por paciente

| Split | Pacientes / clientes | Records preparados |
| --- | ---: | ---: |
| train | 16 | 781 |
| dev | 2 | 100 |
| test | 2 | 100 |

Cada paciente tem um único `client_id`, e cada cliente contém dados de um único paciente. Dev/test têm identificadores próprios, mas não participam no treino federado. A distribuição de classes por cliente é a distribuição natural do respetivo paciente.

`Records` conta registos ECG contínuos, não batimentos. Em treino: **14 × 50 + 40 + 41 = 781**. Dev/test têm 2 × 50 = 100 cada.

### Verificação por paciente

Comparação com os ficheiros `RECORDS` guardados localmente: todos os registos indexados têm `.hea`, `.dat` e `.atr`, e todos aparecem no manifesto preparado. A semente 2024 reproduz a divisão guardada na configuração. Não existem pacientes ou registos partilhados entre train/dev/test.

| Split | Paciente | Registos indexados | Registos preparados |
| --- | --- | ---: | ---: |
| dev | p00877 | 50 | 50 |
| dev | p09134 | 50 | 50 |
| test | p01965 | 50 | 50 |
| test | p04071 | 50 | 50 |
| train | p00290 | 50 | 50 |
| train | p01083 | 50 | 50 |
| train | p01950 | 50 | 50 |
| train | p03281 | 50 | 50 |
| train | p03900 | 50 | 50 |
| train | p04014 | 50 | 50 |
| train | p05133 | 50 | 50 |
| train | p07028 | 50 | 50 |
| train | p07070 | 50 | 50 |
| train | p07179 | 40 | 40 |
| train | p07743 | 50 | 50 |
| train | p07929 | 50 | 50 |
| train | p08689 | 50 | 50 |
| train | p09354 | 41 | 41 |
| train | p09440 | 50 | 50 |
| train | p09951 | 50 | 50 |

Os 40 registos de `p07179` e os 41 de `p09354` já constam nos índices locais; a diferença relativamente a 50 não foi causada pela preparação. Esta verificação é de consistência com os índices locais, sem nova consulta do servidor de origem.



In [ ]:
# Resumo da partição atual: um cliente por paciente.
patient_config = json.loads((OUTPUT_ROOT / "preparation_config.json").read_text())
record_stats = pd.read_csv(OUTPUT_ROOT / "record_processing_stats.csv")
split_client_record_summary = pd.DataFrame([
    {"split": split, "clientes": len(patient_config["patient_ids"][split]),
     "records": record_stats.loc[record_stats["split"] == split, "record_id"].nunique()}
    for split in ("train", "dev", "test")
])
display(split_client_record_summary)
print(split_client_record_summary.to_string(index=False))


split  clientes  records
train        16      781
dev           2      100
test          2      100


## Distribuição de batimentos por cliente (um paciente por cliente)

| Clientes | Split | N | S | V | Q | Total |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| Cliente 1 | train | 271148 | 120 | 344 | 29542 | 301154 |
| Cliente 2 | train | 115815 | 813 | 4 | 71941 | 188573 |
| Cliente 3 | train | 267147 | 3 | 327 | 23542 | 291019 |
| Cliente 4 | train | 246788 | 2 | 3 | 32619 | 279412 |
| Cliente 5 | train | 255270 | 20 | 10 | 30335 | 285635 |
| Cliente 6 | train | 104576 | 11 | 0 | 129527 | 234114 |
| Cliente 7 | train | 146659 | 83 | 13 | 72081 | 218836 |
| Cliente 8 | train | 202577 | 2125 | 400 | 33231 | 238333 |
| Cliente 9 | train | 162762 | 1711 | 3 | 65528 | 230004 |
| Cliente 10 | train | 170541 | 7 | 0 | 43474 | 214022 |
| Cliente 11 | train | 208625 | 10 | 1 | 62251 | 270887 |
| Cliente 12 | train | 207325 | 27 | 1 | 27755 | 235108 |
| Cliente 13 | train | 147723 | 0 | 7956 | 54766 | 210445 |
| Cliente 14 | train | 158965 | 26 | 55 | 43413 | 202459 |
| Cliente 15 | train | 228184 | 19 | 0 | 48067 | 276270 |
| Cliente 16 | train | 152490 | 9 | 31271 | 44181 | 227951 |
| Cliente 17 | dev | 196462 | 6459 | 5 | 35500 | 238426 |
| Cliente 18 | dev | 140942 | 2 | 7 | 90623 | 231574 |
| Cliente 19 | test | 232600 | 271 | 5232 | 18126 | 256229 |
| Cliente 20 | test | 147268 | 36084 | 267 | 51540 | 235159 |

Os números são batimentos aceites após a preparação. Clientes 1–16 são de treino, 17–18 de dev e 19–20 de test. Apenas os clientes de treino participam na federação. Os IDs nos manifestos são `client_00` a `client_19`, na mesma ordem.


In [ ]:
# Recalcular a tabela com as contagens aceites, sem reprocessar os sinais.
config = json.loads((OUTPUT_ROOT / "preparation_config.json").read_text())
stats = pd.read_csv(OUTPUT_ROOT / "record_processing_stats.csv")
accepted = stats.groupby("patient_id")[[f"accepted_{label}" for label in ("N", "S", "V", "Q")]].sum()
client_rows = []
for split in ("train", "dev", "test"):
    for patient_id in sorted(config["patient_ids"][split]):
        row = {"clientes": f"Cliente {len(client_rows) + 1}", "split": split,
               **{label: int(accepted.loc[patient_id, f"accepted_{label}"]) for label in ("N", "S", "V", "Q")}}
        row["Total"] = sum(row[label] for label in ("N", "S", "V", "Q"))
        client_rows.append(row)
client_class_table = pd.DataFrame(client_rows)
assert client_class_table["Total"].sum() == stats["accepted"].sum()
display(client_class_table)
print(client_class_table.to_string(index=False))


clientes,split,N,S,V,Q,Total
Cliente 1,train,271148,120,344,29542,301154
Cliente 2,train,115815,813,4,71941,188573
Cliente 3,train,267147,3,327,23542,291019
Cliente 4,train,246788,2,3,32619,279412
Cliente 5,train,255270,20,10,30335,285635
Cliente 6,train,104576,11,0,129527,234114
Cliente 7,train,146659,83,13,72081,218836
Cliente 8,train,202577,2125,400,33231,238333
Cliente 9,train,162762,1711,3,65528,230004
Cliente 10,train,170541,7,0,43474,214022


  clientes split      N     S     V      Q  Total
 Cliente 1 train 271148   120   344  29542 301154
 Cliente 2 train 115815   813     4  71941 188573
 Cliente 3 train 267147     3   327  23542 291019
 Cliente 4 train 246788     2     3  32619 279412
 Cliente 5 train 255270    20    10  30335 285635
 Cliente 6 train 104576    11     0 129527 234114
 Cliente 7 train 146659    83    13  72081 218836
 Cliente 8 train 202577  2125   400  33231 238333
 Cliente 9 train 162762  1711     3  65528 230004
Cliente 10 train 170541     7     0  43474 214022
Cliente 11 train 208625    10     1  62251 270887
Cliente 12 train 207325    27     1  27755 235108
Cliente 13 train 147723     0  7956  54766 210445
Cliente 14 train 158965    26    55  43413 202459
Cliente 15 train 228184    19     0  48067 276270
Cliente 16 train 152490     9 31271  44181 227951
Cliente 17   dev 196462  6459     5  35500 238426
Cliente 18   dev 140942     2     7  90623 231574
Cliente 19  test 232600   271  5232  18126 256229
